# 02 — Recalibrate Simple Signal Grid  
## Locked production research candidate: Core / Secondary 2621

This notebook is the cleaned research notebook for the simple SPX VRP signal grid.

### What this notebook does

1. Loads the production feature panel.
2. Loads the completed naked ATM put outcome panel.
3. Builds the complete-date recalibration matrix.
4. Evaluates the locked **Core / Secondary 2621** candidate.
5. Produces reusable audit tables for future validation and sizing work.
6. Uses **tenor-level average P&L/day** as the within-layer tenor-selection priority.

### What was intentionally removed

The earlier exploratory search code was removed from the executable path:

- first-pass through fifth-pass grid sweeps
- failed 90% Core search
- pair 144 intermediate tests
- tenor-line fitting
- hybrid line-fitting tests

Those were useful research steps, but 2621 is now locked. This notebook is meant to be smaller, easier to rerun, and easier to extend without carrying dead code.

### Locked selection-priority convention

When multiple tenors qualify inside the same layer, the notebook ranks qualifying tenors by **historical average P&L per day**, not raw average P&L and not tenor length. Raw dollars mechanically favor longer-dated options, so they are not used for tenor selection.


In [1]:
# Cell 1 — Setup, imports, paths, and locked benchmark metadata

from pathlib import Path
from copy import deepcopy
import json
import math
import warnings

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 260)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

warnings.filterwarnings("ignore", category=FutureWarning)

# ---------------------------------------------------------------------
# Project paths
# ---------------------------------------------------------------------
#
# This notebook is intended to run both from the project root and from a
# notebooks/ subfolder. If neither location has a data/ directory, it falls
# back to the project path used during development.
# ---------------------------------------------------------------------

cwd = Path.cwd()

if (cwd / "data").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "data").exists():
    PROJECT_ROOT = cwd.parent
else:
    PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
AUDIT_DIR = DATA_DIR / "audit"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_PANEL_PATH = PROCESSED_DIR / "production_feature_panel_v0_1.parquet"
OUTCOME_PATH = PROCESSED_DIR / "naked_atm_put_eod_panel_v0_1.parquet"

# ---------------------------------------------------------------------
# Tenor universe and grouping
# ---------------------------------------------------------------------

PRODUCTION_TENORS = [9, 12, 15, 18, 21, 24, 27, 30, 33]

GROUP_TENORS = {
    "front": [9, 12, 15],
    "middle": [18, 21, 24],
    "back": [27, 30, 33],
}

TENOR_GROUP_MAP = {
    tenor: group
    for group, tenors in GROUP_TENORS.items()
    for tenor in tenors
}

# ---------------------------------------------------------------------
# Locked candidate and tenor-priority convention
# ---------------------------------------------------------------------
#
# Candidate 2621 refers to the locked Core/Secondary threshold set.
#
# Within each layer, if more than one tenor qualifies on the same date, the
# selected tenor is the qualifying tenor with the highest historical average
# P&L per day. Raw average P&L is deliberately not used because it mechanically
# favors longer-dated options.
#
# The benchmark dictionary is intentionally left empty here. Earlier research
# outputs were produced during exploration and may have used different priority
# conventions. The live notebook should be treated as the source of truth after
# rerunning this cleaned version.
# ---------------------------------------------------------------------

LOCKED_CANDIDATE_LABEL = "locked_core_secondary_2621"
TENOR_SELECTION_PRIORITY_METRIC = "avg_pnl_per_day"

EXPECTED_2621_BENCHMARK = {}

print("Current working directory:", cwd)
print("Project root:", PROJECT_ROOT)
print("Feature panel path:", FEATURE_PANEL_PATH)
print("Outcome path:", OUTCOME_PATH)
print("Feature panel exists:", FEATURE_PANEL_PATH.exists())
print("Outcome file exists:", OUTCOME_PATH.exists())


Current working directory: C:\Users\patri\vrp_project\notebooks v1
Project root: C:\Users\patri\vrp_project
Feature panel path: C:\Users\patri\vrp_project\data\processed\production_feature_panel_v0_1.parquet
Outcome path: C:\Users\patri\vrp_project\data\processed\naked_atm_put_eod_panel_v0_1.parquet
Feature panel exists: True
Outcome file exists: True


In [2]:
# Cell 2 — Small reusable data-cleaning helpers

def get_col(df: pd.DataFrame, candidates, required=True, label=None):
    """
    Return the first matching column from a candidate list.

    Matching is case-insensitive, but the original DataFrame column name is
    returned. This keeps the notebook robust to small naming changes in source
    files.
    """
    lower_map = {str(c).lower(): c for c in df.columns}

    for c in candidates:
        if c in df.columns:
            return c

        c_lower = str(c).lower()
        if c_lower in lower_map:
            return lower_map[c_lower]

    if required:
        raise KeyError(
            f"Missing column for {label or candidates}. "
            f"Tried: {candidates}. Available columns: {list(df.columns)}"
        )

    return None


def parse_project_date_series(s: pd.Series) -> pd.Series:
    """
    Parse project date columns safely.

    Important:
      Numeric yyyymmdd values such as 20190805 must be parsed as calendar
      dates, not as nanoseconds since 1970.
    """
    s = s.copy()

    if pd.api.types.is_datetime64_any_dtype(s):
        return pd.to_datetime(s)

    s_str = s.astype("string").str.strip()
    s_str = s_str.str.replace(r"\.0$", "", regex=True)

    parsed = pd.Series(pd.NaT, index=s.index, dtype="datetime64[ns]")

    is_yyyymmdd = s_str.str.fullmatch(r"\d{8}", na=False)

    if is_yyyymmdd.any():
        parsed.loc[is_yyyymmdd] = pd.to_datetime(
            s_str.loc[is_yyyymmdd],
            format="%Y%m%d",
            errors="coerce",
        )

    remaining = ~is_yyyymmdd

    if remaining.any():
        parsed.loc[remaining] = pd.to_datetime(
            s_str.loc[remaining],
            errors="coerce",
        )

    return parsed


def clean_binary_series(s: pd.Series) -> pd.Series:
    """
    Convert bool / numeric / string binary values into float 0/1.
    Missing or unresolved values stay NaN.
    """
    if pd.api.types.is_bool_dtype(s):
        return s.astype(float)

    numeric = pd.to_numeric(s, errors="coerce")

    if numeric.notna().mean() > 0.95:
        return numeric.astype(float)

    return (
        s.astype("string")
        .str.lower()
        .str.strip()
        .map({
            "true": 1.0,
            "false": 0.0,
            "yes": 1.0,
            "no": 0.0,
            "1": 1.0,
            "0": 0.0,
            "1.0": 1.0,
            "0.0": 0.0,
        })
    )


def require_columns(df: pd.DataFrame, required_cols, label="DataFrame"):
    missing_cols = [c for c in required_cols if c not in df.columns]

    if missing_cols:
        raise ValueError(f"{label} is missing required columns: {missing_cols}")

In [3]:
# Cell 3 — Load and validate the production feature panel

if not FEATURE_PANEL_PATH.exists():
    raise FileNotFoundError(f"Missing feature panel: {FEATURE_PANEL_PATH}")

features = pd.read_parquet(FEATURE_PANEL_PATH)
features["date"] = pd.to_datetime(features["date"]).dt.normalize()
features = features.sort_values(["date", "tenor"]).reset_index(drop=True)

required_feature_cols = [
    "date",
    "tenor",
    "tenor_group",
    "spx_close",
    "implied_variance",
    "forecast_variance",
    "vrp_log",
    "vrp_z_3m",
    "vrp_z_1y",
    "rv21d",
    "rsi14",
]

require_columns(features, required_feature_cols, label="Feature panel")

features = features[features["tenor"].isin(PRODUCTION_TENORS)].copy()
features["tenor"] = features["tenor"].astype(int)

eligible = features.dropna(
    subset=[
        "vrp_log",
        "vrp_z_3m",
        "vrp_z_1y",
        "rv21d",
        "rsi14",
        "implied_variance",
        "forecast_variance",
    ]
).copy()

eligible = eligible.sort_values(["date", "tenor"]).reset_index(drop=True)

dupes = eligible.duplicated(subset=["date", "tenor"]).sum()

print("Loaded feature panel")
print("  Shape:", features.shape)
print("  Date range:", features["date"].min(), "to", features["date"].max())
print("  Tenors:", sorted(features["tenor"].dropna().unique().tolist()))

print("\nEligible feature panel")
print("  Shape:", eligible.shape)
print("  Date range:", eligible["date"].min(), "to", eligible["date"].max())
print("  Tenors:", sorted(eligible["tenor"].dropna().unique().tolist()))
print("  Duplicate date/tenor rows:", dupes)

if dupes > 0:
    display(
        eligible[
            eligible.duplicated(subset=["date", "tenor"], keep=False)
        ].sort_values(["date", "tenor"]).head(30)
    )
    raise ValueError("Duplicate date/tenor rows found in eligible feature panel.")

# rv21d and rsi14 are market-level filters, so they should be identical
# across all tenors on a given date.
market_filter_counts = (
    eligible
    .groupby("date")[["rv21d", "rsi14"]]
    .nunique(dropna=False)
)

bad_market_filter_dates = market_filter_counts[
    (market_filter_counts["rv21d"] > 1) |
    (market_filter_counts["rsi14"] > 1)
]

print("  Dates with inconsistent rv21d/rsi14 across tenors:", len(bad_market_filter_dates))

if len(bad_market_filter_dates) > 0:
    display(bad_market_filter_dates.head(20))
    raise ValueError("rv21d or rsi14 differs across tenors on the same date.")

display(eligible.head(10))

Loaded feature panel
  Shape: (18099, 11)
  Date range: 2018-06-25 00:00:00 to 2026-06-25 00:00:00
  Tenors: [9, 12, 15, 18, 21, 24, 27, 30, 33]

Eligible feature panel
  Shape: (15750, 11)
  Date range: 2019-07-10 00:00:00 to 2026-06-25 00:00:00
  Tenors: [9, 12, 15, 18, 21, 24, 27, 30, 33]
  Duplicate date/tenor rows: 0
  Dates with inconsistent rv21d/rsi14 across tenors: 0


,date,tenor,tenor_group,spx_close,implied_variance,forecast_variance,vrp_log,vrp_z_3m,vrp_z_1y,rv21d,rsi14
0,2019-07-10,9,front,"2,993.070000",0.010209,0.010971,-0.071968,-1.465831,-0.563268,7.712163,67.404430
1,2019-07-10,12,front,"2,993.070000",0.011378,0.009403,0.190606,-1.095112,-0.216897,7.712163,67.404430
2,2019-07-10,15,front,"2,993.070000",0.012079,0.010344,0.155121,-1.128973,-0.216182,7.712163,67.404430
3,2019-07-10,18,middle,"2,993.070000",0.013868,0.009403,0.388501,-0.584906,0.141210,7.712163,67.404430
4,2019-07-10,21,middle,"2,993.070000",0.015711,0.010075,0.444317,-0.401009,0.274458,7.712163,67.404430
5,2019-07-10,24,middle,"2,993.070000",0.016752,0.009991,0.516811,-0.241134,0.377281,7.712163,67.404430
6,2019-07-10,27,back,"2,993.070000",0.016953,0.009926,0.535301,-0.246532,0.381816,7.712163,67.404430
7,2019-07-10,30,back,"2,993.070000",0.017114,0.010344,0.503503,-0.342138,0.323409,7.712163,67.404430
8,2019-07-10,33,back,"2,993.070000",0.017323,0.009831,0.566508,-0.216783,0.392516,7.712163,67.404430
9,2019-07-11,9,front,"2,999.910000",0.010349,0.008859,0.155524,-0.889726,-0.180556,7.676858,68.576313


In [4]:
# Cell 4 — Load naked ATM put outcomes and build the completed research panel

if not OUTCOME_PATH.exists():
    raise FileNotFoundError(f"Missing outcome file: {OUTCOME_PATH}")

outcomes_raw = pd.read_parquet(OUTCOME_PATH)

# ---------------------------------------------------------------------
# Column mapping
# ---------------------------------------------------------------------

date_col = get_col(
    outcomes_raw,
    ["trade_dt", "trade_date", "date", "timestamp", "datetime"],
    label="date",
)

tenor_col = get_col(
    outcomes_raw,
    ["entry_tenor", "tenor", "target_tenor", "actual_dte", "dte"],
    label="tenor",
)

win_col = get_col(
    outcomes_raw,
    ["win_clean", "win", "test_win"],
    label="win",
)

expired_otm_col = get_col(
    outcomes_raw,
    ["expired_otm_clean", "expired_otm"],
    required=False,
    label="expired_otm",
)

pnl_dollars_col = get_col(
    outcomes_raw,
    ["normalized_pnl_dollars", "test_pnl", "sized_pnl"],
    required=False,
    label="normalized_pnl_dollars",
)

pnl_pct_col = get_col(
    outcomes_raw,
    ["normalized_pnl_pct"],
    required=False,
    label="normalized_pnl_pct",
)

actual_dte_col = get_col(
    outcomes_raw,
    ["actual_dte"],
    required=False,
    label="actual_dte",
)

expiry_spx_col = get_col(
    outcomes_raw,
    ["expiry_spx_close"],
    required=False,
    label="expiry_spx_close",
)

entry_credit_col = get_col(
    outcomes_raw,
    ["entry_credit_points", "atm_put_mid"],
    required=False,
    label="entry_credit_points",
)

short_pnl_points_col = get_col(
    outcomes_raw,
    ["short_option_pnl_points"],
    required=False,
    label="short_option_pnl_points",
)

print("Outcome column mapping:")
print({
    "date_col": date_col,
    "tenor_col": tenor_col,
    "win_col": win_col,
    "expired_otm_col": expired_otm_col,
    "pnl_dollars_col": pnl_dollars_col,
    "pnl_pct_col": pnl_pct_col,
    "actual_dte_col": actual_dte_col,
    "expiry_spx_col": expiry_spx_col,
    "entry_credit_col": entry_credit_col,
    "short_pnl_points_col": short_pnl_points_col,
})

# ---------------------------------------------------------------------
# Standardize outcomes
# ---------------------------------------------------------------------

outcomes = pd.DataFrame({
    "date": parse_project_date_series(outcomes_raw[date_col]).dt.normalize(),
    "tenor": pd.to_numeric(outcomes_raw[tenor_col], errors="coerce").astype("Int64"),
    "win_clean": clean_binary_series(outcomes_raw[win_col]),
})

if expired_otm_col is not None:
    outcomes["expired_otm_clean"] = clean_binary_series(outcomes_raw[expired_otm_col])
else:
    outcomes["expired_otm_clean"] = np.nan

if pnl_dollars_col is not None:
    outcomes["normalized_pnl_dollars"] = pd.to_numeric(outcomes_raw[pnl_dollars_col], errors="coerce")
else:
    outcomes["normalized_pnl_dollars"] = np.nan

if pnl_pct_col is not None:
    outcomes["normalized_pnl_pct"] = pd.to_numeric(outcomes_raw[pnl_pct_col], errors="coerce")
else:
    outcomes["normalized_pnl_pct"] = np.nan

if actual_dte_col is not None:
    outcomes["actual_dte"] = pd.to_numeric(outcomes_raw[actual_dte_col], errors="coerce")
else:
    outcomes["actual_dte"] = np.nan

if expiry_spx_col is not None:
    outcomes["expiry_spx_close"] = pd.to_numeric(outcomes_raw[expiry_spx_col], errors="coerce")
else:
    outcomes["expiry_spx_close"] = np.nan

if entry_credit_col is not None:
    outcomes["entry_credit_points"] = pd.to_numeric(outcomes_raw[entry_credit_col], errors="coerce")
else:
    outcomes["entry_credit_points"] = np.nan

if short_pnl_points_col is not None:
    outcomes["short_option_pnl_points"] = pd.to_numeric(outcomes_raw[short_pnl_points_col], errors="coerce")
else:
    outcomes["short_option_pnl_points"] = np.nan

outcomes = outcomes[outcomes["tenor"].isin(PRODUCTION_TENORS)].copy()
outcomes = outcomes.dropna(subset=["date", "tenor"]).copy()
outcomes["tenor"] = outcomes["tenor"].astype(int)

dupes = outcomes.duplicated(subset=["date", "tenor"]).sum()

print("\nStandardized outcome panel")
print("  Shape:", outcomes.shape)
print("  Date range:", outcomes["date"].min(), "to", outcomes["date"].max())
print("  Tenors:", sorted(outcomes["tenor"].dropna().unique().tolist()))
print("  Duplicate date/tenor rows:", dupes)

if dupes > 0:
    display(
        outcomes[
            outcomes.duplicated(subset=["date", "tenor"], keep=False)
        ].sort_values(["date", "tenor"]).head(30)
    )
    raise ValueError("Outcome panel has duplicate date/tenor rows.")

# ---------------------------------------------------------------------
# Join outcomes to eligible features.
# ---------------------------------------------------------------------
#
# Missing outcomes near the right edge are expected because later-dated
# longer-tenor trades may not have expired yet. Those rows are dropped.
#
# Missing outcomes before the last completed outcome date for that tenor are
# treated as a data hole and raise an error.
# ---------------------------------------------------------------------

research_panel_raw = eligible.merge(
    outcomes,
    on=["date", "tenor"],
    how="left",
)

completed_outcomes = outcomes.dropna(subset=["win_clean"]).copy()

last_completed_date_by_tenor = (
    completed_outcomes
    .groupby("tenor")["date"]
    .max()
    .rename("last_completed_outcome_date")
    .reset_index()
)

missing_outcomes = (
    research_panel_raw[research_panel_raw["win_clean"].isna()]
    [["date", "tenor", "tenor_group", "spx_close", "vrp_log", "vrp_z_3m", "vrp_z_1y", "rv21d", "rsi14"]]
    .copy()
)

missing_outcomes = missing_outcomes.merge(
    last_completed_date_by_tenor,
    on="tenor",
    how="left",
)

unexpected_missing = missing_outcomes[
    missing_outcomes["date"] <= missing_outcomes["last_completed_outcome_date"]
].copy()

print("\nResearch panel after outcome join")
print("  Shape:", research_panel_raw.shape)
print("  Rows with missing win_clean:", int(research_panel_raw["win_clean"].isna().sum()))

print("\nLast completed outcome date by tenor:")
display(last_completed_date_by_tenor)

if len(missing_outcomes) > 0:
    missing_summary = (
        missing_outcomes
        .groupby("tenor")
        .agg(
            missing_rows=("date", "size"),
            first_missing_date=("date", "min"),
            last_missing_date=("date", "max"),
            last_completed_outcome_date=("last_completed_outcome_date", "max"),
        )
        .reset_index()
    )

    print("\nMissing outcome summary by tenor:")
    display(missing_summary)

if len(unexpected_missing) > 0:
    print("\nUnexpected missing outcomes found before/equal to last completed date for that tenor:")
    display(unexpected_missing.head(30))
    raise ValueError("Unexpected non-right-edge missing outcomes found.")

print("\nRight-edge censored rows to drop:", len(missing_outcomes))

research_panel = research_panel_raw.dropna(subset=["win_clean"]).copy()
research_panel = research_panel.sort_values(["date", "tenor"]).reset_index(drop=True)

required_outcome_cols = ["win_clean", "normalized_pnl_dollars", "actual_dte"]
require_columns(research_panel, required_outcome_cols, label="Completed research panel")

if research_panel[required_outcome_cols].isna().any().any():
    display(research_panel[research_panel[required_outcome_cols].isna().any(axis=1)].head(20))
    raise ValueError("Completed research panel has missing win/P&L/actual_dte values.")

print("\nFinal completed research panel")
print("  Shape:", research_panel.shape)
print("  Date range:", research_panel["date"].min(), "to", research_panel["date"].max())
print("  Tenors:", sorted(research_panel["tenor"].dropna().unique().tolist()))
print("  Overall win rate:", research_panel["win_clean"].mean())

outcome_by_tenor = (
    research_panel
    .groupby("tenor")
    .agg(
        rows=("win_clean", "size"),
        win_rate=("win_clean", "mean"),
        avg_pnl_dollars=("normalized_pnl_dollars", "mean"),
        avg_pnl_pct=("normalized_pnl_pct", "mean"),
        avg_actual_dte=("actual_dte", "mean"),
    )
    .reset_index()
)

print("\nOutcome summary by tenor:")
display(outcome_by_tenor)

Outcome column mapping:
{'date_col': 'trade_dt', 'tenor_col': 'entry_tenor', 'win_col': 'win', 'expired_otm_col': 'expired_otm', 'pnl_dollars_col': 'normalized_pnl_dollars', 'pnl_pct_col': None, 'actual_dte_col': 'actual_dte', 'expiry_spx_col': 'expiry_spx_close', 'entry_credit_col': 'entry_credit_points', 'short_pnl_points_col': 'short_option_pnl_points'}

Standardized outcome panel
  Shape: (18099, 10)
  Date range: 2018-06-25 00:00:00 to 2026-06-25 00:00:00
  Tenors: [9, 12, 15, 18, 21, 24, 27, 30, 33]
  Duplicate date/tenor rows: 0

Research panel after outcome join
  Shape: (15750, 19)
  Rows with missing win_clean: 153

Last completed outcome date by tenor:


,tenor,last_completed_outcome_date
0,9,2026-06-12
1,12,2026-06-09
2,15,2026-06-05
3,18,2026-06-03
4,21,2026-05-29
5,24,2026-05-28
6,27,2026-05-22
7,30,2026-05-22
8,33,2026-05-19



Missing outcome summary by tenor:


,tenor,missing_rows,first_missing_date,last_missing_date,last_completed_outcome_date
0,9,8,2026-06-15,2026-06-25,2026-06-12
1,12,11,2026-06-10,2026-06-25,2026-06-09
2,15,13,2026-06-08,2026-06-25,2026-06-05
3,18,15,2026-06-04,2026-06-25,2026-06-03
4,21,18,2026-06-01,2026-06-25,2026-05-29
5,24,19,2026-05-29,2026-06-25,2026-05-28
6,27,22,2026-05-26,2026-06-25,2026-05-22
7,30,22,2026-05-26,2026-06-25,2026-05-22
8,33,25,2026-05-20,2026-06-25,2026-05-19



Right-edge censored rows to drop: 153

Final completed research panel
  Shape: (15597, 19)
  Date range: 2019-07-10 00:00:00 to 2026-06-12 00:00:00
  Tenors: [9, 12, 15, 18, 21, 24, 27, 30, 33]
  Overall win rate: 0.7633519266525614

Outcome summary by tenor:


,tenor,rows,win_rate,avg_pnl_dollars,avg_pnl_pct,avg_actual_dte
0,9,1742,0.731343,"1,123.741635",NaN,8.940299
1,12,1739,0.742381,"1,816.837565",NaN,11.848189
2,15,1737,0.748417,"2,462.510510",NaN,15.940127
3,18,1735,0.753890,"2,648.826727",NaN,17.438040
4,21,1732,0.760393,"3,264.124558",NaN,21.727483
5,24,1731,0.767187,"3,535.662513",NaN,23.038128
6,27,1728,0.785301,"4,430.317093",NaN,27.284722
7,30,1728,0.788773,"4,900.713063",NaN,29.941551
8,33,1725,0.793043,"5,573.755492",NaN,32.848116


In [5]:
# Cell 5 — Build complete-date sweep panel and matrix representation

# ---------------------------------------------------------------------
# Tenor-level edge and selection priority
# ---------------------------------------------------------------------
#
# Selection rule:
#   When more than one tenor qualifies inside the same layer on the same date,
#   select the qualifying tenor with the highest full-sample historical
#   average P&L per day.
#
# Why P&L/day?
#   Raw average P&L mechanically favors longer-dated options because they carry
#   more time exposure. Average P&L per day normalizes the outcome by the actual
#   holding period, making 9D, 12D, ..., 33D tenors more comparable.
#
# Tie-breakers:
#   1. avg_pnl_per_day            primary
#   2. aggregate_pnl_per_day      secondary economic check
#   3. historical_win_probability consistency check
#   4. tenor                      deterministic final tie-breaker
#
# Note:
#   This priority rule is independent of the Core/Secondary threshold rules.
#   Core is always checked before Secondary. The P&L/day priority only decides
#   which tenor to take inside the active layer when multiple tenors qualify.
# ---------------------------------------------------------------------

research_panel = research_panel.copy()

with np.errstate(divide="ignore", invalid="ignore"):
    research_panel["outcome_pnl_per_day"] = (
        research_panel["normalized_pnl_dollars"] / research_panel["actual_dte"]
    )

research_panel.loc[
    ~np.isfinite(research_panel["outcome_pnl_per_day"]),
    "outcome_pnl_per_day",
] = np.nan

tenor_edge_summary = (
    research_panel
    .groupby("tenor")
    .agg(
        historical_trades=("win_clean", "size"),
        historical_win_probability=("win_clean", "mean"),
        avg_pnl_dollars=("normalized_pnl_dollars", "mean"),
        avg_actual_dte=("actual_dte", "mean"),
        avg_pnl_per_day=("outcome_pnl_per_day", "mean"),
        median_pnl_per_day=("outcome_pnl_per_day", "median"),
        total_pnl_dollars=("normalized_pnl_dollars", "sum"),
        total_actual_dte=("actual_dte", "sum"),
        worst_pnl_per_day=("outcome_pnl_per_day", "min"),
    )
    .reset_index()
    .sort_values("tenor")
)

tenor_edge_summary["aggregate_pnl_per_day"] = (
    tenor_edge_summary["total_pnl_dollars"] / tenor_edge_summary["total_actual_dte"]
)

tenor_selection_priority = (
    tenor_edge_summary
    .sort_values(
        [
            "avg_pnl_per_day",
            "aggregate_pnl_per_day",
            "historical_win_probability",
            "tenor",
        ],
        ascending=[False, False, False, False],
    )
    .reset_index(drop=True)
)

tenor_selection_priority["selection_rank"] = np.arange(
    1,
    len(tenor_selection_priority) + 1,
)

tenor_selection_priority = tenor_selection_priority[
    [
        "selection_rank",
        "tenor",
        "avg_pnl_per_day",
        "aggregate_pnl_per_day",
        "historical_win_probability",
        "avg_pnl_dollars",
        "avg_actual_dte",
        "median_pnl_per_day",
        "worst_pnl_per_day",
        "historical_trades",
    ]
]

# Maps used for audit columns in the sweep panel and selected-trade output.
tenor_win_prob_map = dict(
    zip(
        tenor_edge_summary["tenor"].astype(int),
        tenor_edge_summary["historical_win_probability"].astype(float),
    )
)

tenor_avg_pnl_per_day_map = dict(
    zip(
        tenor_edge_summary["tenor"].astype(int),
        tenor_edge_summary["avg_pnl_per_day"].astype(float),
    )
)

tenor_aggregate_pnl_per_day_map = dict(
    zip(
        tenor_edge_summary["tenor"].astype(int),
        tenor_edge_summary["aggregate_pnl_per_day"].astype(float),
    )
)

tenor_selection_rank_map = dict(
    zip(
        tenor_selection_priority["tenor"].astype(int),
        tenor_selection_priority["selection_rank"].astype(int),
    )
)

research_panel["tenor_historical_win_probability"] = (
    research_panel["tenor"].astype(int).map(tenor_win_prob_map)
)

research_panel["tenor_avg_pnl_per_day"] = (
    research_panel["tenor"].astype(int).map(tenor_avg_pnl_per_day_map)
)

research_panel["tenor_aggregate_pnl_per_day"] = (
    research_panel["tenor"].astype(int).map(tenor_aggregate_pnl_per_day_map)
)

research_panel["tenor_selection_rank"] = (
    research_panel["tenor"].astype(int).map(tenor_selection_rank_map)
)

print("Tenor edge summary:")
display(tenor_edge_summary)

print("\nSelection priority by tenor-level average P&L/day:")
display(tenor_selection_priority)

# ---------------------------------------------------------------------
# Sweep panel
# ---------------------------------------------------------------------

sweep_cols = [
    "date",
    "tenor",
    "tenor_group",
    "spx_close",
    "vrp_log",
    "vrp_z_3m",
    "vrp_z_1y",
    "rv21d",
    "rsi14",
    "win_clean",
    "normalized_pnl_dollars",
    "actual_dte",
    "outcome_pnl_per_day",
    "tenor_historical_win_probability",
    "tenor_avg_pnl_per_day",
    "tenor_aggregate_pnl_per_day",
    "tenor_selection_rank",
]

sweep_panel = research_panel[sweep_cols].copy()

sweep_panel["date"] = pd.to_datetime(sweep_panel["date"]).dt.normalize()
sweep_panel["tenor"] = sweep_panel["tenor"].astype(int)
sweep_panel["tenor_group"] = sweep_panel["tenor_group"].astype(str)

numeric_cols = [
    "spx_close",
    "vrp_log",
    "vrp_z_3m",
    "vrp_z_1y",
    "rv21d",
    "rsi14",
    "win_clean",
    "normalized_pnl_dollars",
    "actual_dte",
    "outcome_pnl_per_day",
    "tenor_historical_win_probability",
    "tenor_avg_pnl_per_day",
    "tenor_aggregate_pnl_per_day",
    "tenor_selection_rank",
]

for col in numeric_cols:
    sweep_panel[col] = pd.to_numeric(sweep_panel[col], errors="coerce")

before_drop = len(sweep_panel)

sweep_panel = sweep_panel.dropna(
    subset=[
        "date",
        "tenor",
        "tenor_group",
        "spx_close",
        "vrp_log",
        "vrp_z_3m",
        "vrp_z_1y",
        "rv21d",
        "rsi14",
        "win_clean",
        "normalized_pnl_dollars",
        "actual_dte",
        "outcome_pnl_per_day",
        "tenor_historical_win_probability",
        "tenor_avg_pnl_per_day",
        "tenor_aggregate_pnl_per_day",
        "tenor_selection_rank",
    ]
).copy()

after_drop = len(sweep_panel)

print("\nSweep panel row cleaning")
print("  Rows before drop:", before_drop)
print("  Rows after drop: ", after_drop)
print("  Rows dropped:     ", before_drop - after_drop)

dupes = sweep_panel.duplicated(subset=["date", "tenor"]).sum()
print("  Duplicate date/tenor rows:", dupes)

if dupes > 0:
    display(
        sweep_panel[
            sweep_panel.duplicated(subset=["date", "tenor"], keep=False)
        ].sort_values(["date", "tenor"]).head(30)
    )
    raise ValueError("Duplicate date/tenor rows in sweep_panel.")

# ---------------------------------------------------------------------
# Complete-date filter
# ---------------------------------------------------------------------
#
# Keep only dates where all 9 tenors have completed outcomes.
# This keeps trade-frequency comparisons clean.
# ---------------------------------------------------------------------

REQUIRED_TENOR_COUNT = len(PRODUCTION_TENORS)

tenor_count_by_date = (
    sweep_panel
    .groupby("date")["tenor"]
    .nunique()
    .rename("tenor_count")
)

complete_dates = tenor_count_by_date[tenor_count_by_date == REQUIRED_TENOR_COUNT].index
incomplete_dates = tenor_count_by_date[tenor_count_by_date != REQUIRED_TENOR_COUNT]

print("\nComplete-date filter")
print("  Original sweep rows:", len(sweep_panel))
print("  Original sweep dates:", sweep_panel["date"].nunique())
print("  Complete dates:", len(complete_dates))
print("  Incomplete dates:", len(incomplete_dates))

if len(incomplete_dates) > 0:
    print("\nIncomplete date summary:")
    display(
        incomplete_dates
        .reset_index()
        .sort_values("date")
    )

sweep_panel = sweep_panel[sweep_panel["date"].isin(complete_dates)].copy()
sweep_panel = sweep_panel.sort_values(["date", "tenor"]).reset_index(drop=True)

sweep_dates = pd.Index(sorted(sweep_panel["date"].unique()))
num_sweep_dates = len(sweep_dates)
TRADE_FREQUENCY_DENOMINATOR = num_sweep_dates

print("\nClean sweep panel")
print("  Rows:", len(sweep_panel))
print("  Dates:", num_sweep_dates)
print("  Date range:", sweep_dates.min(), "to", sweep_dates.max())
print("  Trade frequency denominator:", TRADE_FREQUENCY_DENOMINATOR)

clean_tenor_count_by_date = sweep_panel.groupby("date")["tenor"].nunique()

if not (clean_tenor_count_by_date == REQUIRED_TENOR_COUNT).all():
    raise ValueError("Some remaining dates do not have all 9 tenors.")

print("\nRows by tenor group after complete-date filter:")
display(
    sweep_panel
    .groupby("tenor_group")
    .agg(
        rows=("win_clean", "size"),
        win_rate=("win_clean", "mean"),
        avg_vrp=("vrp_log", "mean"),
        avg_z3m=("vrp_z_3m", "mean"),
        avg_z1y=("vrp_z_1y", "mean"),
        avg_rv21d=("rv21d", "mean"),
        avg_rsi14=("rsi14", "mean"),
        avg_pnl_per_day=("outcome_pnl_per_day", "mean"),
        total_pnl_dollars=("normalized_pnl_dollars", "sum"),
        total_actual_dte=("actual_dte", "sum"),
    )
    .assign(
        aggregate_pnl_per_day=lambda x: x["total_pnl_dollars"] / x["total_actual_dte"]
    )
    .reset_index()
)

# ---------------------------------------------------------------------
# Matrix representation
# ---------------------------------------------------------------------
#
# All strategy logic below uses matrix arrays:
#   rows = dates
#   columns = tenors [9, 12, ..., 33]
# ---------------------------------------------------------------------

TENOR_ARRAY = np.array(PRODUCTION_TENORS, dtype=int)
TENOR_TO_COL = {int(t): i for i, t in enumerate(TENOR_ARRAY)}

GROUP_COLS = {
    group: [TENOR_TO_COL[t] for t in tenors]
    for group, tenors in GROUP_TENORS.items()
}


def pivot_to_matrix(value_col):
    mat_df = (
        sweep_panel
        .pivot(index="date", columns="tenor", values=value_col)
        .reindex(index=sweep_dates, columns=PRODUCTION_TENORS)
    )

    if mat_df.isna().any().any():
        bad = mat_df[mat_df.isna().any(axis=1)].head(10)
        display(bad)
        raise ValueError(f"Matrix for {value_col} contains missing values.")

    return mat_df.to_numpy()


spx_mat = pivot_to_matrix("spx_close")
vrp_mat = pivot_to_matrix("vrp_log")
z3m_mat = pivot_to_matrix("vrp_z_3m")
z1y_mat = pivot_to_matrix("vrp_z_1y")
win_mat = pivot_to_matrix("win_clean")
pnl_mat = pivot_to_matrix("normalized_pnl_dollars")
actual_dte_mat = pivot_to_matrix("actual_dte")
pnl_per_day_mat = pivot_to_matrix("outcome_pnl_per_day")
tenor_win_prob_mat = pivot_to_matrix("tenor_historical_win_probability")
tenor_avg_pnl_per_day_mat = pivot_to_matrix("tenor_avg_pnl_per_day")
tenor_aggregate_pnl_per_day_mat = pivot_to_matrix("tenor_aggregate_pnl_per_day")
tenor_selection_rank_mat = pivot_to_matrix("tenor_selection_rank")

rv21d_by_date = (
    sweep_panel.groupby("date")["rv21d"].first().reindex(sweep_dates).to_numpy()
)

rsi_by_date = (
    sweep_panel.groupby("date")["rsi14"].first().reindex(sweep_dates).to_numpy()
)

# Highest tenor-level average P&L/day gets first priority.
selection_priority = tenor_selection_priority.copy()

PRIORITY_COLS = [
    TENOR_TO_COL[int(t)]
    for t in selection_priority["tenor"].tolist()
]

print("\nSelection priority rule:")
print("  Primary metric: tenor-level average P&L/day")
print("  Tie-breakers: aggregate P&L/day, historical win probability, tenor")
print("\nPriority tenor order:", [int(TENOR_ARRAY[c]) for c in PRIORITY_COLS])


Tenor edge summary:


,tenor,historical_trades,historical_win_probability,avg_pnl_dollars,avg_actual_dte,avg_pnl_per_day,median_pnl_per_day,total_pnl_dollars,total_actual_dte,worst_pnl_per_day,aggregate_pnl_per_day
0,9,1742,0.731343,"1,123.741635",8.940299,126.165184,725.663653,"1,957,557.928546",15574,"-16,736.346509",125.693973
1,12,1739,0.742381,"1,816.837565",11.848189,145.025903,652.371475,"3,159,480.525123",20604,"-16,736.346509",153.343066
2,15,1737,0.748417,"2,462.510510",15.940127,155.694642,572.381029,"4,277,380.755638",27688,"-15,066.914368",154.485003
3,18,1735,0.753890,"2,648.826727",17.438040,150.500320,551.048239,"4,595,714.371777",30255,"-15,066.914368",151.899335
4,21,1732,0.760393,"3,264.124558",21.727483,150.908577,504.336275,"5,653,463.734222",37632,"-12,731.522681",150.230223
5,24,1731,0.767187,"3,535.662513",23.038128,152.159027,494.118969,"6,120,231.809378",39879,"-10,503.519959",153.470042
6,27,1728,0.785301,"4,430.317093",27.284722,162.845537,464.462760,"7,655,587.936270",47148,"-10,503.519959",162.373546
7,30,1728,0.788773,"4,900.713063",29.941551,163.462803,448.958391,"8,468,432.173667",51739,"-10,491.241962",163.675992
8,33,1725,0.793043,"5,573.755492",32.848116,169.290389,439.077469,"9,614,728.223532",56663,"-10,206.084590",169.682654



Selection priority by tenor-level average P&L/day:


,selection_rank,tenor,avg_pnl_per_day,aggregate_pnl_per_day,historical_win_probability,avg_pnl_dollars,avg_actual_dte,median_pnl_per_day,worst_pnl_per_day,historical_trades
0,1,33,169.290389,169.682654,0.793043,"5,573.755492",32.848116,439.077469,"-10,206.084590",1725
1,2,30,163.462803,163.675992,0.788773,"4,900.713063",29.941551,448.958391,"-10,491.241962",1728
2,3,27,162.845537,162.373546,0.785301,"4,430.317093",27.284722,464.462760,"-10,503.519959",1728
3,4,15,155.694642,154.485003,0.748417,"2,462.510510",15.940127,572.381029,"-15,066.914368",1737
4,5,24,152.159027,153.470042,0.767187,"3,535.662513",23.038128,494.118969,"-10,503.519959",1731
5,6,21,150.908577,150.230223,0.760393,"3,264.124558",21.727483,504.336275,"-12,731.522681",1732
6,7,18,150.500320,151.899335,0.753890,"2,648.826727",17.438040,551.048239,"-15,066.914368",1735
7,8,12,145.025903,153.343066,0.742381,"1,816.837565",11.848189,652.371475,"-16,736.346509",1739
8,9,9,126.165184,125.693973,0.731343,"1,123.741635",8.940299,725.663653,"-16,736.346509",1742



Sweep panel row cleaning
  Rows before drop: 15597
  Rows after drop:  15597
  Rows dropped:      0
  Duplicate date/tenor rows: 0

Complete-date filter
  Original sweep rows: 15597
  Original sweep dates: 1742
  Complete dates: 1725
  Incomplete dates: 17

Incomplete date summary:


,date,tenor_count
0,2026-05-20,8
1,2026-05-21,8
2,2026-05-22,8
3,2026-05-26,6
4,2026-05-27,6
5,2026-05-28,6
6,2026-05-29,5
7,2026-06-01,4
8,2026-06-02,4
9,2026-06-03,4



Clean sweep panel
  Rows: 15525
  Dates: 1725
  Date range: 2019-07-10 00:00:00 to 2026-05-19 00:00:00
  Trade frequency denominator: 1725

Rows by tenor group after complete-date filter:


,tenor_group,rows,win_rate,avg_vrp,avg_z3m,avg_z1y,avg_rv21d,avg_rsi14,avg_pnl_per_day,total_pnl_dollars,total_actual_dte,aggregate_pnl_per_day
0,back,5175,0.788792,0.508307,0.068748,0.076143,16.720316,55.807832,164.798276,"25,653,165.719078",155382,165.097410
1,front,5175,0.743188,0.667672,0.071348,0.087078,16.720316,55.807832,144.656581,"9,471,878.998276",63373,149.462374
2,middle,5175,0.760580,0.524071,0.068264,0.079054,16.720316,55.807832,150.822859,"16,247,873.322091",107302,151.421906



Selection priority rule:
  Primary metric: tenor-level average P&L/day
  Tie-breakers: aggregate P&L/day, historical win probability, tenor

Priority tenor order: [33, 30, 27, 15, 24, 21, 18, 12, 9]


## Locked 2621 parameters

Candidate **2621** is the locked research winner.

### Rules

- Use front / middle / back tenor buckets.
- Do **not** fit a line across the 9 tenors.
- Check **Core first**.
- If Core qualifies, take the highest-priority qualifying tenor.
- Tenor priority is based on **tenor-level average P&L/day**, not raw average P&L or tenor length.
- Tie-breakers are aggregate P&L/day, historical win probability, then tenor.
- If Core does not qualify, check **Secondary**.
- If Secondary qualifies, take the highest-priority qualifying tenor using the same P&L/day priority rule.
- Maximum one selected trade per date.
- Trade frequency denominator is the number of complete-date rows.

### Locked thresholds

| Layer | Parameter | Front | Middle | Back |
|---|---|---:|---:|---:|
| Core | VRP | 0.60 | 0.65 | 0.70 |
| Core | z3m | 0.55 | 0.75 | 0.75 |
| Core | z1y | 0.65 | 0.65 | 0.75 |
| Core | RSI cap | 70 | 68 | 66 |
| Core | RV21D floor | 8.5 | 8.5 | 8.5 |
| Secondary | VRP | 0.60 | 0.60 | 0.70 |
| Secondary | z3m | 0.50 | 0.50 | 0.50 |
| Secondary | z1y | 0.40 | 0.50 | 0.50 |
| Secondary | RSI cap | 74 | 70 | 68 |
| Secondary | RV21D floor | 6.5 | 6.5 | 6.5 |


In [6]:
# Cell 6 — Locked 2621 parameter dictionary

LOCKED_2621 = {
    "candidate_label": LOCKED_CANDIDATE_LABEL,

    "core": {
        "front": {
            "vrp": 0.60,
            "z3m": 0.55,
            "z1y": 0.65,
            "rsi_cap": 70.0,
            "rv21d_floor": 8.5,
        },
        "middle": {
            "vrp": 0.65,
            "z3m": 0.75,
            "z1y": 0.65,
            "rsi_cap": 68.0,
            "rv21d_floor": 8.5,
        },
        "back": {
            "vrp": 0.70,
            "z3m": 0.75,
            "z1y": 0.75,
            "rsi_cap": 66.0,
            "rv21d_floor": 8.5,
        },
    },

    "secondary": {
        "front": {
            "vrp": 0.60,
            "z3m": 0.50,
            "z1y": 0.40,
            "rsi_cap": 74.0,
            "rv21d_floor": 6.5,
        },
        "middle": {
            "vrp": 0.60,
            "z3m": 0.50,
            "z1y": 0.50,
            "rsi_cap": 70.0,
            "rv21d_floor": 6.5,
        },
        "back": {
            "vrp": 0.70,
            "z3m": 0.50,
            "z1y": 0.50,
            "rsi_cap": 68.0,
            "rv21d_floor": 6.5,
        },
    },
}


def candidate_params_to_table(candidate_params):
    rows = []

    for layer in ["core", "secondary"]:
        for group in ["front", "middle", "back"]:
            p = candidate_params[layer][group]

            rows.append({
                "candidate": candidate_params.get("candidate_label", ""),
                "layer": layer,
                "tenor_group": group,
                "tenors": ", ".join(map(str, GROUP_TENORS[group])),
                "vrp_threshold": p["vrp"],
                "z3m_threshold": p["z3m"],
                "z1y_threshold": p["z1y"],
                "rsi_cap": p["rsi_cap"],
                "rv21d_floor": p["rv21d_floor"],
            })

    return pd.DataFrame(rows)


locked_2621_parameter_table = candidate_params_to_table(LOCKED_2621)

display(locked_2621_parameter_table)

,candidate,layer,tenor_group,tenors,vrp_threshold,z3m_threshold,z1y_threshold,rsi_cap,rv21d_floor
0,locked_core_secondary_2621,core,front,"9, 12, 15",0.600000,0.550000,0.650000,70.000000,8.500000
1,locked_core_secondary_2621,core,middle,"18, 21, 24",0.650000,0.750000,0.650000,68.000000,8.500000
2,locked_core_secondary_2621,core,back,"27, 30, 33",0.700000,0.750000,0.750000,66.000000,8.500000
3,locked_core_secondary_2621,secondary,front,"9, 12, 15",0.600000,0.500000,0.400000,74.000000,6.500000
4,locked_core_secondary_2621,secondary,middle,"18, 21, 24",0.600000,0.500000,0.500000,70.000000,6.500000
5,locked_core_secondary_2621,secondary,back,"27, 30, 33",0.700000,0.500000,0.500000,68.000000,6.500000


In [7]:
# Cell 7 — Core/Secondary selection and evaluation utilities
#
# These functions are intentionally reusable. Future tests should modify the
# parameter dictionary and call evaluate_core_secondary_candidate().
#
# No line-fitting or exploratory grid-search code lives in this notebook.

REQUIRED_LAYER_KEYS = ["vrp", "z3m", "z1y", "rsi_cap", "rv21d_floor"]


def validate_layer_params(layer_params, layer_name):
    for group in ["front", "middle", "back"]:
        if group not in layer_params:
            raise ValueError(f"{layer_name} missing group: {group}")

        missing = [k for k in REQUIRED_LAYER_KEYS if k not in layer_params[group]]
        if missing:
            raise ValueError(f"{layer_name}.{group} missing keys: {missing}")


def validate_core_secondary_candidate(candidate_params):
    """
    Validate basic candidate shape and monotonic strictness.

    Core must be stricter than or equal to Secondary:
      - VRP, z3m, z1y, RV floors: Core >= Secondary
      - RSI cap: Core <= Secondary
    """
    validate_layer_params(candidate_params["core"], "core")
    validate_layer_params(candidate_params["secondary"], "secondary")

    for group in ["front", "middle", "back"]:
        core = candidate_params["core"][group]
        secondary = candidate_params["secondary"][group]

        for key in ["vrp", "z3m", "z1y", "rv21d_floor"]:
            if core[key] < secondary[key]:
                raise ValueError(
                    f"Core must be stricter than Secondary for {group}.{key}. "
                    f"Core={core[key]}, Secondary={secondary[key]}"
                )

        if core["rsi_cap"] > secondary["rsi_cap"]:
            raise ValueError(
                f"Core RSI cap must be lower/tighter than Secondary for {group}. "
                f"Core={core['rsi_cap']}, Secondary={secondary['rsi_cap']}"
            )


def build_selected_cols_for_layer(layer_params):
    """
    Build selected tenor column by date for a single layer.

    Return:
      selected_col_by_date:
        int array length num_sweep_dates
        -1 means no qualifying tenor
        otherwise column index into TENOR_ARRAY
    """
    qualifies = np.zeros((num_sweep_dates, len(TENOR_ARRAY)), dtype=bool)

    for col, tenor in enumerate(TENOR_ARRAY):
        group = TENOR_GROUP_MAP[int(tenor)]
        p = layer_params[group]

        qualifies[:, col] = (
            (vrp_mat[:, col] >= float(p["vrp"])) &
            (z3m_mat[:, col] >= float(p["z3m"])) &
            (z1y_mat[:, col] >= float(p["z1y"])) &
            (rsi_by_date <= float(p["rsi_cap"])) &
            (rv21d_by_date >= float(p["rv21d_floor"]))
        )

    selected_col_by_date = np.full(num_sweep_dates, -1, dtype=np.int8)

    for col in PRIORITY_COLS:
        take = (selected_col_by_date == -1) & qualifies[:, col]
        selected_col_by_date[take] = col

    return selected_col_by_date


def summarize_selected_cols(selected_col_by_date):
    """
    Summary metrics for a selected-col array.
    """
    trade_mask = selected_col_by_date >= 0
    trade_count = int(trade_mask.sum())

    selected_tenor_counts_empty = {
        f"selected_tenor_{int(t)}_count": 0
        for t in TENOR_ARRAY
    }

    if trade_count == 0:
        return {
            "trade_count": 0,
            "trade_frequency": 0.0,
            "win_rate": np.nan,
            "avg_pnl_per_day": np.nan,
            "median_pnl_per_day": np.nan,
            "aggregate_pnl_per_day": np.nan,
            "worst_pnl_per_day": np.nan,
            "avg_actual_dte": np.nan,
            "avg_selected_tenor": np.nan,
            "front_count": 0,
            "middle_count": 0,
            "back_count": 0,
            **selected_tenor_counts_empty,
        }

    row_idx = np.where(trade_mask)[0]
    col_idx = selected_col_by_date[trade_mask].astype(int)

    selected_wins = win_mat[row_idx, col_idx]
    selected_pnls = pnl_mat[row_idx, col_idx]
    selected_dtes = actual_dte_mat[row_idx, col_idx]
    selected_pnl_per_day = pnl_per_day_mat[row_idx, col_idx]
    selected_tenors = TENOR_ARRAY[col_idx]

    selected_tenor_counts = {
        f"selected_tenor_{int(t)}_count": int((selected_tenors == t).sum())
        for t in TENOR_ARRAY
    }

    return {
        "trade_count": trade_count,
        "trade_frequency": float(trade_count / TRADE_FREQUENCY_DENOMINATOR),
        "win_rate": float(np.nanmean(selected_wins)),

        "avg_pnl_per_day": float(np.nanmean(selected_pnl_per_day)),
        "median_pnl_per_day": float(np.nanmedian(selected_pnl_per_day)),
        "aggregate_pnl_per_day": float(np.nansum(selected_pnls) / np.nansum(selected_dtes)),
        "worst_pnl_per_day": float(np.nanmin(selected_pnl_per_day)),

        "avg_actual_dte": float(np.nanmean(selected_dtes)),
        "avg_selected_tenor": float(np.nanmean(selected_tenors)),

        "front_count": int(np.isin(selected_tenors, GROUP_TENORS["front"]).sum()),
        "middle_count": int(np.isin(selected_tenors, GROUP_TENORS["middle"]).sum()),
        "back_count": int(np.isin(selected_tenors, GROUP_TENORS["back"]).sum()),

        **selected_tenor_counts,
    }


def combine_core_secondary(core_selected, secondary_raw_selected):
    """
    Core gets first priority. Secondary only fills dates where Core does not trade.
    """
    layer = np.full(num_sweep_dates, "None", dtype=object)

    final_selected = np.where(
        core_selected >= 0,
        core_selected,
        secondary_raw_selected,
    ).astype(np.int8)

    layer[core_selected >= 0] = "Core"

    secondary_only_mask = (core_selected < 0) & (secondary_raw_selected >= 0)
    layer[secondary_only_mask] = "Secondary"

    secondary_only_selected = np.where(
        secondary_only_mask,
        secondary_raw_selected,
        -1,
    ).astype(np.int8)

    return final_selected, secondary_only_selected, layer


def selected_cols_to_trades(selected_col_by_date, layer_by_date=None, candidate_label="candidate"):
    """
    Convert selected columns into a trade-level audit DataFrame.
    """
    trade_mask = selected_col_by_date >= 0

    if layer_by_date is None:
        layer_by_date = np.full(num_sweep_dates, "Selected", dtype=object)

    row_idx = np.where(trade_mask)[0]
    col_idx = selected_col_by_date[trade_mask].astype(int)

    trades = pd.DataFrame({
        "candidate": candidate_label,
        "date": sweep_dates[row_idx],
        "layer": layer_by_date[trade_mask],
        "selected_col": col_idx,
        "selected_tenor": TENOR_ARRAY[col_idx],
        "tenor_group": [TENOR_GROUP_MAP[int(t)] for t in TENOR_ARRAY[col_idx]],

        "spx_close": spx_mat[row_idx, col_idx],
        "rv21d": rv21d_by_date[row_idx],
        "rsi14": rsi_by_date[row_idx],

        "vrp_log": vrp_mat[row_idx, col_idx],
        "vrp_z_3m": z3m_mat[row_idx, col_idx],
        "vrp_z_1y": z1y_mat[row_idx, col_idx],

        "tenor_selection_rank": tenor_selection_rank_mat[row_idx, col_idx],
        "tenor_avg_pnl_per_day": tenor_avg_pnl_per_day_mat[row_idx, col_idx],
        "tenor_aggregate_pnl_per_day": tenor_aggregate_pnl_per_day_mat[row_idx, col_idx],
        "tenor_historical_win_probability": tenor_win_prob_mat[row_idx, col_idx],
        "win": win_mat[row_idx, col_idx],
        "normalized_pnl_dollars": pnl_mat[row_idx, col_idx],
        "actual_dte": actual_dte_mat[row_idx, col_idx],
        "pnl_per_day": pnl_per_day_mat[row_idx, col_idx],
    })

    trades = trades.sort_values("date").reset_index(drop=True)
    trades["trade_sequence"] = np.arange(1, len(trades) + 1)

    trades["cum_pnl"] = trades["normalized_pnl_dollars"].cumsum()
    trades["running_max_cum_pnl"] = trades["cum_pnl"].cummax()
    trades["drawdown"] = trades["cum_pnl"] - trades["running_max_cum_pnl"]
    trades["rolling_20_trade_pnl"] = trades["normalized_pnl_dollars"].rolling(20).sum()

    return trades


def summarize_trade_frame(df, label=None, denominator=TRADE_FREQUENCY_DENOMINATOR):
    """
    Summary metrics for any trade-level subset.
    """
    n = len(df)

    if n == 0:
        return {
            "label": label,
            "trade_count": 0,
            "trade_frequency": 0.0,
            "win_rate": np.nan,
            "avg_pnl_per_day": np.nan,
            "median_pnl_per_day": np.nan,
            "aggregate_pnl_per_day": np.nan,
            "total_pnl": 0.0,
            "total_actual_dte": 0.0,
            "worst_trade_pnl": np.nan,
            "worst_pnl_per_day": np.nan,
            "max_drawdown": np.nan,
            "worst_20_trade_sum": np.nan,
            "avg_selected_tenor": np.nan,
        }

    total_pnl = float(df["normalized_pnl_dollars"].sum())
    total_dte = float(df["actual_dte"].sum())

    # Drawdown is meaningful only for date-ordered full paths.
    path = df.sort_values("date").copy()
    cum_pnl = path["normalized_pnl_dollars"].cumsum()
    drawdown = cum_pnl - cum_pnl.cummax()
    rolling_20 = path["normalized_pnl_dollars"].rolling(20).sum()

    return {
        "label": label,
        "trade_count": int(n),
        "trade_frequency": float(n / denominator),
        "win_rate": float(df["win"].mean()),
        "avg_pnl_per_day": float(df["pnl_per_day"].mean()),
        "median_pnl_per_day": float(df["pnl_per_day"].median()),
        "aggregate_pnl_per_day": float(total_pnl / total_dte) if total_dte else np.nan,
        "total_pnl": total_pnl,
        "total_actual_dte": total_dte,
        "worst_trade_pnl": float(df["normalized_pnl_dollars"].min()),
        "worst_pnl_per_day": float(df["pnl_per_day"].min()),
        "max_drawdown": float(drawdown.min()),
        "worst_20_trade_sum": float(rolling_20.min()) if rolling_20.notna().any() else np.nan,
        "avg_selected_tenor": float(df["selected_tenor"].mean()),
    }


def evaluate_core_secondary_candidate(candidate_params, candidate_label=None):
    """
    Evaluate a Core/Secondary candidate.

    Returns:
      result_summary: one-row DataFrame-friendly dict
      selected_outputs: selected arrays
      trades: selected trade DataFrame with Core / Secondary layer labels
    """
    validate_core_secondary_candidate(candidate_params)

    if candidate_label is None:
        candidate_label = candidate_params.get("candidate_label", "candidate")

    core_selected = build_selected_cols_for_layer(candidate_params["core"])
    secondary_raw_selected = build_selected_cols_for_layer(candidate_params["secondary"])

    combined_selected, secondary_only_selected, layer_by_date = combine_core_secondary(
        core_selected,
        secondary_raw_selected,
    )

    core_metrics = summarize_selected_cols(core_selected)
    secondary_only_metrics = summarize_selected_cols(secondary_only_selected)
    combined_metrics = summarize_selected_cols(combined_selected)

    combined_trade_count = combined_metrics["trade_count"]

    if combined_trade_count > 0:
        core_share = core_metrics["trade_count"] / combined_trade_count
        secondary_share = secondary_only_metrics["trade_count"] / combined_trade_count
    else:
        core_share = np.nan
        secondary_share = np.nan

    result = {
        "candidate": candidate_label,

        "core_trade_count": int(core_metrics["trade_count"]),
        "core_trade_frequency": float(core_metrics["trade_frequency"]),
        "core_win_rate": float(core_metrics["win_rate"]),
        "core_avg_pnl_per_day": float(core_metrics["avg_pnl_per_day"]),
        "core_aggregate_pnl_per_day": float(core_metrics["aggregate_pnl_per_day"]),
        "core_worst_pnl_per_day": float(core_metrics["worst_pnl_per_day"]),

        "secondary_only_trade_count": int(secondary_only_metrics["trade_count"]),
        "secondary_only_trade_frequency": float(secondary_only_metrics["trade_frequency"]),
        "secondary_only_win_rate": float(secondary_only_metrics["win_rate"]),
        "secondary_only_avg_pnl_per_day": float(secondary_only_metrics["avg_pnl_per_day"]),
        "secondary_only_aggregate_pnl_per_day": float(secondary_only_metrics["aggregate_pnl_per_day"]),
        "secondary_only_worst_pnl_per_day": float(secondary_only_metrics["worst_pnl_per_day"]),

        "combined_trade_count": int(combined_metrics["trade_count"]),
        "combined_trade_frequency": float(combined_metrics["trade_frequency"]),
        "combined_win_rate": float(combined_metrics["win_rate"]),
        "combined_avg_pnl_per_day": float(combined_metrics["avg_pnl_per_day"]),
        "combined_median_pnl_per_day": float(combined_metrics["median_pnl_per_day"]),
        "combined_aggregate_pnl_per_day": float(combined_metrics["aggregate_pnl_per_day"]),
        "combined_worst_pnl_per_day": float(combined_metrics["worst_pnl_per_day"]),
        "combined_avg_actual_dte": float(combined_metrics["avg_actual_dte"]),
        "combined_avg_selected_tenor": float(combined_metrics["avg_selected_tenor"]),

        "core_share_of_combined": float(core_share),
        "secondary_share_of_combined": float(secondary_share),

        "combined_front_count": int(combined_metrics["front_count"]),
        "combined_middle_count": int(combined_metrics["middle_count"]),
        "combined_back_count": int(combined_metrics["back_count"]),
    }

    for tenor in TENOR_ARRAY:
        result[f"combined_selected_tenor_{int(tenor)}_count"] = int(
            combined_metrics[f"selected_tenor_{int(tenor)}_count"]
        )

    trades = selected_cols_to_trades(
        combined_selected,
        layer_by_date=layer_by_date,
        candidate_label=candidate_label,
    )

    selected_outputs = {
        "core_selected": core_selected,
        "secondary_raw_selected": secondary_raw_selected,
        "secondary_only_selected": secondary_only_selected,
        "combined_selected": combined_selected,
        "layer_by_date": layer_by_date,
    }

    return result, selected_outputs, trades


In [8]:
# Cell 8 — Evaluate locked 2621 and compare to benchmark

locked_2621_result, locked_2621_selected_outputs, locked_2621_trades = evaluate_core_secondary_candidate(
    LOCKED_2621,
    candidate_label=LOCKED_CANDIDATE_LABEL,
)

locked_2621_summary = pd.DataFrame([locked_2621_result])

summary_display_cols = [
    "candidate",

    "core_trade_count",
    "core_trade_frequency",
    "core_win_rate",

    "secondary_only_trade_count",
    "secondary_only_trade_frequency",
    "secondary_only_win_rate",

    "combined_trade_count",
    "combined_trade_frequency",
    "combined_win_rate",

    "core_share_of_combined",
    "secondary_share_of_combined",

    "combined_avg_pnl_per_day",
    "combined_median_pnl_per_day",
    "combined_aggregate_pnl_per_day",
    "combined_worst_pnl_per_day",
    "combined_avg_selected_tenor",

    "combined_front_count",
    "combined_middle_count",
    "combined_back_count",

    "combined_selected_tenor_9_count",
    "combined_selected_tenor_12_count",
    "combined_selected_tenor_15_count",
    "combined_selected_tenor_18_count",
    "combined_selected_tenor_21_count",
    "combined_selected_tenor_24_count",
    "combined_selected_tenor_27_count",
    "combined_selected_tenor_30_count",
    "combined_selected_tenor_33_count",
]

print("Locked 2621 summary:")
display(locked_2621_summary[summary_display_cols])

# ---------------------------------------------------------------------
# Benchmark drift check
# ---------------------------------------------------------------------
#
# This cell is deliberately non-fatal. It warns if values differ from the
# research benchmark, but does not fail the run. The feature/outcome panel may
# legitimately change when new history is appended.
# ---------------------------------------------------------------------

benchmark_rows = []

for key, expected_value in EXPECTED_2621_BENCHMARK.items():
    actual_value = locked_2621_result.get(key, np.nan)

    benchmark_rows.append({
        "metric": key,
        "expected_research_value": expected_value,
        "actual_current_value": actual_value,
        "difference": actual_value - expected_value if pd.notna(actual_value) else np.nan,
    })

benchmark_check = pd.DataFrame(benchmark_rows)

print("\nBenchmark drift check:")

if len(benchmark_check) == 0:
    print("  No fixed benchmark values are enforced in this cleaned notebook.")
    print("  The current run becomes the source-of-truth benchmark under the P&L/day tenor-priority rule.")
else:
    display(benchmark_check)

    max_abs_frequency_or_win_diff = (
        benchmark_check[
            benchmark_check["metric"].str.contains("frequency|win_rate|share", regex=True)
        ]["difference"]
        .abs()
        .max()
    )

    if pd.notna(max_abs_frequency_or_win_diff) and max_abs_frequency_or_win_diff > 0.0025:
        print(
            "\nWARNING: Current results differ from the locked research benchmark. "
            "This may be expected if upstream panels have been updated."
        )
    else:
        print("\nBenchmark check passed or drift is immaterial.")


Locked 2621 summary:


,candidate,core_trade_count,core_trade_frequency,core_win_rate,secondary_only_trade_count,secondary_only_trade_frequency,secondary_only_win_rate,combined_trade_count,combined_trade_frequency,combined_win_rate,core_share_of_combined,secondary_share_of_combined,combined_avg_pnl_per_day,combined_median_pnl_per_day,combined_aggregate_pnl_per_day,combined_worst_pnl_per_day,combined_avg_selected_tenor,combined_front_count,combined_middle_count,combined_back_count,combined_selected_tenor_9_count,combined_selected_tenor_12_count,combined_selected_tenor_15_count,combined_selected_tenor_18_count,combined_selected_tenor_21_count,combined_selected_tenor_24_count,combined_selected_tenor_27_count,combined_selected_tenor_30_count,combined_selected_tenor_33_count
0,locked_core_secondary_2621,386,0.223768,0.867876,191,0.110725,0.780105,577,0.334493,0.838821,0.668977,0.331023,392.541525,516.494466,380.966423,"-5,535.611173",23.890815,245,26,306,53,44,148,8,2,16,10,16,280



Benchmark drift check:
  No fixed benchmark values are enforced in this cleaned notebook.
  The current run becomes the source-of-truth benchmark under the P&L/day tenor-priority rule.


In [9]:
# Cell 9 — Locked 2621 audit tables

# ---------------------------------------------------------------------
# Path-level drawdown and worst clusters
# ---------------------------------------------------------------------

locked_2621_path_summary = pd.DataFrame([
    summarize_trade_frame(
        locked_2621_trades,
        label="Combined",
        denominator=TRADE_FREQUENCY_DENOMINATOR,
    )
])

locked_2621_summary_by_layer = pd.DataFrame(
    [
        summarize_trade_frame(
            df,
            label=layer,
            denominator=TRADE_FREQUENCY_DENOMINATOR,
        )
        for layer, df in locked_2621_trades.groupby("layer", sort=False)
    ]
)

locked_2621_trades["year"] = pd.to_datetime(locked_2621_trades["date"]).dt.year
locked_2621_trades["month"] = pd.to_datetime(locked_2621_trades["date"]).dt.to_period("M").astype(str)

locked_2621_summary_by_year_layer = pd.DataFrame(
    [
        {
            "year": year,
            "layer": layer,
            **summarize_trade_frame(
                df,
                label=f"{year}_{layer}",
                denominator=TRADE_FREQUENCY_DENOMINATOR,
            ),
        }
        for (year, layer), df in locked_2621_trades.groupby(["year", "layer"])
    ]
).drop(columns=["label"], errors="ignore")

locked_2621_summary_by_tenor_layer = pd.DataFrame(
    [
        {
            "selected_tenor": tenor,
            "layer": layer,
            **summarize_trade_frame(
                df,
                label=f"{tenor}_{layer}",
                denominator=TRADE_FREQUENCY_DENOMINATOR,
            ),
        }
        for (tenor, layer), df in locked_2621_trades.groupby(["selected_tenor", "layer"])
    ]
).drop(columns=["label"], errors="ignore")

locked_2621_summary_by_group_layer = pd.DataFrame(
    [
        {
            "tenor_group": tenor_group,
            "layer": layer,
            **summarize_trade_frame(
                df,
                label=f"{tenor_group}_{layer}",
                denominator=TRADE_FREQUENCY_DENOMINATOR,
            ),
        }
        for (tenor_group, layer), df in locked_2621_trades.groupby(["tenor_group", "layer"])
    ]
).drop(columns=["label"], errors="ignore")

locked_2621_worst_trades = (
    locked_2621_trades
    .sort_values("normalized_pnl_dollars")
    .head(50)
    .reset_index(drop=True)
)

locked_2621_worst_20_trade_windows = (
    locked_2621_trades
    .dropna(subset=["rolling_20_trade_pnl"])
    .sort_values("rolling_20_trade_pnl")
    .head(20)
    .loc[
        :,
        [
            "trade_sequence",
            "date",
            "layer",
            "selected_tenor",
            "tenor_group",
            "normalized_pnl_dollars",
            "rolling_20_trade_pnl",
            "cum_pnl",
            "drawdown",
        ],
    ]
    .reset_index(drop=True)
)

print("Locked 2621 path summary:")
display(locked_2621_path_summary)

print("\nSummary by layer:")
display(locked_2621_summary_by_layer)

print("\nSummary by year and layer:")
display(locked_2621_summary_by_year_layer)

print("\nSummary by tenor and layer:")
display(locked_2621_summary_by_tenor_layer)

print("\nSummary by tenor group and layer:")
display(locked_2621_summary_by_group_layer)

print("\nWorst 20 individual trades:")
display(locked_2621_worst_trades.head(20))

print("\nWorst 20-trade rolling windows:")
display(locked_2621_worst_20_trade_windows)

Locked 2621 path summary:


,label,trade_count,trade_frequency,win_rate,avg_pnl_per_day,median_pnl_per_day,aggregate_pnl_per_day,total_pnl,total_actual_dte,worst_trade_pnl,worst_pnl_per_day,max_drawdown,worst_20_trade_sum,avg_selected_tenor
0,Combined,577,0.334493,0.838821,392.541525,516.494466,380.966423,"5,204,763.265223","13,662.000000","-114,955.549452","-5,535.611173","-340,806.930594","-292,116.140685",23.890815



Summary by layer:


,label,trade_count,trade_frequency,win_rate,avg_pnl_per_day,median_pnl_per_day,aggregate_pnl_per_day,total_pnl,total_actual_dte,worst_trade_pnl,worst_pnl_per_day,max_drawdown,worst_20_trade_sum,avg_selected_tenor
0,Secondary,191,0.110725,0.780105,275.580447,456.181243,227.700547,"948,145.076534","4,164.000000","-87,932.829005","-2,958.021794","-249,162.776157","-176,622.419873",21.973822
1,Core,386,0.223768,0.867876,450.416048,549.684110,448.159422,"4,256,618.188689","9,498.000000","-114,955.549452","-5,535.611173","-326,919.349145","-163,732.644808",24.839378



Summary by year and layer:


...

,tenor_group,layer,trade_count,trade_frequency,win_rate,avg_pnl_per_day,median_pnl_per_day,aggregate_pnl_per_day,total_pnl,total_actual_dte,worst_trade_pnl,worst_pnl_per_day,max_drawdown,worst_20_trade_sum,avg_selected_tenor
0,back,Core,224,0.129855,0.897321,443.054716,503.518604,438.725251,"3,167,157.590040","7,219.000000","-114,955.549452","-3,193.209707","-259,853.641796","-162,587.226670",32.558036
1,back,Secondary,82,0.047536,0.780488,197.148408,410.958827,194.766164,"522,947.149951","2,685.000000","-87,932.829005","-2,512.366543","-121,772.219065","-23,191.281047",32.890244
2,front,Core,146,0.084638,0.815068,457.665695,766.853597,478.282570,"936,477.272144","1,958.000000","-60,891.722904","-5,535.611173","-163,985.642996","-10,100.344738",13.273973
3,front,Secondary,99,0.057391,0.767677,328.228215,542.818908,274.647599,"352,922.165176","1,285.000000","-42,928.530585","-2,958.021794","-111,540.767204","-30,506.645291",13.000000
4,middle,Core,16,0.009275,0.937500,487.321651,463.563477,476.583572,"152,983.326505",321.000000,"-2,002.285028",-91.012956,"-2,002.285028",NaN,22.312500
5,middle,Secondary,10,0.005797,0.900000,397.510268,518.298577,372.555471,"72,275.761407",194.000000,"-33,098.518410","-1,504.478110","-33,098.518410",NaN,21.300000



Worst 20 individual trades:


,candidate,date,layer,selected_col,selected_tenor,tenor_group,spx_close,rv21d,rsi14,vrp_log,vrp_z_3m,vrp_z_1y,tenor_selection_rank,tenor_avg_pnl_per_day,tenor_aggregate_pnl_per_day,tenor_historical_win_probability,win,normalized_pnl_dollars,actual_dte,pnl_per_day,trade_sequence,cum_pnl,running_max_cum_pnl,drawdown,rolling_20_trade_pnl,year,month
0,locked_core_secondary_2621,2025-02-27,Core,8,33,back,"5,861.570000",11.428641,33.832673,0.939687,1.423970,1.059198,1,169.290389,169.682654,0.793043,0.000000,"-114,955.549452",36,"-3,193.209707",461,"4,132,190.567935","4,421,729.248969","-289,538.681035","-138,134.030310",2025,2025-02
1,locked_core_secondary_2621,2020-01-24,Secondary,8,33,back,"3,295.470000",7.866687,61.573549,1.300539,1.061030,1.738160,1,169.290389,169.682654,0.793043,0.000000,"-87,932.829005",35,"-2,512.366543",54,"126,258.296119","214,191.125123","-87,932.829005","101,620.934570",2020,2020-01
2,locked_core_secondary_2621,2020-02-24,Core,0,9,front,"3,225.890000",18.033963,36.699503,1.425272,0.638828,1.491332,9,126.165184,125.693973,0.731343,0.000000,"-60,891.722904",11,"-5,535.611173",60,"128,454.201772","214,191.125123","-85,736.923351","38,589.422511",2020,2020-02
3,locked_core_secondary_2621,2020-02-27,Core,1,12,front,"2,978.760000",24.489208,20.145213,1.447358,0.664102,1.532049,8,145.025903,153.343066,0.742381,0.000000,"-56,963.300165",15,"-3,797.553344",61,"71,490.901607","214,191.125123","-142,700.223516","-24,609.893623",2020,2020-02
4,locked_core_secondary_2621,2026-02-19,Core,8,33,back,"6,861.890000",12.300219,46.444609,1.045459,0.772548,0.800369,1,169.290389,169.682654,0.793043,0.000000,"-52,092.062100",36,"-1,447.001725",552,"4,900,285.192823","4,964,588.670982","-64,303.478158","-16,667.045463",2026,2026-02
5,locked_core_secondary_2621,2025-02-24,Core,7,30,back,"5,983.250000",11.772753,43.422979,0.796970,1.085260,0.761401,2,163.462803,163.675992,0.788773,0.000000,"-51,336.648143",32,"-1,604.270254",458,"4,340,707.561588","4,421,729.248969","-81,021.687382","118,416.659960",2025,2025-02
6,locked_core_secondary_2621,2026-02-23,Core,8,33,back,"6,837.750000",12.262766,44.737667,1.044061,0.791125,0.791481,1,169.290389,169.682654,0.793043,0.000000,"-48,751.416767",32,"-1,523.481774",554,"4,838,684.818605","4,964,588.670982","-125,903.852377","-93,549.873403",2026,2026-02
7,locked_core_secondary_2621,2025-02-26,Core,8,33,back,"5,956.060000",10.753467,41.127776,0.953633,1.522225,1.104877,1,169.290389,169.682654,0.793043,0.000000,"-47,172.123854",30,"-1,572.404128",460,"4,247,146.117386","4,421,729.248969","-174,583.131583","-6,999.089272",2025,2025-02
8,locked_core_secondary_2621,2025-02-25,Core,8,33,back,"5,955.250000",11.824882,41.026556,0.937331,1.503144,1.069750,1,169.290389,169.682654,0.793043,0.000000,"-46,389.320348",31,"-1,496.429689",459,"4,294,318.241240","4,421,729.248969","-127,411.007729","56,645.879399",2025,2025-02
9,locked_core_secondary_2621,2020-02-28,Core,2,15,front,"2,954.220000",24.483900,19.164013,1.440955,0.832745,1.691735,4,155.694642,154.485003,0.748417,0.000000,"-46,130.619927",14,"-3,295.044280",62,"25,360.281680","214,191.125123","-188,830.843443","-76,168.904032",2020,2020-02



Worst 20-trade rolling windows:


,trade_sequence,date,layer,selected_tenor,tenor_group,normalized_pnl_dollars,rolling_20_trade_pnl,cum_pnl,drawdown
0,468,2025-03-10,Core,15,front,"15,449.118007","-292,116.140685","4,100,549.842579","-321,179.406390"
1,467,2025-03-07,Core,15,front,"-1,826.626460","-289,202.630928","4,085,100.724572","-336,628.524397"
2,469,2025-03-11,Core,15,front,"21,051.422541","-287,977.057580","4,121,601.265121","-300,127.983849"
3,466,2025-03-06,Core,15,front,"6,005.032657","-267,982.763301","4,086,927.351032","-334,801.897937"
4,465,2025-03-05,Secondary,15,front,"-13,887.581449","-254,392.496726","4,080,922.318376","-340,806.930594"
5,470,2025-03-13,Secondary,9,front,"13,420.217621","-250,437.176202","4,135,021.482742","-286,707.766227"
6,464,2025-03-04,Core,15,front,"-1,841.419832","-224,680.143661","4,094,809.899824","-326,919.349145"
7,471,2025-04-04,Core,15,front,"31,316.021821","-220,734.086928","4,166,337.504563","-255,391.744407"
8,463,2025-03-03,Core,15,front,"-14,110.077064","-206,826.160654","4,096,651.319656","-325,077.929313"
9,472,2025-04-07,Core,15,front,"36,248.703640","-182,552.580296","4,202,586.208202","-219,143.040767"


In [10]:
# Cell 10 — Save locked 2621 outputs

LOCKED_2621_SELECTED_TRADES_PARQUET = (
    PROCESSED_DIR / "simple_signal_grid_locked_2621_selected_trades_v0_1.parquet"
)

LOCKED_2621_SELECTED_TRADES_CSV = (
    AUDIT_DIR / "simple_signal_grid_locked_2621_selected_trades_v0_1.csv"
)

LOCKED_2621_SUMMARY_CSV = (
    AUDIT_DIR / "simple_signal_grid_locked_2621_summary_v0_1.csv"
)

LOCKED_2621_PARAMETERS_CSV = (
    AUDIT_DIR / "simple_signal_grid_locked_2621_parameters_v0_1.csv"
)

LOCKED_2621_SUMMARY_BY_LAYER_CSV = (
    AUDIT_DIR / "simple_signal_grid_locked_2621_summary_by_layer_v0_1.csv"
)

LOCKED_2621_SUMMARY_BY_YEAR_LAYER_CSV = (
    AUDIT_DIR / "simple_signal_grid_locked_2621_summary_by_year_layer_v0_1.csv"
)

LOCKED_2621_SUMMARY_BY_TENOR_LAYER_CSV = (
    AUDIT_DIR / "simple_signal_grid_locked_2621_summary_by_tenor_layer_v0_1.csv"
)

LOCKED_2621_SUMMARY_BY_GROUP_LAYER_CSV = (
    AUDIT_DIR / "simple_signal_grid_locked_2621_summary_by_group_layer_v0_1.csv"
)

LOCKED_2621_WORST_TRADES_CSV = (
    AUDIT_DIR / "simple_signal_grid_locked_2621_worst_trades_v0_1.csv"
)

LOCKED_2621_WORST_20_WINDOWS_CSV = (
    AUDIT_DIR / "simple_signal_grid_locked_2621_worst_20_trade_windows_v0_1.csv"
)

LOCKED_2621_SUMMARY_JSON = (
    AUDIT_DIR / "simple_signal_grid_locked_2621_summary_v0_1.json"
)

# Save trade-level path.
locked_2621_trades.to_parquet(LOCKED_2621_SELECTED_TRADES_PARQUET, index=False)
locked_2621_trades.to_csv(LOCKED_2621_SELECTED_TRADES_CSV, index=False)

# Save summary/audit tables.
locked_2621_summary.to_csv(LOCKED_2621_SUMMARY_CSV, index=False)
locked_2621_parameter_table.to_csv(LOCKED_2621_PARAMETERS_CSV, index=False)
locked_2621_summary_by_layer.to_csv(LOCKED_2621_SUMMARY_BY_LAYER_CSV, index=False)
locked_2621_summary_by_year_layer.to_csv(LOCKED_2621_SUMMARY_BY_YEAR_LAYER_CSV, index=False)
locked_2621_summary_by_tenor_layer.to_csv(LOCKED_2621_SUMMARY_BY_TENOR_LAYER_CSV, index=False)
locked_2621_summary_by_group_layer.to_csv(LOCKED_2621_SUMMARY_BY_GROUP_LAYER_CSV, index=False)
locked_2621_worst_trades.to_csv(LOCKED_2621_WORST_TRADES_CSV, index=False)
locked_2621_worst_20_trade_windows.to_csv(LOCKED_2621_WORST_20_WINDOWS_CSV, index=False)

summary_payload = {
    "candidate_label": LOCKED_CANDIDATE_LABEL,
    "project_root": str(PROJECT_ROOT),
    "feature_panel_path": str(FEATURE_PANEL_PATH),
    "outcome_path": str(OUTCOME_PATH),
    "trade_frequency_denominator": int(TRADE_FREQUENCY_DENOMINATOR),
    "tenor_selection_priority_metric": TENOR_SELECTION_PRIORITY_METRIC,
    "tenor_selection_priority_rule": "rank qualifying tenors by historical average P&L/day; tie-break by aggregate P&L/day, historical win probability, then tenor",
    "sweep_start_date": str(sweep_dates.min().date()),
    "sweep_end_date": str(sweep_dates.max().date()),
    "parameters": LOCKED_2621,
    "summary": locked_2621_result,
    "expected_research_benchmark": EXPECTED_2621_BENCHMARK,
}

with open(LOCKED_2621_SUMMARY_JSON, "w") as f:
    json.dump(summary_payload, f, indent=2, default=str)

print("Saved locked 2621 outputs:")
print("  ", LOCKED_2621_SELECTED_TRADES_PARQUET)
print("  ", LOCKED_2621_SELECTED_TRADES_CSV)
print("  ", LOCKED_2621_SUMMARY_CSV)
print("  ", LOCKED_2621_PARAMETERS_CSV)
print("  ", LOCKED_2621_SUMMARY_BY_LAYER_CSV)
print("  ", LOCKED_2621_SUMMARY_BY_YEAR_LAYER_CSV)
print("  ", LOCKED_2621_SUMMARY_BY_TENOR_LAYER_CSV)
print("  ", LOCKED_2621_SUMMARY_BY_GROUP_LAYER_CSV)
print("  ", LOCKED_2621_WORST_TRADES_CSV)
print("  ", LOCKED_2621_WORST_20_WINDOWS_CSV)
print("  ", LOCKED_2621_SUMMARY_JSON)


Saved locked 2621 outputs:
   C:\Users\patri\vrp_project\data\processed\simple_signal_grid_locked_2621_selected_trades_v0_1.parquet
   C:\Users\patri\vrp_project\data\audit\simple_signal_grid_locked_2621_selected_trades_v0_1.csv
   C:\Users\patri\vrp_project\data\audit\simple_signal_grid_locked_2621_summary_v0_1.csv
   C:\Users\patri\vrp_project\data\audit\simple_signal_grid_locked_2621_parameters_v0_1.csv
   C:\Users\patri\vrp_project\data\audit\simple_signal_grid_locked_2621_summary_by_layer_v0_1.csv
   C:\Users\patri\vrp_project\data\audit\simple_signal_grid_locked_2621_summary_by_year_layer_v0_1.csv
   C:\Users\patri\vrp_project\data\audit\simple_signal_grid_locked_2621_summary_by_tenor_layer_v0_1.csv
   C:\Users\patri\vrp_project\data\audit\simple_signal_grid_locked_2621_summary_by_group_layer_v0_1.csv
   C:\Users\patri\vrp_project\data\audit\simple_signal_grid_locked_2621_worst_trades_v0_1.csv
   C:\Users\patri\vrp_project\data\audit\simple_signal_grid_locked_2621_worst_20_trade_

## Future testing guide

This notebook is now the **locked 2621 baseline**. Future tests should be incremental and auditable. The within-layer tenor priority is locked to **historical average P&L/day**.

Recommended next tests:

1. **Sizing tests**
   - Core = 1.00x
   - Secondary = 0.50x / 0.75x / 1.00x
   - Compare weighted P&L, max drawdown, and worst rolling trade clusters.

2. **Validation tests**
   - Year-by-year Core vs Secondary.
   - Tenor-level attribution.
   - Worst cluster review.
   - Stress behavior around high-volatility regimes.

3. **New parameter tests**
   - Copy `LOCKED_2621`.
   - Modify one small group of thresholds.
   - Re-run `evaluate_core_secondary_candidate()`.
   - Compare against `locked_2621_summary`.

Avoid reintroducing broad grid sweeps into this notebook. If broad searches are needed again, create a separate research notebook and bring only the winning candidate back here.


In [11]:
# Cell 11 — Optional future-test template
#
# Leave RUN_EXAMPLE_VARIANT_TEST = False for normal locked-candidate runs.
# To test a small incremental change, copy LOCKED_2621, edit one or two values,
# and compare the result to locked_2621_summary.

RUN_EXAMPLE_VARIANT_TEST = False

if RUN_EXAMPLE_VARIANT_TEST:
    test_candidate = deepcopy(LOCKED_2621)
    test_candidate["candidate_label"] = "example_variant"

    # Example edit:
    # test_candidate["secondary"]["front"]["rsi_cap"] = 73.0

    test_result, test_selected_outputs, test_trades = evaluate_core_secondary_candidate(
        test_candidate,
        candidate_label=test_candidate["candidate_label"],
    )

    comparison = pd.concat(
        [
            locked_2621_summary.assign(test_case="locked_2621"),
            pd.DataFrame([test_result]).assign(test_case="example_variant"),
        ],
        ignore_index=True,
    )

    display(comparison[["test_case"] + summary_display_cols[1:]])

## Archived research that led to 2621:

- Pair 144 established the Core/Secondary structure.
- Relaxing front/middle z-score equality produced pair 2621.
- 90% Core target was not attainable with the tested monotonic rules.
- Full tenor-line fitting underperformed.
- Hybrid VRP/RSI line smoothing also underperformed.
- Therefore, the bucketed 2621 rule is the locked candidate for this phase.

In [12]:
# Cell — Tenor-priority bakeoff for locked 2621 using 2621-conditional win probabilities
#
# Purpose:
#   Keep 2621 thresholds locked.
#   Test only the within-layer tenor-selection priority rule.
#
# Key correction:
#   Tenor win probability is NOT unconditional across all dates.
#   It is conditional on the locked 2621 candidate universe.
#
# Core tenor ranking metric:
#   For each tenor, use all dates where that tenor passes the locked 2621 Core rule.
#
# Secondary tenor ranking metric:
#   For each tenor, use all dates where:
#       1. no Core tenor qualifies, and
#       2. that tenor passes the locked 2621 Secondary rule.
#
# This avoids:
#   - raw P&L duration bias
#   - unconditional tenor statistics
#   - endogeneity from using only the final selected trades
#
# Candidate priority rules:
#   1. win_only
#   2. win_then_pnl_day
#   3. win_band_25bps_then_pnl_day
#   4. win_band_50bps_then_pnl_day
#   5. win_band_100bps_then_pnl_day
#   6. pnl_day_only
#
# Recommendation:
#   Win probability should be the primary selector.
#   P&L/day should only be a tie-break or near-tie-break.

import numpy as np
import pandas as pd
import json

# ---------------------------------------------------------------------
# Setup
# ---------------------------------------------------------------------

# ---------------------------------------------------------------------
# Robust tenor setup
# ---------------------------------------------------------------------
#
# Prefer TENOR_ORDER if it already exists.
# Otherwise infer the tenor order from the research/sweep panel.
# Final fallback is the locked production tenor list.

if "TENOR_ORDER" in globals():
    TENOR_ARRAY = np.array(TENOR_ORDER, dtype=int)

elif "sweep_panel" in globals() and "tenor" in sweep_panel.columns:
    TENOR_ARRAY = np.array(sorted(sweep_panel["tenor"].dropna().unique()), dtype=int)
    TENOR_ORDER = TENOR_ARRAY.tolist()

elif "research_panel" in globals() and "tenor" in research_panel.columns:
    TENOR_ARRAY = np.array(sorted(research_panel["tenor"].dropna().unique()), dtype=int)
    TENOR_ORDER = TENOR_ARRAY.tolist()

else:
    TENOR_ARRAY = np.array([9, 12, 15, 18, 21, 24, 27, 30, 33], dtype=int)
    TENOR_ORDER = TENOR_ARRAY.tolist()

TENOR_TO_COL = {int(t): i for i, t in enumerate(TENOR_ARRAY)}

print("Using TENOR_ORDER:", TENOR_ORDER)

if "pnl_per_day_mat" not in globals():
    with np.errstate(divide="ignore", invalid="ignore"):
        pnl_per_day_mat = pnl_mat / actual_dte_mat
    pnl_per_day_mat[~np.isfinite(pnl_per_day_mat)] = np.nan

print("Running locked 2621 tenor-priority bakeoff.")
print("Tenor ranking is layer-specific and 2621-conditional.")
print("Tenors:", TENOR_ARRAY.tolist())


# ---------------------------------------------------------------------
# Locked 2621 thresholds
# ---------------------------------------------------------------------

LOCKED_2621_CORE = {
    "front": {
        "tenors": [9, 12, 15],
        "vrp": 0.60,
        "z3m": 0.55,
        "z1y": 0.65,
        "rsi_cap": 70,
        "rv21d_floor": 8.5,
    },
    "middle": {
        "tenors": [18, 21, 24],
        "vrp": 0.65,
        "z3m": 0.75,
        "z1y": 0.65,
        "rsi_cap": 68,
        "rv21d_floor": 8.5,
    },
    "back": {
        "tenors": [27, 30, 33],
        "vrp": 0.70,
        "z3m": 0.75,
        "z1y": 0.75,
        "rsi_cap": 66,
        "rv21d_floor": 8.5,
    },
}

LOCKED_2621_SECONDARY = {
    "front": {
        "tenors": [9, 12, 15],
        "vrp": 0.60,
        "z3m": 0.50,
        "z1y": 0.40,
        "rsi_cap": 74,
        "rv21d_floor": 6.5,
    },
    "middle": {
        "tenors": [18, 21, 24],
        "vrp": 0.60,
        "z3m": 0.50,
        "z1y": 0.50,
        "rsi_cap": 70,
        "rv21d_floor": 6.5,
    },
    "back": {
        "tenors": [27, 30, 33],
        "vrp": 0.70,
        "z3m": 0.50,
        "z1y": 0.50,
        "rsi_cap": 68,
        "rv21d_floor": 6.5,
    },
}


# ---------------------------------------------------------------------
# Helpers: tenor groups and locked 2621 qualification matrices
# ---------------------------------------------------------------------

def tenor_group_for_tenor(tenor):
    tenor = int(tenor)

    if tenor in [9, 12, 15]:
        return "front"
    elif tenor in [18, 21, 24]:
        return "middle"
    elif tenor in [27, 30, 33]:
        return "back"
    else:
        raise ValueError(f"Unexpected tenor: {tenor}")


def build_qualifies_matrix(threshold_spec):
    """
    Builds a date x tenor boolean matrix for a locked threshold spec.
    """
    qualifies = np.zeros((num_sweep_dates, len(TENOR_ARRAY)), dtype=bool)

    for col, tenor in enumerate(TENOR_ARRAY):
        group = tenor_group_for_tenor(tenor)
        params = threshold_spec[group]

        qualifies[:, col] = (
            (vrp_mat[:, col] >= float(params["vrp"])) &
            (z3m_mat[:, col] >= float(params["z3m"])) &
            (z1y_mat[:, col] >= float(params["z1y"])) &
            (rsi_by_date <= float(params["rsi_cap"])) &
            (rv21d_by_date >= float(params["rv21d_floor"]))
        )

    return qualifies


core_qualifies_matrix = build_qualifies_matrix(LOCKED_2621_CORE)
secondary_qualifies_matrix = build_qualifies_matrix(LOCKED_2621_SECONDARY)

any_core_qualifies_by_date = core_qualifies_matrix.any(axis=1)
no_core_qualifies_by_date = ~any_core_qualifies_by_date

print("\nLocked 2621 qualification counts:")
print("Dates with at least one Core tenor:", int(any_core_qualifies_by_date.sum()))
print("Dates with no Core tenor:", int(no_core_qualifies_by_date.sum()))
print("Dates with at least one Secondary tenor:", int(secondary_qualifies_matrix.any(axis=1).sum()))
print(
    "Dates with at least one Secondary tenor and no Core tenor:",
    int((secondary_qualifies_matrix.any(axis=1) & no_core_qualifies_by_date).sum()),
)


# ---------------------------------------------------------------------
# Build layer-specific 2621-conditional tenor metrics
# ---------------------------------------------------------------------

def build_layer_conditional_tenor_metrics(layer_name, qualifies_matrix, date_mask):
    """
    Builds tenor ranking metrics conditional on the relevant layer's 2621 candidate universe.

    Core:
      date_mask should be all True.

    Secondary:
      date_mask should be no_core_qualifies_by_date.
    """
    rows = []

    for col, tenor in enumerate(TENOR_ARRAY):
        candidate_mask = (
            date_mask &
            qualifies_matrix[:, col] &
            np.isfinite(win_mat[:, col]) &
            np.isfinite(pnl_mat[:, col]) &
            np.isfinite(actual_dte_mat[:, col]) &
            np.isfinite(pnl_per_day_mat[:, col])
        )

        sample_count = int(candidate_mask.sum())

        if sample_count == 0:
            rows.append({
                "layer": layer_name,
                "selected_col": int(col),
                "selected_tenor": int(tenor),
                "tenor_group": tenor_group_for_tenor(tenor),
                "conditional_win_probability": np.nan,
                "conditional_avg_pnl_per_day": np.nan,
                "conditional_median_pnl_per_day": np.nan,
                "conditional_aggregate_pnl_per_day": np.nan,
                "conditional_avg_raw_pnl": np.nan,
                "conditional_avg_actual_dte": np.nan,
                "conditional_worst_trade_pnl": np.nan,
                "conditional_worst_pnl_per_day": np.nan,
                "candidate_count": 0,
            })
            continue

        rows.append({
            "layer": layer_name,
            "selected_col": int(col),
            "selected_tenor": int(tenor),
            "tenor_group": tenor_group_for_tenor(tenor),

            "conditional_win_probability": float(np.nanmean(win_mat[candidate_mask, col])),
            "conditional_avg_pnl_per_day": float(np.nanmean(pnl_per_day_mat[candidate_mask, col])),
            "conditional_median_pnl_per_day": float(np.nanmedian(pnl_per_day_mat[candidate_mask, col])),
            "conditional_aggregate_pnl_per_day": float(
                np.nansum(pnl_mat[candidate_mask, col]) /
                np.nansum(actual_dte_mat[candidate_mask, col])
            ),
            "conditional_avg_raw_pnl": float(np.nanmean(pnl_mat[candidate_mask, col])),
            "conditional_avg_actual_dte": float(np.nanmean(actual_dte_mat[candidate_mask, col])),
            "conditional_worst_trade_pnl": float(np.nanmin(pnl_mat[candidate_mask, col])),
            "conditional_worst_pnl_per_day": float(np.nanmin(pnl_per_day_mat[candidate_mask, col])),
            "candidate_count": sample_count,
        })

    return pd.DataFrame(rows)


core_tenor_priority_metrics = build_layer_conditional_tenor_metrics(
    layer_name="Core",
    qualifies_matrix=core_qualifies_matrix,
    date_mask=np.ones(num_sweep_dates, dtype=bool),
)

secondary_tenor_priority_metrics = build_layer_conditional_tenor_metrics(
    layer_name="Secondary",
    qualifies_matrix=secondary_qualifies_matrix,
    date_mask=no_core_qualifies_by_date,
)

tenor_priority_metrics = pd.concat(
    [
        core_tenor_priority_metrics,
        secondary_tenor_priority_metrics,
    ],
    ignore_index=True,
)

print("\nCore 2621-conditional tenor ranking metrics:")
display(
    core_tenor_priority_metrics.sort_values(
        [
            "conditional_win_probability",
            "conditional_avg_pnl_per_day",
            "conditional_aggregate_pnl_per_day",
        ],
        ascending=[False, False, False],
    )
)

print("\nSecondary 2621-conditional tenor ranking metrics:")
display(
    secondary_tenor_priority_metrics.sort_values(
        [
            "conditional_win_probability",
            "conditional_avg_pnl_per_day",
            "conditional_aggregate_pnl_per_day",
        ],
        ascending=[False, False, False],
    )
)


# ---------------------------------------------------------------------
# Priority rule
# ---------------------------------------------------------------------

def parse_win_band(rule_name):
    if "25bps" in rule_name:
        return 0.0025
    elif "50bps" in rule_name:
        return 0.0050
    elif "100bps" in rule_name:
        return 0.0100
    else:
        raise ValueError(f"Cannot parse win band from rule: {rule_name}")


def make_priority_order(rule_name, layer_name, qualifying_cols):
    """
    Returns qualifying tenor columns ordered by the specified rule.

    Uses layer-specific 2621-conditional tenor metrics.

    rule_name options:
      - win_only
      - pnl_day_only
      - win_then_pnl_day
      - win_band_25bps_then_pnl_day
      - win_band_50bps_then_pnl_day
      - win_band_100bps_then_pnl_day
    """
    if len(qualifying_cols) == 0:
        return []

    df = tenor_priority_metrics[
        (tenor_priority_metrics["layer"].eq(layer_name)) &
        (tenor_priority_metrics["selected_col"].isin(qualifying_cols))
    ].copy()

    df = df[df["candidate_count"] > 0].copy()

    if df.empty:
        return []

    if rule_name == "win_only":
        # Win probability is primary.
        # P&L/day only breaks exact ties.
        df = df.sort_values(
            [
                "conditional_win_probability",
                "conditional_avg_pnl_per_day",
                "conditional_aggregate_pnl_per_day",
                "selected_tenor",
            ],
            ascending=[False, False, False, False],
        )

    elif rule_name == "win_then_pnl_day":
        # Same as win_only unless exact win-probability ties exist.
        df = df.sort_values(
            [
                "conditional_win_probability",
                "conditional_avg_pnl_per_day",
                "conditional_aggregate_pnl_per_day",
                "selected_tenor",
            ],
            ascending=[False, False, False, False],
        )

    elif rule_name.startswith("win_band_"):
        # Keep tenors within a small win-rate band of the best qualifying tenor.
        # Then select the best P&L/day within that near-equivalent win-rate set.
        band = parse_win_band(rule_name)
        best_win = df["conditional_win_probability"].max()

        df = df[
            df["conditional_win_probability"] >= best_win - band
        ].copy()

        df = df.sort_values(
            [
                "conditional_avg_pnl_per_day",
                "conditional_aggregate_pnl_per_day",
                "conditional_win_probability",
                "selected_tenor",
            ],
            ascending=[False, False, False, False],
        )

    elif rule_name == "pnl_day_only":
        # Diagnostic only. This is not preferred for production.
        df = df.sort_values(
            [
                "conditional_avg_pnl_per_day",
                "conditional_aggregate_pnl_per_day",
                "conditional_win_probability",
                "selected_tenor",
            ],
            ascending=[False, False, False, False],
        )

    else:
        raise ValueError(f"Unknown priority rule: {rule_name}")

    return df["selected_col"].astype(int).tolist()


def select_by_priority_rule(qualifies, rule_name, layer_name):
    """
    For each date, select one tenor among qualifying tenors using the chosen priority rule.
    """
    selected = np.full(num_sweep_dates, -1, dtype=np.int8)

    for row in range(num_sweep_dates):
        qualifying_cols = np.where(qualifies[row, :])[0].astype(int).tolist()

        if not qualifying_cols:
            continue

        ordered_cols = make_priority_order(
            rule_name=rule_name,
            layer_name=layer_name,
            qualifying_cols=qualifying_cols,
        )

        if not ordered_cols:
            continue

        selected[row] = int(ordered_cols[0])

    return selected


# ---------------------------------------------------------------------
# Summary helpers
# ---------------------------------------------------------------------

def summarize_selected_cols(selected_col_by_date):
    trade_mask = selected_col_by_date >= 0
    trade_count = int(trade_mask.sum())

    tenor_count_defaults = {
        f"selected_tenor_{t}_count": 0
        for t in TENOR_ARRAY
    }

    if trade_count == 0:
        return {
            "trade_count": 0,
            "trade_frequency": 0.0,
            "win_rate": np.nan,
            "avg_pnl_per_day": np.nan,
            "median_pnl_per_day": np.nan,
            "aggregate_pnl_per_day": np.nan,
            "total_pnl": np.nan,
            "total_actual_dte": np.nan,
            "worst_trade_pnl": np.nan,
            "worst_pnl_per_day": np.nan,
            "max_drawdown": np.nan,
            "worst_20_trade_sum": np.nan,
            "avg_selected_tenor": np.nan,
            "front_count": 0,
            "middle_count": 0,
            "back_count": 0,
            **tenor_count_defaults,
        }

    row_idx = np.where(trade_mask)[0]
    col_idx = selected_col_by_date[trade_mask].astype(int)

    selected_wins = win_mat[row_idx, col_idx]
    selected_pnls = pnl_mat[row_idx, col_idx]
    selected_dtes = actual_dte_mat[row_idx, col_idx]
    selected_pnl_per_day = pnl_per_day_mat[row_idx, col_idx]
    selected_tenors = TENOR_ARRAY[col_idx]

    tmp = pd.DataFrame({
        "date": sweep_dates[row_idx],
        "pnl": selected_pnls,
    }).sort_values("date")

    tmp["cum_pnl"] = tmp["pnl"].cumsum()
    tmp["running_max"] = tmp["cum_pnl"].cummax()
    tmp["drawdown"] = tmp["cum_pnl"] - tmp["running_max"]
    tmp["rolling_20_trade_pnl"] = tmp["pnl"].rolling(20).sum()

    tenor_counts = {
        f"selected_tenor_{t}_count": int((selected_tenors == t).sum())
        for t in TENOR_ARRAY
    }

    return {
        "trade_count": trade_count,
        "trade_frequency": float(trade_count / TRADE_FREQUENCY_DENOMINATOR),
        "win_rate": float(np.nanmean(selected_wins)),

        "avg_pnl_per_day": float(np.nanmean(selected_pnl_per_day)),
        "median_pnl_per_day": float(np.nanmedian(selected_pnl_per_day)),
        "aggregate_pnl_per_day": float(np.nansum(selected_pnls) / np.nansum(selected_dtes)),
        "total_pnl": float(np.nansum(selected_pnls)),
        "total_actual_dte": float(np.nansum(selected_dtes)),
        "worst_trade_pnl": float(np.nanmin(selected_pnls)),
        "worst_pnl_per_day": float(np.nanmin(selected_pnl_per_day)),
        "max_drawdown": float(tmp["drawdown"].min()),
        "worst_20_trade_sum": float(tmp["rolling_20_trade_pnl"].min()),

        "avg_selected_tenor": float(np.nanmean(selected_tenors)),

        "front_count": int(np.isin(selected_tenors, [9, 12, 15]).sum()),
        "middle_count": int(np.isin(selected_tenors, [18, 21, 24]).sum()),
        "back_count": int(np.isin(selected_tenors, [27, 30, 33]).sum()),

        **tenor_counts,
    }


def build_selected_trade_path(candidate_label, combined_selected, core_selected, secondary_only_selected):
    """
    Builds selected-trade path for a given priority rule.
    """
    trade_mask = combined_selected >= 0
    row_idx = np.where(trade_mask)[0]
    col_idx = combined_selected[trade_mask].astype(int)

    layer = np.where(
        core_selected[trade_mask] >= 0,
        "Core",
        np.where(secondary_only_selected[trade_mask] >= 0, "Secondary", "Unknown"),
    )

    selected_tenors = TENOR_ARRAY[col_idx]

    trades = pd.DataFrame({
        "candidate": candidate_label,
        "date": sweep_dates[row_idx],
        "layer": layer,
        "selected_col": col_idx,
        "selected_tenor": selected_tenors,
        "tenor_group": [tenor_group_for_tenor(t) for t in selected_tenors],

        "win": win_mat[row_idx, col_idx],
        "normalized_pnl_dollars": pnl_mat[row_idx, col_idx],
        "actual_dte": actual_dte_mat[row_idx, col_idx],
        "pnl_per_day": pnl_per_day_mat[row_idx, col_idx],

        "spx_close": spx_close_mat[row_idx, col_idx] if "spx_close_mat" in globals() else np.nan,
        "vrp_log": vrp_mat[row_idx, col_idx],
        "vrp_z_3m": z3m_mat[row_idx, col_idx],
        "vrp_z_1y": z1y_mat[row_idx, col_idx],
        "rv21d": rv21d_by_date[row_idx],
        "rsi14": rsi_by_date[row_idx],
    }).sort_values("date").reset_index(drop=True)

    # Attach layer-specific conditional ranking metrics for transparency.
    trades = trades.merge(
        tenor_priority_metrics.rename(
            columns={
                "selected_tenor": "selected_tenor",
                "selected_col": "selected_col",
            }
        ),
        on=["layer", "selected_col", "selected_tenor", "tenor_group"],
        how="left",
    )

    trades["trade_sequence"] = np.arange(1, len(trades) + 1)
    trades["cum_pnl"] = trades["normalized_pnl_dollars"].cumsum()
    trades["running_max_cum_pnl"] = trades["cum_pnl"].cummax()
    trades["drawdown"] = trades["cum_pnl"] - trades["running_max_cum_pnl"]
    trades["rolling_20_trade_pnl"] = trades["normalized_pnl_dollars"].rolling(20).sum()
    trades["year"] = pd.to_datetime(trades["date"]).dt.year
    trades["month"] = pd.to_datetime(trades["date"]).dt.to_period("M").astype(str)

    return trades


# ---------------------------------------------------------------------
# Evaluate locked 2621 under a given priority rule
# ---------------------------------------------------------------------

def evaluate_locked_2621_with_priority_rule(rule_name):
    """
    Core-first / Secondary-second evaluation under a given within-layer tenor priority rule.
    """
    core_selected = select_by_priority_rule(
        qualifies=core_qualifies_matrix,
        rule_name=rule_name,
        layer_name="Core",
    )

    secondary_raw_selected = select_by_priority_rule(
        qualifies=secondary_qualifies_matrix,
        rule_name=rule_name,
        layer_name="Secondary",
    )

    secondary_only_selected = np.where(
        core_selected >= 0,
        -1,
        secondary_raw_selected,
    ).astype(np.int8)

    combined_selected = np.where(
        core_selected >= 0,
        core_selected,
        secondary_raw_selected,
    ).astype(np.int8)

    core_summary = summarize_selected_cols(core_selected)
    secondary_summary = summarize_selected_cols(secondary_only_selected)
    combined_summary = summarize_selected_cols(combined_selected)

    result = {
        "priority_rule": rule_name,

        "combined_trade_count": combined_summary["trade_count"],
        "combined_trade_frequency": combined_summary["trade_frequency"],
        "combined_win_rate": combined_summary["win_rate"],
        "combined_avg_pnl_per_day": combined_summary["avg_pnl_per_day"],
        "combined_median_pnl_per_day": combined_summary["median_pnl_per_day"],
        "combined_aggregate_pnl_per_day": combined_summary["aggregate_pnl_per_day"],
        "combined_total_pnl": combined_summary["total_pnl"],
        "combined_total_actual_dte": combined_summary["total_actual_dte"],
        "combined_worst_trade_pnl": combined_summary["worst_trade_pnl"],
        "combined_worst_pnl_per_day": combined_summary["worst_pnl_per_day"],
        "combined_max_drawdown": combined_summary["max_drawdown"],
        "combined_worst_20_trade_sum": combined_summary["worst_20_trade_sum"],
        "combined_avg_selected_tenor": combined_summary["avg_selected_tenor"],

        "core_trade_count": core_summary["trade_count"],
        "core_trade_frequency": core_summary["trade_frequency"],
        "core_win_rate": core_summary["win_rate"],
        "core_avg_pnl_per_day": core_summary["avg_pnl_per_day"],
        "core_aggregate_pnl_per_day": core_summary["aggregate_pnl_per_day"],
        "core_worst_20_trade_sum": core_summary["worst_20_trade_sum"],

        "secondary_trade_count": secondary_summary["trade_count"],
        "secondary_trade_frequency": secondary_summary["trade_frequency"],
        "secondary_win_rate": secondary_summary["win_rate"],
        "secondary_avg_pnl_per_day": secondary_summary["avg_pnl_per_day"],
        "secondary_aggregate_pnl_per_day": secondary_summary["aggregate_pnl_per_day"],
        "secondary_worst_20_trade_sum": secondary_summary["worst_20_trade_sum"],

        "front_count": combined_summary["front_count"],
        "middle_count": combined_summary["middle_count"],
        "back_count": combined_summary["back_count"],
    }

    for tenor in TENOR_ARRAY:
        result[f"selected_tenor_{tenor}_count"] = combined_summary[
            f"selected_tenor_{tenor}_count"
        ]

    selected_outputs = {
        "core_selected": core_selected,
        "secondary_raw_selected": secondary_raw_selected,
        "secondary_only_selected": secondary_only_selected,
        "combined_selected": combined_selected,
    }

    return result, selected_outputs


# ---------------------------------------------------------------------
# Run bakeoff
# ---------------------------------------------------------------------

PRIORITY_RULES_TO_TEST = [
    "win_only",
    "win_then_pnl_day",
    "win_band_25bps_then_pnl_day",
    "win_band_50bps_then_pnl_day",
    "win_band_100bps_then_pnl_day",
    "pnl_day_only",
]

priority_results = []
priority_selected_outputs = {}
priority_trade_paths = {}

for rule_name in PRIORITY_RULES_TO_TEST:
    result, outputs = evaluate_locked_2621_with_priority_rule(rule_name)

    priority_results.append(result)
    priority_selected_outputs[rule_name] = outputs

    priority_trade_paths[rule_name] = build_selected_trade_path(
        candidate_label=f"locked_2621__{rule_name}",
        combined_selected=outputs["combined_selected"],
        core_selected=outputs["core_selected"],
        secondary_only_selected=outputs["secondary_only_selected"],
    )

priority_bakeoff_summary = pd.DataFrame(priority_results)

priority_bakeoff_summary = priority_bakeoff_summary.sort_values(
    [
        "combined_win_rate",
        "combined_trade_frequency",
        "combined_avg_pnl_per_day",
        "combined_worst_20_trade_sum",
    ],
    ascending=[False, False, False, False],
).reset_index(drop=True)

priority_bakeoff_summary["rank"] = np.arange(1, len(priority_bakeoff_summary) + 1)

display_cols = [
    "rank",
    "priority_rule",

    "combined_trade_count",
    "combined_trade_frequency",
    "combined_win_rate",
    "combined_avg_pnl_per_day",
    "combined_aggregate_pnl_per_day",
    "combined_max_drawdown",
    "combined_worst_20_trade_sum",
    "combined_avg_selected_tenor",

    "core_trade_count",
    "core_win_rate",
    "core_avg_pnl_per_day",
    "core_worst_20_trade_sum",

    "secondary_trade_count",
    "secondary_win_rate",
    "secondary_avg_pnl_per_day",
    "secondary_worst_20_trade_sum",

    "front_count",
    "middle_count",
    "back_count",

    "selected_tenor_9_count",
    "selected_tenor_12_count",
    "selected_tenor_15_count",
    "selected_tenor_18_count",
    "selected_tenor_21_count",
    "selected_tenor_24_count",
    "selected_tenor_27_count",
    "selected_tenor_30_count",
    "selected_tenor_33_count",
]

print("\nLocked 2621 tenor-priority bakeoff using 2621-conditional metrics:")
display(priority_bakeoff_summary[display_cols])


# ---------------------------------------------------------------------
# Overlap diagnostics
# ---------------------------------------------------------------------

def overlap_vs_reference(reference_rule):
    reference_selected = priority_selected_outputs[reference_rule]["combined_selected"]
    reference_mask = reference_selected >= 0

    rows = []

    for rule_name, outputs in priority_selected_outputs.items():
        selected = outputs["combined_selected"]
        selected_mask = selected >= 0

        overlap_dates = reference_mask & selected_mask
        same_tenor_dates = overlap_dates & (reference_selected == selected)
        union_dates = reference_mask | selected_mask

        rows.append({
            "reference_rule": reference_rule,
            "priority_rule": rule_name,
            "reference_trades": int(reference_mask.sum()),
            "rule_trades": int(selected_mask.sum()),
            "overlap_trade_dates": int(overlap_dates.sum()),
            "same_selected_tenor_dates": int(same_tenor_dates.sum()),
            "reference_overlap_pct": float(overlap_dates.sum() / reference_mask.sum()) if reference_mask.sum() else np.nan,
            "rule_overlap_pct": float(overlap_dates.sum() / selected_mask.sum()) if selected_mask.sum() else np.nan,
            "same_tenor_pct_of_overlap": float(same_tenor_dates.sum() / overlap_dates.sum()) if overlap_dates.sum() else np.nan,
            "jaccard_trade_date_overlap": float(overlap_dates.sum() / union_dates.sum()) if union_dates.sum() else np.nan,
        })

    return pd.DataFrame(rows)


overlap_vs_win_only = overlap_vs_reference("win_only")
overlap_vs_pnl_day_only = overlap_vs_reference("pnl_day_only")

print("\nOverlap versus win_only:")
display(
    overlap_vs_win_only.sort_values(
        ["same_tenor_pct_of_overlap", "jaccard_trade_date_overlap"],
        ascending=[False, False],
    )
)

print("\nOverlap versus pnl_day_only:")
display(
    overlap_vs_pnl_day_only.sort_values(
        ["same_tenor_pct_of_overlap", "jaccard_trade_date_overlap"],
        ascending=[False, False],
    )
)


# ---------------------------------------------------------------------
# Attribution for top-ranked rule
# ---------------------------------------------------------------------

best_rule = str(priority_bakeoff_summary.iloc[0]["priority_rule"])
best_trades = priority_trade_paths[best_rule].copy()

print(f"\nBest rule by combined win rate: {best_rule}")

best_summary_by_layer = (
    best_trades
    .groupby("layer", dropna=False)
    .agg(
        trade_count=("win", "size"),
        win_rate=("win", "mean"),
        avg_pnl_per_day=("pnl_per_day", "mean"),
        aggregate_pnl_per_day=("normalized_pnl_dollars", lambda x: np.nan),
        total_pnl=("normalized_pnl_dollars", "sum"),
        total_actual_dte=("actual_dte", "sum"),
        avg_selected_tenor=("selected_tenor", "mean"),
        worst_trade_pnl=("normalized_pnl_dollars", "min"),
        worst_pnl_per_day=("pnl_per_day", "min"),
    )
    .reset_index()
)

best_summary_by_layer["aggregate_pnl_per_day"] = (
    best_summary_by_layer["total_pnl"] / best_summary_by_layer["total_actual_dte"]
)

best_summary_by_tenor_layer = (
    best_trades
    .groupby(["selected_tenor", "layer"], dropna=False)
    .agg(
        trade_count=("win", "size"),
        win_rate=("win", "mean"),
        avg_pnl_per_day=("pnl_per_day", "mean"),
        total_pnl=("normalized_pnl_dollars", "sum"),
        total_actual_dte=("actual_dte", "sum"),
        avg_selected_tenor=("selected_tenor", "mean"),
        worst_trade_pnl=("normalized_pnl_dollars", "min"),
        worst_pnl_per_day=("pnl_per_day", "min"),
    )
    .reset_index()
)

best_summary_by_tenor_layer["aggregate_pnl_per_day"] = (
    best_summary_by_tenor_layer["total_pnl"] /
    best_summary_by_tenor_layer["total_actual_dte"]
)

best_summary_by_year_layer = (
    best_trades
    .groupby(["year", "layer"], dropna=False)
    .agg(
        trade_count=("win", "size"),
        win_rate=("win", "mean"),
        avg_pnl_per_day=("pnl_per_day", "mean"),
        total_pnl=("normalized_pnl_dollars", "sum"),
        total_actual_dte=("actual_dte", "sum"),
        avg_selected_tenor=("selected_tenor", "mean"),
        worst_trade_pnl=("normalized_pnl_dollars", "min"),
        worst_pnl_per_day=("pnl_per_day", "min"),
    )
    .reset_index()
)

best_summary_by_year_layer["aggregate_pnl_per_day"] = (
    best_summary_by_year_layer["total_pnl"] /
    best_summary_by_year_layer["total_actual_dte"]
)

worst_trades_best_rule = (
    best_trades
    .sort_values("normalized_pnl_dollars", ascending=True)
    .head(20)
    .reset_index(drop=True)
)

worst_20_trade_windows_best_rule = (
    best_trades
    .dropna(subset=["rolling_20_trade_pnl"])
    .sort_values("rolling_20_trade_pnl", ascending=True)
    .head(20)
    .reset_index(drop=True)
)

print("\nBest rule summary by layer:")
display(best_summary_by_layer)

print("\nBest rule summary by tenor and layer:")
display(best_summary_by_tenor_layer)

print("\nBest rule summary by year and layer:")
display(best_summary_by_year_layer)

print("\nWorst 20 trades for best rule:")
display(worst_trades_best_rule)

print("\nWorst 20-trade rolling windows for best rule:")
display(worst_20_trade_windows_best_rule)


# ---------------------------------------------------------------------
# Save outputs
# ---------------------------------------------------------------------

PRIORITY_BAKEOFF_PATH = AUDIT_DIR / "locked_2621_tenor_priority_bakeoff_conditional_v0_1.csv"
PRIORITY_METRICS_PATH = AUDIT_DIR / "locked_2621_tenor_priority_metrics_conditional_v0_1.csv"
PRIORITY_OVERLAP_WIN_PATH = AUDIT_DIR / "locked_2621_tenor_priority_overlap_vs_win_only_conditional_v0_1.csv"
PRIORITY_OVERLAP_PNL_PATH = AUDIT_DIR / "locked_2621_tenor_priority_overlap_vs_pnl_day_only_conditional_v0_1.csv"

BEST_TRADES_PATH = AUDIT_DIR / f"locked_2621_selected_trades_{best_rule}_conditional_v0_1.csv"
BEST_TRADES_PARQUET_PATH = PROCESSED_DIR / f"locked_2621_selected_trades_{best_rule}_conditional_v0_1.parquet"

BEST_BY_LAYER_PATH = AUDIT_DIR / f"locked_2621_summary_by_layer_{best_rule}_conditional_v0_1.csv"
BEST_BY_TENOR_LAYER_PATH = AUDIT_DIR / f"locked_2621_summary_by_tenor_layer_{best_rule}_conditional_v0_1.csv"
BEST_BY_YEAR_LAYER_PATH = AUDIT_DIR / f"locked_2621_summary_by_year_layer_{best_rule}_conditional_v0_1.csv"
BEST_WORST_TRADES_PATH = AUDIT_DIR / f"locked_2621_worst_trades_{best_rule}_conditional_v0_1.csv"
BEST_WORST_WINDOWS_PATH = AUDIT_DIR / f"locked_2621_worst_20_trade_windows_{best_rule}_conditional_v0_1.csv"

PRIORITY_SUMMARY_JSON_PATH = AUDIT_DIR / "locked_2621_tenor_priority_bakeoff_conditional_summary_v0_1.json"

priority_bakeoff_summary.to_csv(PRIORITY_BAKEOFF_PATH, index=False)
tenor_priority_metrics.to_csv(PRIORITY_METRICS_PATH, index=False)
overlap_vs_win_only.to_csv(PRIORITY_OVERLAP_WIN_PATH, index=False)
overlap_vs_pnl_day_only.to_csv(PRIORITY_OVERLAP_PNL_PATH, index=False)

best_trades.to_csv(BEST_TRADES_PATH, index=False)
best_trades.to_parquet(BEST_TRADES_PARQUET_PATH, index=False)

best_summary_by_layer.to_csv(BEST_BY_LAYER_PATH, index=False)
best_summary_by_tenor_layer.to_csv(BEST_BY_TENOR_LAYER_PATH, index=False)
best_summary_by_year_layer.to_csv(BEST_BY_YEAR_LAYER_PATH, index=False)
worst_trades_best_rule.to_csv(BEST_WORST_TRADES_PATH, index=False)
worst_20_trade_windows_best_rule.to_csv(BEST_WORST_WINDOWS_PATH, index=False)

priority_summary_json = {
    "locked_thresholds": "2621",
    "tenor_priority_metric": "layer-specific 2621-conditional win probability",
    "tie_break_metric": "layer-specific 2621-conditional average P&L per day",
    "rules_tested": PRIORITY_RULES_TO_TEST,
    "best_by_combined_win_rate": best_rule,
    "best_combined_win_rate": float(priority_bakeoff_summary.iloc[0]["combined_win_rate"]),
    "best_combined_trade_frequency": float(priority_bakeoff_summary.iloc[0]["combined_trade_frequency"]),
    "best_combined_avg_pnl_per_day": float(priority_bakeoff_summary.iloc[0]["combined_avg_pnl_per_day"]),
    "best_combined_aggregate_pnl_per_day": float(priority_bakeoff_summary.iloc[0]["combined_aggregate_pnl_per_day"]),
    "best_combined_max_drawdown": float(priority_bakeoff_summary.iloc[0]["combined_max_drawdown"]),
    "best_combined_worst_20_trade_sum": float(priority_bakeoff_summary.iloc[0]["combined_worst_20_trade_sum"]),
    "best_front_count": int(priority_bakeoff_summary.iloc[0]["front_count"]),
    "best_middle_count": int(priority_bakeoff_summary.iloc[0]["middle_count"]),
    "best_back_count": int(priority_bakeoff_summary.iloc[0]["back_count"]),
}

with open(PRIORITY_SUMMARY_JSON_PATH, "w") as f:
    json.dump(priority_summary_json, f, indent=2)

print("\nSaved tenor-priority bakeoff outputs:")
print(PRIORITY_BAKEOFF_PATH)
print(PRIORITY_METRICS_PATH)
print(PRIORITY_OVERLAP_WIN_PATH)
print(PRIORITY_OVERLAP_PNL_PATH)
print(BEST_TRADES_PATH)
print(BEST_TRADES_PARQUET_PATH)
print(BEST_BY_LAYER_PATH)
print(BEST_BY_TENOR_LAYER_PATH)
print(BEST_BY_YEAR_LAYER_PATH)
print(BEST_WORST_TRADES_PATH)
print(BEST_WORST_WINDOWS_PATH)
print(PRIORITY_SUMMARY_JSON_PATH)

print("\nConditional tenor-priority bakeoff complete.")

Using TENOR_ORDER: [9, 12, 15, 18, 21, 24, 27, 30, 33]
Running locked 2621 tenor-priority bakeoff.
Tenor ranking is layer-specific and 2621-conditional.
Tenors: [9, 12, 15, 18, 21, 24, 27, 30, 33]

Locked 2621 qualification counts:
Dates with at least one Core tenor: 386
Dates with no Core tenor: 1339
Dates with at least one Secondary tenor: 577
Dates with at least one Secondary tenor and no Core tenor: 191

Core 2621-conditional tenor ranking metrics:


,layer,selected_col,selected_tenor,tenor_group,conditional_win_probability,conditional_avg_pnl_per_day,conditional_median_pnl_per_day,conditional_aggregate_pnl_per_day,conditional_avg_raw_pnl,conditional_avg_actual_dte,conditional_worst_trade_pnl,conditional_worst_pnl_per_day,candidate_count
8,Core,8,33,back,0.895522,431.999112,499.797363,430.069297,"14,063.907900",32.701493,"-114,955.549452","-3,193.209707",201
6,Core,6,27,back,0.894231,490.191037,554.917740,488.341492,"13,262.697550",27.158654,"-57,319.352129","-2,292.774085",208
7,Core,7,30,back,0.880000,440.960597,529.383338,440.761991,"13,156.745421",29.850000,"-57,036.901356","-2,037.032191",200
5,Core,5,24,middle,0.851240,426.816292,589.114564,430.610667,"9,886.251501",22.958678,"-57,319.352129","-2,469.960158",242
4,Core,4,21,middle,0.834783,451.823181,613.672423,450.613423,"9,811.617490",21.773913,"-56,809.083631","-2,487.128419",230
3,Core,3,18,middle,0.799213,445.583034,683.385498,451.521041,"7,805.625564",17.287402,"-45,195.769858","-2,716.007704",254
2,Core,2,15,front,0.771523,409.251409,706.225901,413.442202,"6,584.956923",15.927152,"-46,130.619927","-3,295.044280",302
0,Core,0,9,front,0.749164,459.029727,989.749243,454.055688,"3,924.013040",8.642140,"-60,891.722904","-6,576.872609",299
1,Core,1,12,front,0.745763,402.590954,843.753027,389.340289,"4,612.692582",11.847458,"-56,963.300165","-4,926.141041",295



Secondary 2621-conditional tenor ranking metrics:


,layer,selected_col,selected_tenor,tenor_group,conditional_win_probability,conditional_avg_pnl_per_day,conditional_median_pnl_per_day,conditional_aggregate_pnl_per_day,conditional_avg_raw_pnl,conditional_avg_actual_dte,conditional_worst_trade_pnl,conditional_worst_pnl_per_day,candidate_count
8,Secondary,8,33,back,0.772152,184.108263,409.122134,182.709066,"6,003.958689",32.860759,"-87,932.829005","-2,512.366543",79
7,Secondary,7,30,back,0.739726,164.877502,409.122134,165.283628,"4,924.546456",29.794521,"-33,497.150782","-1,046.785962",73
2,Secondary,2,15,front,0.730435,168.629531,491.896754,164.934882,"2,650.431838",16.069565,"-42,928.530585","-2,384.918366",115
6,Secondary,6,27,back,0.710145,105.979093,395.526250,114.435826,"3,134.546532",27.391304,"-44,003.345303","-1,833.472721",69
3,Secondary,3,18,middle,0.703704,154.582398,490.623102,159.492943,"2,725.163367",17.086420,"-37,360.735722","-2,197.690337",81
5,Secondary,5,24,middle,0.703704,152.966157,444.563191,147.733099,"3,365.031706",22.777778,"-44,003.345303","-1,833.472721",81
0,Secondary,0,9,front,0.683333,107.213381,656.402509,71.258978,614.014859,8.616667,"-27,346.824178","-3,755.720924",120
4,Secondary,4,21,middle,0.662162,104.292259,448.669768,87.565238,"1,894.485762",21.635135,"-44,003.345303","-1,833.472721",74
1,Secondary,1,12,front,0.658120,-1.777573,515.605288,26.556819,304.155017,11.452991,"-29,580.217937","-3,038.536020",117



Locked 2621 tenor-priority bakeoff using 2621-conditional metrics:


...


Worst 20 trades for best rule:


,candidate,date,layer,selected_col,selected_tenor,tenor_group,win,normalized_pnl_dollars,actual_dte,pnl_per_day,spx_close,vrp_log,vrp_z_3m,vrp_z_1y,rv21d,rsi14,conditional_win_probability,conditional_avg_pnl_per_day,conditional_median_pnl_per_day,conditional_aggregate_pnl_per_day,conditional_avg_raw_pnl,conditional_avg_actual_dte,conditional_worst_trade_pnl,conditional_worst_pnl_per_day,candidate_count,trade_sequence,cum_pnl,running_max_cum_pnl,drawdown,rolling_20_trade_pnl,year,month
0,locked_2621__pnl_day_only,2020-01-24,Secondary,8,33,back,0.000000,"-87,932.829005",35,"-2,512.366543",NaN,1.300539,1.061030,1.738160,7.866687,61.573549,0.772152,184.108263,409.122134,182.709066,"6,003.958689",32.860759,"-87,932.829005","-2,512.366543",79,54,"71,133.119554","159,065.948559","-87,932.829005","101,620.934570",2020,2020-01
1,locked_2621__pnl_day_only,2020-02-24,Core,0,9,front,0.000000,"-60,891.722904",11,"-5,535.611173",NaN,1.425272,0.638828,1.491332,18.033963,36.699503,0.749164,459.029727,989.749243,454.055688,"3,924.013040",8.642140,"-60,891.722904","-6,576.872609",299,60,"65,107.234251","159,065.948559","-93,958.714308","30,367.631554",2020,2020-02
2,locked_2621__pnl_day_only,2026-03-02,Core,6,27,back,0.000000,"-57,319.352129",25,"-2,292.774085",NaN,1.222648,1.671398,1.140005,12.896987,48.647576,0.894231,490.191037,554.917740,488.341492,"13,262.697550",27.158654,"-57,319.352129","-2,292.774085",208,557,"4,561,276.909866","4,717,445.867616","-156,168.957750","-68,448.602522",2026,2026-03
3,locked_2621__pnl_day_only,2026-02-27,Core,6,27,back,0.000000,"-57,036.901356",28,"-2,037.032191",NaN,1.001127,0.847920,0.778522,12.893787,48.384049,0.894231,490.191037,554.917740,488.341492,"13,262.697550",27.158654,"-57,319.352129","-2,292.774085",208,556,"4,618,596.261996","4,717,445.867616","-98,849.605621","-3,532.757704",2026,2026-02
4,locked_2621__pnl_day_only,2025-02-26,Core,6,27,back,0.000000,"-47,172.123854",30,"-1,572.404128",NaN,0.946285,1.605369,1.141095,10.753467,41.127776,0.894231,490.191037,554.917740,488.341492,"13,262.697550",27.158654,"-57,319.352129","-2,292.774085",208,460,"3,926,035.144880","4,046,169.154741","-120,134.009861","35,797.402483",2025,2025-02
5,locked_2621__pnl_day_only,2024-07-15,Secondary,2,15,front,0.000000,"-42,928.530585",18,"-2,384.918366",NaN,0.973419,0.966242,1.086085,6.883493,73.584174,0.730435,168.629531,491.896754,164.934882,"2,650.431838",16.069565,"-42,928.530585","-2,384.918366",115,411,"3,636,486.644098","3,704,481.052935","-67,994.408836","68,116.133223",2024,2024-07
6,locked_2621__pnl_day_only,2025-02-24,Core,6,27,back,0.000000,"-37,862.365771",25,"-1,514.494631",NaN,0.825419,1.219892,0.833731,11.772753,43.422979,0.894231,490.191037,554.917740,488.341492,"13,262.697550",27.158654,"-57,319.352129","-2,292.774085",208,458,"4,006,344.416628","4,046,169.154741","-39,824.738113","146,337.380684",2025,2025-02
7,locked_2621__pnl_day_only,2019-07-18,Secondary,8,33,back,0.000000,"-37,023.681935",36,"-1,028.435609",NaN,0.857898,0.758000,0.927364,7.757400,61.997910,0.772152,184.108263,409.122134,182.709066,"6,003.958689",32.860759,"-87,932.829005","-2,512.366543",79,2,"-57,268.821694","-20,245.139759","-37,023.681935",NaN,2019,2019-07
8,locked_2621__pnl_day_only,2023-03-03,Core,0,9,front,0.000000,"-36,461.474575",7,"-5,208.782082",NaN,0.751698,1.015500,0.994409,15.716256,52.319603,0.749164,459.029727,989.749243,454.055688,"3,924.013040",8.642140,"-60,891.722904","-6,576.872609",299,273,"2,605,025.143432","2,703,845.660260","-98,820.516828","12,172.998595",2023,2023-03
9,locked_2621__pnl_day_only,2019-07-29,Secondary,2,15,front,0.000000,"-34,631.260820",18,"-1,923.958934",NaN,0.960231,1.106724,1.237968,7.533697,62.640138,0.730435,168.629531,491.896754,164.934882,"2,650.431838",16.069565,"-42,928.530585","-2,384.918366",115,9,"-244,609.784166","-20,245.139759","-224,364.644407",NaN,2019,2019-07



Worst 20-trade rolling windows for best rule:


,candidate,date,layer,selected_col,selected_tenor,tenor_group,win,normalized_pnl_dollars,actual_dte,pnl_per_day,spx_close,vrp_log,vrp_z_3m,vrp_z_1y,rv21d,rsi14,conditional_win_probability,conditional_avg_pnl_per_day,conditional_median_pnl_per_day,conditional_aggregate_pnl_per_day,conditional_avg_raw_pnl,conditional_avg_actual_dte,conditional_worst_trade_pnl,conditional_worst_pnl_per_day,candidate_count,trade_sequence,cum_pnl,running_max_cum_pnl,drawdown,rolling_20_trade_pnl,year,month
0,locked_2621__pnl_day_only,2019-09-26,Core,0,9,front,0.000000,-769.070600,8,-96.133825,NaN,0.841904,0.628889,0.912380,9.290683,52.605490,0.749164,459.029727,989.749243,454.055688,"3,924.013040",8.642140,"-60,891.722904","-6,576.872609",299,20,"-198,744.768243","-20,245.139759","-178,499.628484","-198,744.768243",2019,2019-09
1,locked_2621__pnl_day_only,2025-03-07,Core,0,9,front,0.000000,"-10,443.312190",7,"-1,491.901741",NaN,1.159666,1.431840,1.170429,15.747961,36.833340,0.749164,459.029727,989.749243,454.055688,"3,924.013040",8.642140,"-60,891.722904","-6,576.872609",299,467,"3,816,146.421343","4,046,169.154741","-230,022.733399","-186,473.788847",2025,2025-03
2,locked_2621__pnl_day_only,2025-03-10,Core,0,9,front,1.000000,"17,418.996324",11,"1,583.545120",NaN,1.047558,1.069668,0.896554,17.789582,29.915345,0.749164,459.029727,989.749243,454.055688,"3,924.013040",8.642140,"-60,891.722904","-6,576.872609",299,468,"3,833,565.417667","4,046,169.154741","-212,603.737075","-185,929.504787",2025,2025-03
3,locked_2621__pnl_day_only,2025-03-11,Core,0,9,front,1.000000,"17,174.945756",10,"1,717.494576",NaN,0.964113,0.804946,0.693816,17.724709,28.349890,0.749164,459.029727,989.749243,454.055688,"3,924.013040",8.642140,"-60,891.722904","-6,576.872609",299,469,"3,850,740.363423","4,046,169.154741","-195,428.791319","-184,283.476018",2025,2025-03
4,locked_2621__pnl_day_only,2019-09-27,Core,0,9,front,1.000000,"4,865.301051",7,695.043007,NaN,1.257446,1.420800,1.656532,9.432342,48.655212,0.749164,459.029727,989.749243,454.055688,"3,924.013040",8.642140,"-60,891.722904","-6,576.872609",299,21,"-193,879.467192","-20,245.139759","-173,634.327433","-173,634.327433",2019,2019-09
5,locked_2621__pnl_day_only,2025-03-06,Core,0,9,front,0.000000,"-2,293.274224",8,-286.659278,NaN,1.195339,1.592217,1.265629,15.656730,33.946232,0.749164,459.029727,989.749243,454.055688,"3,924.013040",8.642140,"-60,891.722904","-6,576.872609",299,466,"3,826,589.733533","4,046,169.154741","-219,579.421208","-158,225.700788",2025,2025-03
6,locked_2621__pnl_day_only,2026-03-09,Core,6,27,back,0.000000,"-12,685.421844",24,-528.559244,NaN,1.436394,2.039998,1.442139,13.563080,43.879944,0.894231,490.191037,554.917740,488.341492,"13,262.697550",27.158654,"-57,319.352129","-2,292.774085",208,562,"4,495,127.208794","4,717,445.867616","-222,318.658822","-149,121.616130",2026,2026-03
7,locked_2621__pnl_day_only,2025-03-13,Secondary,0,9,front,1.000000,"13,420.217621",8,"1,677.527203",NaN,0.900698,0.603020,0.542724,17.872498,27.860112,0.683333,107.213381,656.402509,71.258978,614.014859,8.616667,"-27,346.824178","-3,755.720924",120,470,"3,864,160.581044","4,046,169.154741","-182,008.573697","-146,743.594640",2025,2025-03
8,locked_2621__pnl_day_only,2025-03-05,Secondary,2,15,front,0.000000,"-13,887.581449",16,-867.973841,NaN,0.789344,0.601787,0.437414,14.916158,39.448300,0.730435,168.629531,491.896754,164.934882,"2,650.431838",16.069565,"-42,928.530585","-2,384.918366",115,465,"3,828,883.007757","4,046,169.154741","-217,286.146984","-138,107.364561",2025,2025-03
9,locked_2621__pnl_day_only,2026-03-06,Core,6,27,back,1.000000,339.761603,27,12.583763,NaN,1.733877,3.386581,1.941130,13.851073,38.453183,0.894231,490.191037,554.917740,488.341492,"13,262.697550",27.158654,"-57,319.352129","-2,292.774085",208,561,"4,507,812.630639","4,717,445.867616","-209,633.236978","-133,595.044682",2026,2026-03



Saved tenor-priority bakeoff outputs:
C:\Users\patri\vrp_project\data\audit\locked_2621_tenor_priority_bakeoff_conditional_v0_1.csv
C:\Users\patri\vrp_project\data\audit\locked_2621_tenor_priority_metrics_conditional_v0_1.csv
C:\Users\patri\vrp_project\data\audit\locked_2621_tenor_priority_overlap_vs_win_only_conditional_v0_1.csv
C:\Users\patri\vrp_project\data\audit\locked_2621_tenor_priority_overlap_vs_pnl_day_only_conditional_v0_1.csv
C:\Users\patri\vrp_project\data\audit\locked_2621_selected_trades_pnl_day_only_conditional_v0_1.csv
C:\Users\patri\vrp_project\data\processed\locked_2621_selected_trades_pnl_day_only_conditional_v0_1.parquet
C:\Users\patri\vrp_project\data\audit\locked_2621_summary_by_layer_pnl_day_only_conditional_v0_1.csv
C:\Users\patri\vrp_project\data\audit\locked_2621_summary_by_tenor_layer_pnl_day_only_conditional_v0_1.csv
C:\Users\patri\vrp_project\data\audit\locked_2621_summary_by_year_layer_pnl_day_only_conditional_v0_1.csv
C:\Users\patri\vrp_project\data\aud

In [13]:
# Cell — Final locked 2621 blended tenor-priority model
#
# Final model convention:
#   1. Thresholds are locked to 2621.
#   2. Core is checked first.
#   3. Secondary is checked only if no Core tenor qualifies.
#   4. Within the active layer:
#        a. Rank tenors by layer-specific 2621-conditional win probability.
#        b. Keep tenors within 25 bps of the best qualifying tenor's win probability.
#        c. Select the tenor with highest layer-specific 2621-conditional avg P&L/day.
#        d. Tie-break by aggregate P&L/day, then conditional win probability, then longer tenor.
#
# Why:
#   - Avoids raw P&L duration bias.
#   - Avoids unconditional tenor statistics.
#   - Avoids pure P&L/day selection taking over the model.
#   - Keeps win probability as the primary criterion.
#
# Expected tenor behavior from bakeoff:
#   - Same front/middle/back mix as win_only.
#   - Main change: inside the back bucket, 27D can beat 33D when their conditional
#     win probabilities are effectively tied and 27D has better P&L/day.

import numpy as np
import pandas as pd
import json
from pathlib import Path

# ---------------------------------------------------------------------
# Robust setup
# ---------------------------------------------------------------------

if "TENOR_ORDER" in globals():
    TENOR_ARRAY = np.array(TENOR_ORDER, dtype=int)

elif "sweep_panel" in globals() and "tenor" in sweep_panel.columns:
    TENOR_ARRAY = np.array(sorted(sweep_panel["tenor"].dropna().unique()), dtype=int)
    TENOR_ORDER = TENOR_ARRAY.tolist()

elif "research_panel" in globals() and "tenor" in research_panel.columns:
    TENOR_ARRAY = np.array(sorted(research_panel["tenor"].dropna().unique()), dtype=int)
    TENOR_ORDER = TENOR_ARRAY.tolist()

else:
    TENOR_ARRAY = np.array([9, 12, 15, 18, 21, 24, 27, 30, 33], dtype=int)
    TENOR_ORDER = TENOR_ARRAY.tolist()

TENOR_TO_COL = {int(t): i for i, t in enumerate(TENOR_ARRAY)}

required_objects = [
    "num_sweep_dates",
    "sweep_dates",
    "vrp_mat",
    "z3m_mat",
    "z1y_mat",
    "win_mat",
    "pnl_mat",
    "actual_dte_mat",
    "rv21d_by_date",
    "rsi_by_date",
    "TRADE_FREQUENCY_DENOMINATOR",
]

missing_objects = [name for name in required_objects if name not in globals()]

if missing_objects:
    raise NameError(
        "Missing required objects. Run the data-loading / matrix-construction cells first. "
        f"Missing: {missing_objects}"
    )

if "pnl_per_day_mat" not in globals():
    with np.errstate(divide="ignore", invalid="ignore"):
        pnl_per_day_mat = pnl_mat / actual_dte_mat
    pnl_per_day_mat[~np.isfinite(pnl_per_day_mat)] = np.nan

if "AUDIT_DIR" not in globals():
    AUDIT_DIR = Path("data/audit")
else:
    AUDIT_DIR = Path(AUDIT_DIR)

if "PROCESSED_DIR" not in globals():
    PROCESSED_DIR = Path("data/processed")
else:
    PROCESSED_DIR = Path(PROCESSED_DIR)

AUDIT_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

MODEL_LABEL = "locked_2621_win_band_25bps_conditional"
WIN_BAND = 0.0025

print("Final locked 2621 blended-priority model")
print("Model label:", MODEL_LABEL)
print("Tenors:", TENOR_ARRAY.tolist())
print("Win band:", WIN_BAND)


# ---------------------------------------------------------------------
# Locked 2621 thresholds
# ---------------------------------------------------------------------

LOCKED_2621_CORE = {
    "front": {
        "tenors": [9, 12, 15],
        "vrp": 0.60,
        "z3m": 0.55,
        "z1y": 0.65,
        "rsi_cap": 70,
        "rv21d_floor": 8.5,
    },
    "middle": {
        "tenors": [18, 21, 24],
        "vrp": 0.65,
        "z3m": 0.75,
        "z1y": 0.65,
        "rsi_cap": 68,
        "rv21d_floor": 8.5,
    },
    "back": {
        "tenors": [27, 30, 33],
        "vrp": 0.70,
        "z3m": 0.75,
        "z1y": 0.75,
        "rsi_cap": 66,
        "rv21d_floor": 8.5,
    },
}

LOCKED_2621_SECONDARY = {
    "front": {
        "tenors": [9, 12, 15],
        "vrp": 0.60,
        "z3m": 0.50,
        "z1y": 0.40,
        "rsi_cap": 74,
        "rv21d_floor": 6.5,
    },
    "middle": {
        "tenors": [18, 21, 24],
        "vrp": 0.60,
        "z3m": 0.50,
        "z1y": 0.50,
        "rsi_cap": 70,
        "rv21d_floor": 6.5,
    },
    "back": {
        "tenors": [27, 30, 33],
        "vrp": 0.70,
        "z3m": 0.50,
        "z1y": 0.50,
        "rsi_cap": 68,
        "rv21d_floor": 6.5,
    },
}


# ---------------------------------------------------------------------
# Helper functions
# ---------------------------------------------------------------------

def tenor_group_for_tenor(tenor):
    tenor = int(tenor)

    if tenor in [9, 12, 15]:
        return "front"
    elif tenor in [18, 21, 24]:
        return "middle"
    elif tenor in [27, 30, 33]:
        return "back"
    else:
        raise ValueError(f"Unexpected tenor: {tenor}")


def build_threshold_table(threshold_spec, layer):
    rows = []

    for group, params in threshold_spec.items():
        for tenor in params["tenors"]:
            rows.append({
                "layer": layer,
                "tenor_group": group,
                "selected_tenor": int(tenor),
                "vrp_threshold": float(params["vrp"]),
                "z3m_threshold": float(params["z3m"]),
                "z1y_threshold": float(params["z1y"]),
                "rsi_cap": float(params["rsi_cap"]),
                "rv21d_floor": float(params["rv21d_floor"]),
            })

    return pd.DataFrame(rows)


def build_qualifies_matrix(threshold_spec):
    """
    Builds date x tenor boolean qualification matrix for locked 2621 thresholds.
    """
    qualifies = np.zeros((num_sweep_dates, len(TENOR_ARRAY)), dtype=bool)

    for col, tenor in enumerate(TENOR_ARRAY):
        group = tenor_group_for_tenor(tenor)
        params = threshold_spec[group]

        qualifies[:, col] = (
            (vrp_mat[:, col] >= float(params["vrp"])) &
            (z3m_mat[:, col] >= float(params["z3m"])) &
            (z1y_mat[:, col] >= float(params["z1y"])) &
            (rsi_by_date <= float(params["rsi_cap"])) &
            (rv21d_by_date >= float(params["rv21d_floor"]))
        )

    return qualifies


def build_layer_conditional_tenor_metrics(layer_name, qualifies_matrix, date_mask):
    """
    Builds tenor ranking metrics conditional on the relevant 2621 candidate universe.

    Core:
      date_mask = all dates.

    Secondary:
      date_mask = dates where no Core tenor qualifies.
    """
    rows = []

    for col, tenor in enumerate(TENOR_ARRAY):
        candidate_mask = (
            date_mask &
            qualifies_matrix[:, col] &
            np.isfinite(win_mat[:, col]) &
            np.isfinite(pnl_mat[:, col]) &
            np.isfinite(actual_dte_mat[:, col]) &
            np.isfinite(pnl_per_day_mat[:, col])
        )

        candidate_count = int(candidate_mask.sum())

        if candidate_count == 0:
            rows.append({
                "layer": layer_name,
                "selected_col": int(col),
                "selected_tenor": int(tenor),
                "tenor_group": tenor_group_for_tenor(tenor),
                "conditional_win_probability": np.nan,
                "conditional_avg_pnl_per_day": np.nan,
                "conditional_median_pnl_per_day": np.nan,
                "conditional_aggregate_pnl_per_day": np.nan,
                "conditional_avg_raw_pnl": np.nan,
                "conditional_avg_actual_dte": np.nan,
                "conditional_worst_trade_pnl": np.nan,
                "conditional_worst_pnl_per_day": np.nan,
                "candidate_count": 0,
            })
            continue

        rows.append({
            "layer": layer_name,
            "selected_col": int(col),
            "selected_tenor": int(tenor),
            "tenor_group": tenor_group_for_tenor(tenor),

            "conditional_win_probability": float(np.nanmean(win_mat[candidate_mask, col])),
            "conditional_avg_pnl_per_day": float(np.nanmean(pnl_per_day_mat[candidate_mask, col])),
            "conditional_median_pnl_per_day": float(np.nanmedian(pnl_per_day_mat[candidate_mask, col])),
            "conditional_aggregate_pnl_per_day": float(
                np.nansum(pnl_mat[candidate_mask, col]) /
                np.nansum(actual_dte_mat[candidate_mask, col])
            ),
            "conditional_avg_raw_pnl": float(np.nanmean(pnl_mat[candidate_mask, col])),
            "conditional_avg_actual_dte": float(np.nanmean(actual_dte_mat[candidate_mask, col])),
            "conditional_worst_trade_pnl": float(np.nanmin(pnl_mat[candidate_mask, col])),
            "conditional_worst_pnl_per_day": float(np.nanmin(pnl_per_day_mat[candidate_mask, col])),
            "candidate_count": candidate_count,
        })

    return pd.DataFrame(rows)


def blended_priority_order(layer_name, qualifying_cols, win_band=WIN_BAND):
    """
    Final blended priority rule.

    Step 1:
      Find best conditional win probability among qualifying tenors.

    Step 2:
      Keep tenors within win_band of that best conditional win probability.

    Step 3:
      Select highest conditional avg P&L/day.
      Tie-break by aggregate P&L/day, then conditional win probability, then longer tenor.
    """
    if len(qualifying_cols) == 0:
        return pd.DataFrame()

    df = tenor_priority_metrics[
        (tenor_priority_metrics["layer"].eq(layer_name)) &
        (tenor_priority_metrics["selected_col"].isin(qualifying_cols)) &
        (tenor_priority_metrics["candidate_count"] > 0)
    ].copy()

    if df.empty:
        return df

    best_win = df["conditional_win_probability"].max()

    df["best_conditional_win_probability"] = best_win
    df["win_probability_gap_to_best"] = best_win - df["conditional_win_probability"]
    df["inside_win_band"] = df["win_probability_gap_to_best"] <= win_band

    df = df[df["inside_win_band"]].copy()

    df = df.sort_values(
        [
            "conditional_avg_pnl_per_day",
            "conditional_aggregate_pnl_per_day",
            "conditional_win_probability",
            "selected_tenor",
        ],
        ascending=[False, False, False, False],
    ).reset_index(drop=True)

    df["priority_rank_within_signal"] = np.arange(1, len(df) + 1)

    return df


def select_final_model_path():
    """
    Core-first / Secondary-second final path selection.
    """
    selected_rows = []

    core_selected = np.full(num_sweep_dates, -1, dtype=np.int8)
    secondary_selected = np.full(num_sweep_dates, -1, dtype=np.int8)
    combined_selected = np.full(num_sweep_dates, -1, dtype=np.int8)

    for row in range(num_sweep_dates):
        core_qualifying_cols = np.where(core_qualifies_matrix[row, :])[0].astype(int).tolist()

        if core_qualifying_cols:
            ordered = blended_priority_order(
                layer_name="Core",
                qualifying_cols=core_qualifying_cols,
                win_band=WIN_BAND,
            )

            if not ordered.empty:
                selected_col = int(ordered.iloc[0]["selected_col"])

                core_selected[row] = selected_col
                combined_selected[row] = selected_col

                selected_rows.append({
                    "row_idx": int(row),
                    "date": sweep_dates[row],
                    "layer": "Core",
                    "selected_col": selected_col,
                    "selected_tenor": int(TENOR_ARRAY[selected_col]),
                    "tenor_group": tenor_group_for_tenor(TENOR_ARRAY[selected_col]),
                    "num_qualifying_tenors_in_layer": int(len(core_qualifying_cols)),
                    "qualifying_tenors_in_layer": ",".join(
                        [str(int(TENOR_ARRAY[c])) for c in core_qualifying_cols]
                    ),
                    "selection_rule": "win_band_25bps_then_pnl_day",
                    "win_band": WIN_BAND,
                })

            continue

        secondary_qualifying_cols = np.where(secondary_qualifies_matrix[row, :])[0].astype(int).tolist()

        if secondary_qualifying_cols:
            ordered = blended_priority_order(
                layer_name="Secondary",
                qualifying_cols=secondary_qualifying_cols,
                win_band=WIN_BAND,
            )

            if not ordered.empty:
                selected_col = int(ordered.iloc[0]["selected_col"])

                secondary_selected[row] = selected_col
                combined_selected[row] = selected_col

                selected_rows.append({
                    "row_idx": int(row),
                    "date": sweep_dates[row],
                    "layer": "Secondary",
                    "selected_col": selected_col,
                    "selected_tenor": int(TENOR_ARRAY[selected_col]),
                    "tenor_group": tenor_group_for_tenor(TENOR_ARRAY[selected_col]),
                    "num_qualifying_tenors_in_layer": int(len(secondary_qualifying_cols)),
                    "qualifying_tenors_in_layer": ",".join(
                        [str(int(TENOR_ARRAY[c])) for c in secondary_qualifying_cols]
                    ),
                    "selection_rule": "win_band_25bps_then_pnl_day",
                    "win_band": WIN_BAND,
                })

    selected_path = pd.DataFrame(selected_rows)

    return selected_path, {
        "core_selected": core_selected,
        "secondary_selected": secondary_selected,
        "combined_selected": combined_selected,
    }


def attach_trade_outcomes(selected_path):
    """
    Adds outcomes, features, and conditional ranking metrics to selected path.
    """
    if selected_path.empty:
        return selected_path.copy()

    row_idx = selected_path["row_idx"].to_numpy(dtype=int)
    col_idx = selected_path["selected_col"].to_numpy(dtype=int)

    trades = selected_path.copy()

    trades["candidate"] = MODEL_LABEL

    trades["win"] = win_mat[row_idx, col_idx]
    trades["normalized_pnl_dollars"] = pnl_mat[row_idx, col_idx]
    trades["actual_dte"] = actual_dte_mat[row_idx, col_idx]
    trades["pnl_per_day"] = pnl_per_day_mat[row_idx, col_idx]

    trades["vrp_log"] = vrp_mat[row_idx, col_idx]
    trades["vrp_z_3m"] = z3m_mat[row_idx, col_idx]
    trades["vrp_z_1y"] = z1y_mat[row_idx, col_idx]
    trades["rv21d"] = rv21d_by_date[row_idx]
    trades["rsi14"] = rsi_by_date[row_idx]

    if "spx_close_mat" in globals():
        trades["spx_close"] = spx_close_mat[row_idx, col_idx]
    elif "spx_close_by_date" in globals():
        trades["spx_close"] = np.asarray(spx_close_by_date)[row_idx]
    else:
        trades["spx_close"] = np.nan

    trades = trades.merge(
        tenor_priority_metrics,
        on=["layer", "selected_col", "selected_tenor", "tenor_group"],
        how="left",
    )

    trades = trades.sort_values("date").reset_index(drop=True)

    trades["trade_sequence"] = np.arange(1, len(trades) + 1)
    trades["cum_pnl"] = trades["normalized_pnl_dollars"].cumsum()
    trades["running_max_cum_pnl"] = trades["cum_pnl"].cummax()
    trades["drawdown"] = trades["cum_pnl"] - trades["running_max_cum_pnl"]
    trades["rolling_20_trade_pnl"] = trades["normalized_pnl_dollars"].rolling(20).sum()

    trades["year"] = pd.to_datetime(trades["date"]).dt.year
    trades["month"] = pd.to_datetime(trades["date"]).dt.to_period("M").astype(str)

    return trades


def summarize_trade_df(df, label):
    """
    Summary for one path or subgroup.
    """
    trade_count = int(len(df))

    if trade_count == 0:
        return pd.Series({
            "label": label,
            "trade_count": 0,
            "trade_frequency": 0.0,
            "win_rate": np.nan,
            "avg_pnl_per_day": np.nan,
            "median_pnl_per_day": np.nan,
            "aggregate_pnl_per_day": np.nan,
            "total_pnl": 0.0,
            "total_actual_dte": 0.0,
            "worst_trade_pnl": np.nan,
            "worst_pnl_per_day": np.nan,
            "max_drawdown": np.nan,
            "worst_20_trade_sum": np.nan,
            "avg_selected_tenor": np.nan,
        })

    df = df.sort_values("date").copy()

    df["local_cum_pnl"] = df["normalized_pnl_dollars"].cumsum()
    df["local_running_max"] = df["local_cum_pnl"].cummax()
    df["local_drawdown"] = df["local_cum_pnl"] - df["local_running_max"]
    df["local_rolling_20_trade_pnl"] = df["normalized_pnl_dollars"].rolling(20).sum()

    total_pnl = float(df["normalized_pnl_dollars"].sum())
    total_actual_dte = float(df["actual_dte"].sum())

    return pd.Series({
        "label": label,
        "trade_count": trade_count,
        "trade_frequency": float(trade_count / TRADE_FREQUENCY_DENOMINATOR),
        "win_rate": float(df["win"].mean()),
        "avg_pnl_per_day": float(df["pnl_per_day"].mean()),
        "median_pnl_per_day": float(df["pnl_per_day"].median()),
        "aggregate_pnl_per_day": float(total_pnl / total_actual_dte) if total_actual_dte else np.nan,
        "total_pnl": total_pnl,
        "total_actual_dte": total_actual_dte,
        "worst_trade_pnl": float(df["normalized_pnl_dollars"].min()),
        "worst_pnl_per_day": float(df["pnl_per_day"].min()),
        "max_drawdown": float(df["local_drawdown"].min()),
        "worst_20_trade_sum": float(df["local_rolling_20_trade_pnl"].min()),
        "avg_selected_tenor": float(df["selected_tenor"].mean()),
    })


def summarize_by_group(df, group_cols):
    rows = []

    for keys, sub in df.groupby(group_cols, dropna=False):
        if not isinstance(keys, tuple):
            keys = (keys,)

        row = {col: key for col, key in zip(group_cols, keys)}
        summary = summarize_trade_df(sub, label="group").to_dict()
        summary.pop("label", None)

        row.update(summary)
        rows.append(row)

    return pd.DataFrame(rows)


# ---------------------------------------------------------------------
# Build qualification matrices and conditional tenor metrics
# ---------------------------------------------------------------------

core_qualifies_matrix = build_qualifies_matrix(LOCKED_2621_CORE)
secondary_qualifies_matrix = build_qualifies_matrix(LOCKED_2621_SECONDARY)

any_core_qualifies_by_date = core_qualifies_matrix.any(axis=1)
no_core_qualifies_by_date = ~any_core_qualifies_by_date

core_tenor_priority_metrics = build_layer_conditional_tenor_metrics(
    layer_name="Core",
    qualifies_matrix=core_qualifies_matrix,
    date_mask=np.ones(num_sweep_dates, dtype=bool),
)

secondary_tenor_priority_metrics = build_layer_conditional_tenor_metrics(
    layer_name="Secondary",
    qualifies_matrix=secondary_qualifies_matrix,
    date_mask=no_core_qualifies_by_date,
)

tenor_priority_metrics = pd.concat(
    [core_tenor_priority_metrics, secondary_tenor_priority_metrics],
    ignore_index=True,
)

locked_2621_threshold_table = pd.concat(
    [
        build_threshold_table(LOCKED_2621_CORE, "Core"),
        build_threshold_table(LOCKED_2621_SECONDARY, "Secondary"),
    ],
    ignore_index=True,
)

print("\nLocked 2621 qualification counts:")
print("Dates with at least one Core tenor:", int(any_core_qualifies_by_date.sum()))
print("Dates with no Core tenor:", int(no_core_qualifies_by_date.sum()))
print("Dates with at least one Secondary tenor:", int(secondary_qualifies_matrix.any(axis=1).sum()))
print(
    "Dates with at least one Secondary tenor and no Core tenor:",
    int((secondary_qualifies_matrix.any(axis=1) & no_core_qualifies_by_date).sum()),
)

print("\nCore 2621-conditional tenor priority metrics:")
display(
    core_tenor_priority_metrics.sort_values(
        ["conditional_win_probability", "conditional_avg_pnl_per_day"],
        ascending=[False, False],
    )
)

print("\nSecondary 2621-conditional tenor priority metrics:")
display(
    secondary_tenor_priority_metrics.sort_values(
        ["conditional_win_probability", "conditional_avg_pnl_per_day"],
        ascending=[False, False],
    )
)


# ---------------------------------------------------------------------
# Select final model path
# ---------------------------------------------------------------------

selected_path, selected_arrays = select_final_model_path()
locked_2621_selected_trades = attach_trade_outcomes(selected_path)

locked_2621_path_summary = pd.DataFrame(
    [summarize_trade_df(locked_2621_selected_trades, label="Combined")]
)

summary_by_layer = summarize_by_group(
    locked_2621_selected_trades,
    ["layer"],
)

summary_by_year_layer = summarize_by_group(
    locked_2621_selected_trades,
    ["year", "layer"],
)

summary_by_tenor_layer = summarize_by_group(
    locked_2621_selected_trades,
    ["selected_tenor", "layer"],
)

summary_by_group_layer = summarize_by_group(
    locked_2621_selected_trades,
    ["tenor_group", "layer"],
)

worst_trades = (
    locked_2621_selected_trades
    .sort_values("normalized_pnl_dollars", ascending=True)
    .head(20)
    .reset_index(drop=True)
)

worst_20_trade_windows = (
    locked_2621_selected_trades
    .dropna(subset=["rolling_20_trade_pnl"])
    .sort_values("rolling_20_trade_pnl", ascending=True)
    .head(20)
    .reset_index(drop=True)
)

print("\nFinal locked 2621 blended-priority path summary:")
display(locked_2621_path_summary)

print("\nSummary by layer:")
display(summary_by_layer)

print("\nSummary by year and layer:")
display(summary_by_year_layer)

print("\nSummary by tenor and layer:")
display(summary_by_tenor_layer)

print("\nSummary by tenor group and layer:")
display(summary_by_group_layer)

print("\nWorst 20 individual trades:")
display(worst_trades)

print("\nWorst 20-trade rolling windows:")
display(worst_20_trade_windows)


# ---------------------------------------------------------------------
# Save final model outputs
# ---------------------------------------------------------------------

SELECTED_TRADES_CSV_PATH = AUDIT_DIR / f"{MODEL_LABEL}_selected_trades_v0_1.csv"
SELECTED_TRADES_PARQUET_PATH = PROCESSED_DIR / f"{MODEL_LABEL}_selected_trades_v0_1.parquet"

SUMMARY_PATH = AUDIT_DIR / f"{MODEL_LABEL}_summary_v0_1.csv"
SUMMARY_BY_LAYER_PATH = AUDIT_DIR / f"{MODEL_LABEL}_summary_by_layer_v0_1.csv"
SUMMARY_BY_YEAR_LAYER_PATH = AUDIT_DIR / f"{MODEL_LABEL}_summary_by_year_layer_v0_1.csv"
SUMMARY_BY_TENOR_LAYER_PATH = AUDIT_DIR / f"{MODEL_LABEL}_summary_by_tenor_layer_v0_1.csv"
SUMMARY_BY_GROUP_LAYER_PATH = AUDIT_DIR / f"{MODEL_LABEL}_summary_by_group_layer_v0_1.csv"

WORST_TRADES_PATH = AUDIT_DIR / f"{MODEL_LABEL}_worst_trades_v0_1.csv"
WORST_WINDOWS_PATH = AUDIT_DIR / f"{MODEL_LABEL}_worst_20_trade_windows_v0_1.csv"

TENOR_PRIORITY_METRICS_PATH = AUDIT_DIR / f"{MODEL_LABEL}_tenor_priority_metrics_v0_1.csv"
THRESHOLD_TABLE_PATH = AUDIT_DIR / f"{MODEL_LABEL}_thresholds_v0_1.csv"
SUMMARY_JSON_PATH = AUDIT_DIR / f"{MODEL_LABEL}_summary_v0_1.json"

locked_2621_selected_trades.to_csv(SELECTED_TRADES_CSV_PATH, index=False)
locked_2621_selected_trades.to_parquet(SELECTED_TRADES_PARQUET_PATH, index=False)

locked_2621_path_summary.to_csv(SUMMARY_PATH, index=False)
summary_by_layer.to_csv(SUMMARY_BY_LAYER_PATH, index=False)
summary_by_year_layer.to_csv(SUMMARY_BY_YEAR_LAYER_PATH, index=False)
summary_by_tenor_layer.to_csv(SUMMARY_BY_TENOR_LAYER_PATH, index=False)
summary_by_group_layer.to_csv(SUMMARY_BY_GROUP_LAYER_PATH, index=False)

worst_trades.to_csv(WORST_TRADES_PATH, index=False)
worst_20_trade_windows.to_csv(WORST_WINDOWS_PATH, index=False)

tenor_priority_metrics.to_csv(TENOR_PRIORITY_METRICS_PATH, index=False)
locked_2621_threshold_table.to_csv(THRESHOLD_TABLE_PATH, index=False)

summary_json = {
    "model_label": MODEL_LABEL,
    "thresholds": "locked_2621",
    "priority_rule": "layer-specific 2621-conditional win probability within 25 bps, then conditional avg P&L/day",
    "win_band": WIN_BAND,
    "trade_count": int(locked_2621_path_summary.iloc[0]["trade_count"]),
    "trade_frequency": float(locked_2621_path_summary.iloc[0]["trade_frequency"]),
    "win_rate": float(locked_2621_path_summary.iloc[0]["win_rate"]),
    "avg_pnl_per_day": float(locked_2621_path_summary.iloc[0]["avg_pnl_per_day"]),
    "aggregate_pnl_per_day": float(locked_2621_path_summary.iloc[0]["aggregate_pnl_per_day"]),
    "total_pnl": float(locked_2621_path_summary.iloc[0]["total_pnl"]),
    "max_drawdown": float(locked_2621_path_summary.iloc[0]["max_drawdown"]),
    "worst_20_trade_sum": float(locked_2621_path_summary.iloc[0]["worst_20_trade_sum"]),
    "avg_selected_tenor": float(locked_2621_path_summary.iloc[0]["avg_selected_tenor"]),
    "selected_trades_csv": str(SELECTED_TRADES_CSV_PATH),
    "selected_trades_parquet": str(SELECTED_TRADES_PARQUET_PATH),
}

with open(SUMMARY_JSON_PATH, "w") as f:
    json.dump(summary_json, f, indent=2)

print("\nSaved final locked 2621 blended-priority outputs:")
print(SELECTED_TRADES_CSV_PATH)
print(SELECTED_TRADES_PARQUET_PATH)
print(SUMMARY_PATH)
print(SUMMARY_BY_LAYER_PATH)
print(SUMMARY_BY_YEAR_LAYER_PATH)
print(SUMMARY_BY_TENOR_LAYER_PATH)
print(SUMMARY_BY_GROUP_LAYER_PATH)
print(WORST_TRADES_PATH)
print(WORST_WINDOWS_PATH)
print(TENOR_PRIORITY_METRICS_PATH)
print(THRESHOLD_TABLE_PATH)
print(SUMMARY_JSON_PATH)

print("\nFinal locked 2621 blended-priority model complete.")

Final locked 2621 blended-priority model
Model label: locked_2621_win_band_25bps_conditional
Tenors: [9, 12, 15, 18, 21, 24, 27, 30, 33]
Win band: 0.0025

Locked 2621 qualification counts:
Dates with at least one Core tenor: 386
Dates with no Core tenor: 1339
Dates with at least one Secondary tenor: 577
Dates with at least one Secondary tenor and no Core tenor: 191

Core 2621-conditional tenor priority metrics:


,layer,selected_col,selected_tenor,tenor_group,conditional_win_probability,conditional_avg_pnl_per_day,conditional_median_pnl_per_day,conditional_aggregate_pnl_per_day,conditional_avg_raw_pnl,conditional_avg_actual_dte,conditional_worst_trade_pnl,conditional_worst_pnl_per_day,candidate_count
8,Core,8,33,back,0.895522,431.999112,499.797363,430.069297,"14,063.907900",32.701493,"-114,955.549452","-3,193.209707",201
6,Core,6,27,back,0.894231,490.191037,554.917740,488.341492,"13,262.697550",27.158654,"-57,319.352129","-2,292.774085",208
7,Core,7,30,back,0.880000,440.960597,529.383338,440.761991,"13,156.745421",29.850000,"-57,036.901356","-2,037.032191",200
5,Core,5,24,middle,0.851240,426.816292,589.114564,430.610667,"9,886.251501",22.958678,"-57,319.352129","-2,469.960158",242
4,Core,4,21,middle,0.834783,451.823181,613.672423,450.613423,"9,811.617490",21.773913,"-56,809.083631","-2,487.128419",230
3,Core,3,18,middle,0.799213,445.583034,683.385498,451.521041,"7,805.625564",17.287402,"-45,195.769858","-2,716.007704",254
2,Core,2,15,front,0.771523,409.251409,706.225901,413.442202,"6,584.956923",15.927152,"-46,130.619927","-3,295.044280",302
0,Core,0,9,front,0.749164,459.029727,989.749243,454.055688,"3,924.013040",8.642140,"-60,891.722904","-6,576.872609",299
1,Core,1,12,front,0.745763,402.590954,843.753027,389.340289,"4,612.692582",11.847458,"-56,963.300165","-4,926.141041",295



Secondary 2621-conditional tenor priority metrics:


,layer,selected_col,selected_tenor,tenor_group,conditional_win_probability,conditional_avg_pnl_per_day,conditional_median_pnl_per_day,conditional_aggregate_pnl_per_day,conditional_avg_raw_pnl,conditional_avg_actual_dte,conditional_worst_trade_pnl,conditional_worst_pnl_per_day,candidate_count
8,Secondary,8,33,back,0.772152,184.108263,409.122134,182.709066,"6,003.958689",32.860759,"-87,932.829005","-2,512.366543",79
7,Secondary,7,30,back,0.739726,164.877502,409.122134,165.283628,"4,924.546456",29.794521,"-33,497.150782","-1,046.785962",73
2,Secondary,2,15,front,0.730435,168.629531,491.896754,164.934882,"2,650.431838",16.069565,"-42,928.530585","-2,384.918366",115
6,Secondary,6,27,back,0.710145,105.979093,395.526250,114.435826,"3,134.546532",27.391304,"-44,003.345303","-1,833.472721",69
3,Secondary,3,18,middle,0.703704,154.582398,490.623102,159.492943,"2,725.163367",17.086420,"-37,360.735722","-2,197.690337",81
5,Secondary,5,24,middle,0.703704,152.966157,444.563191,147.733099,"3,365.031706",22.777778,"-44,003.345303","-1,833.472721",81
0,Secondary,0,9,front,0.683333,107.213381,656.402509,71.258978,614.014859,8.616667,"-27,346.824178","-3,755.720924",120
4,Secondary,4,21,middle,0.662162,104.292259,448.669768,87.565238,"1,894.485762",21.635135,"-44,003.345303","-1,833.472721",74
1,Secondary,1,12,front,0.658120,-1.777573,515.605288,26.556819,304.155017,11.452991,"-29,580.217937","-3,038.536020",117



Final locked 2621 blended-priority path summary:


...


Worst 20 individual trades:


,row_idx,date,layer,selected_col,selected_tenor,tenor_group,num_qualifying_tenors_in_layer,qualifying_tenors_in_layer,selection_rule,win_band,candidate,win,normalized_pnl_dollars,actual_dte,pnl_per_day,vrp_log,vrp_z_3m,vrp_z_1y,rv21d,rsi14,spx_close,conditional_win_probability,conditional_avg_pnl_per_day,conditional_median_pnl_per_day,conditional_aggregate_pnl_per_day,conditional_avg_raw_pnl,conditional_avg_actual_dte,conditional_worst_trade_pnl,conditional_worst_pnl_per_day,candidate_count,trade_sequence,cum_pnl,running_max_cum_pnl,drawdown,rolling_20_trade_pnl,year,month
0,137,2020-01-24,Secondary,8,33,back,9,"9,12,15,18,21,24,27,30,33",win_band_25bps_then_pnl_day,0.002500,locked_2621_win_band_25bps_conditional,0.000000,"-87,932.829005",35,"-2,512.366543",1.300539,1.061030,1.738160,7.866687,61.573549,NaN,0.772152,184.108263,409.122134,182.709066,"6,003.958689",32.860759,"-87,932.829005","-2,512.366543",79,54,"111,267.485524","199,200.314529","-87,932.829005","101,620.934570",2020,2020-01
1,157,2020-02-24,Core,0,9,front,1,9,win_band_25bps_then_pnl_day,0.002500,locked_2621_win_band_25bps_conditional,0.000000,"-60,891.722904",11,"-5,535.611173",1.425272,0.638828,1.491332,18.033963,36.699503,NaN,0.749164,459.029727,989.749243,454.055688,"3,924.013040",8.642140,"-60,891.722904","-6,576.872609",299,60,"110,753.000943","199,200.314529","-88,447.313586","35,879.032276",2020,2020-02
2,1669,2026-03-02,Core,6,27,back,9,"9,12,15,18,21,24,27,30,33",win_band_25bps_then_pnl_day,0.002500,locked_2621_win_band_25bps_conditional,0.000000,"-57,319.352129",25,"-2,292.774085",1.222648,1.671398,1.140005,12.896987,48.647576,NaN,0.894231,490.191037,554.917740,488.341492,"13,262.697550",27.158654,"-57,319.352129","-2,292.774085",208,557,"4,618,965.009047","4,840,074.986348","-221,109.977301","-182,593.574628",2026,2026-03
3,1668,2026-02-27,Core,6,27,back,9,"9,12,15,18,21,24,27,30,33",win_band_25bps_then_pnl_day,0.002500,locked_2621_win_band_25bps_conditional,0.000000,"-57,036.901356",28,"-2,037.032191",1.001127,0.847920,0.778522,12.893787,48.384049,NaN,0.894231,490.191037,554.917740,488.341492,"13,262.697550",27.158654,"-57,319.352129","-2,292.774085",208,556,"4,676,284.361177","4,840,074.986348","-163,790.625172","-117,677.729809",2026,2026-02
4,1662,2026-02-19,Core,8,33,back,8,"9,12,15,18,21,24,30,33",win_band_25bps_then_pnl_day,0.002500,locked_2621_win_band_25bps_conditional,0.000000,"-52,092.062100",36,"-1,447.001725",1.045459,0.772548,0.800369,12.300219,46.444609,NaN,0.895522,431.999112,499.797363,430.069297,"14,063.907900",32.701493,"-114,955.549452","-3,193.209707",201,552,"4,787,982.924249","4,840,074.986348","-52,092.062100","32,019.152907",2026,2026-02
5,1413,2025-02-21,Core,5,24,middle,3,"9,15,24",win_band_25bps_then_pnl_day,0.002500,locked_2621_win_band_25bps_conditional,0.000000,"-49,418.522467",21,"-2,353.262975",0.741002,0.996426,0.660997,11.880124,46.090761,NaN,0.851240,426.816292,589.114564,430.610667,"9,886.251501",22.958678,"-57,319.352129","-2,469.960158",242,457,"4,164,457.294878","4,213,875.817345","-49,418.522467","150,375.347720",2025,2025-02
6,1416,2025-02-26,Core,6,27,back,9,"9,12,15,18,21,24,27,30,33",win_band_25bps_then_pnl_day,0.002500,locked_2621_win_band_25bps_conditional,0.000000,"-47,172.123854",30,"-1,572.404128",0.946285,1.605369,1.141095,10.753467,41.127776,NaN,0.894231,490.191037,554.917740,488.341492,"13,262.697550",27.158654,"-57,319.352129","-2,292.774085",208,460,"4,046,285.657359","4,213,875.817345","-167,590.159986","-11,658.747642",2025,2025-02
7,161,2020-02-28,Core,2,15,front,3,"9,12,15",win_band_25bps_then_pnl_day,0.002500,locked_2621_win_band_25bps_conditional,0.000000,"-46,130.619927",14,"-3,295.044280",1.440955,0.832745,1.691735,24.483900,19.164013,NaN,0.771523,409.251409,706.225901,413.442202,"6,584.956923",15.927152,"-46,130.619927","-3,295.044280",302,62,"90,059.139936","199,200.314529","-109,141.174593","3,520.764818",2020,2020-02
8,1261,2024-07-15,Secondary,2,15,front,3,"9,12,15",win_band_25bps_th


Worst 20-trade rolling windows:


,row_idx,date,layer,selected_col,selected_tenor,tenor_group,num_qualifying_tenors_in_layer,qualifying_tenors_in_layer,selection_rule,win_band,candidate,win,normalized_pnl_dollars,actual_dte,pnl_per_day,vrp_log,vrp_z_3m,vrp_z_1y,rv21d,rsi14,spx_close,conditional_win_probability,conditional_avg_pnl_per_day,conditional_median_pnl_per_day,conditional_aggregate_pnl_per_day,conditional_avg_raw_pnl,conditional_avg_actual_dte,conditional_worst_trade_pnl,conditional_worst_pnl_per_day,candidate_count,trade_sequence,cum_pnl,running_max_cum_pnl,drawdown,rolling_20_trade_pnl,year,month
0,1673,2026-03-06,Core,6,27,back,9,"9,12,15,18,21,24,27,30,33",win_band_25bps_then_pnl_day,0.002500,locked_2621_win_band_25bps_conditional,1.000000,339.761603,27,12.583763,1.733877,3.386581,1.941130,13.851073,38.453183,NaN,0.894231,490.191037,554.917740,488.341492,"13,262.697550",27.158654,"-57,319.352129","-2,292.774085",208,561,"4,565,500.729820","4,840,074.986348","-274,574.256528","-247,740.016788",2026,2026-03
1,1674,2026-03-09,Core,6,27,back,9,"9,12,15,18,21,24,27,30,33",win_band_25bps_then_pnl_day,0.002500,locked_2621_win_band_25bps_conditional,0.000000,"-12,685.421844",24,-528.559244,1.436394,2.039998,1.442139,13.563080,43.879944,NaN,0.894231,490.191037,554.917740,488.341492,"13,262.697550",27.158654,"-57,319.352129","-2,292.774085",208,562,"4,552,815.307976","4,840,074.986348","-287,259.678373","-243,668.395666",2026,2026-03
2,1672,2026-03-05,Core,6,27,back,9,"9,12,15,18,21,24,27,30,33",win_band_25bps_then_pnl_day,0.002500,locked_2621_win_band_25bps_conditional,0.000000,"-17,217.829479",28,-614.922481,1.379605,2.120608,1.376651,13.201215,45.000430,NaN,0.894231,490.191037,554.917740,488.341492,"13,262.697550",27.158654,"-57,319.352129","-2,292.774085",208,560,"4,565,160.968217","4,840,074.986348","-274,914.018132","-232,396.065618",2026,2026-03
3,1424,2025-03-10,Core,2,15,front,3,"9,12,15",win_band_25bps_then_pnl_day,0.002500,locked_2621_win_band_25bps_conditional,1.000000,"15,449.118007",18,858.284334,0.935211,1.005079,0.791598,17.789582,29.915345,NaN,0.771523,409.251409,706.225901,413.442202,"6,584.956923",15.927152,"-46,130.619927","-3,295.044280",302,468,"3,959,245.856490","4,213,875.817345","-254,629.960854","-227,955.728566",2025,2025-03
4,1675,2026-03-10,Core,6,27,back,9,"9,12,15,18,21,24,27,30,33",win_band_25bps_then_pnl_day,0.002500,locked_2621_win_band_25bps_conditional,1.000000,"22,148.557542",31,714.469598,1.530865,2.298511,1.583290,11.562767,42.825712,NaN,0.894231,490.191037,554.917740,488.341492,"13,262.697550",27.158654,"-57,319.352129","-2,292.774085",208,563,"4,574,963.865518","4,840,074.986348","-265,111.120831","-227,324.684046",2026,2026-03
5,1423,2025-03-07,Core,2,15,front,2,"9,15",win_band_25bps_then_pnl_day,0.002500,locked_2621_win_band_25bps_conditional,0.000000,"-1,826.626460",14,-130.473319,0.903200,0.923910,0.714830,15.747961,36.833340,NaN,0.771523,409.251409,706.225901,413.442202,"6,584.956923",15.927152,"-46,130.619927","-3,295.044280",302,467,"3,943,796.738483","4,213,875.817345","-270,079.078862","-226,530.134310",2025,2025-03
6,1425,2025-03-11,Core,2,15,front,3,"9,12,15",win_band_25bps_then_pnl_day,0.002500,locked_2621_win_band_25bps_conditional,1.000000,"21,051.422541",17,"1,238.318973",0.890234,0.837298,0.674164,17.724709,28.349890,NaN,0.771523,409.251409,706.225901,413.442202,"6,584.956923",15.927152,"-46,130.619927","-3,295.044280",302,469,"3,980,297.279032","4,213,875.817345","-233,578.538313","-222,433.223013",2025,2025-03
7,1671,2026-03-04,Core,6,27,back,9,"9,12,15,18,21,24,27,30,33",win_band_25bps_then_pnl_day,0.002500,locked_2621_win_band_25bps_conditional,0.000000,"-23,176.359269",29,-799.184802,1.164994,1.328891,1.033053,13.369281,48.264262,NaN,0.894231,490.191037,554.917740,488.341492,"13,262.697550",27.158654,"-57,319.352129","-2,292.774085",208,559,"4,582,378.797696","4,840,074.986348","-257,696.188653","-217,343.939070",2026,2026-03
8,1422,2025-03-06,Core,5,24,middle,6,"9,12,15,18,21,24",win_band_25bps_t


Saved final locked 2621 blended-priority outputs:
C:\Users\patri\vrp_project\data\audit\locked_2621_win_band_25bps_conditional_selected_trades_v0_1.csv
C:\Users\patri\vrp_project\data\processed\locked_2621_win_band_25bps_conditional_selected_trades_v0_1.parquet
C:\Users\patri\vrp_project\data\audit\locked_2621_win_band_25bps_conditional_summary_v0_1.csv
C:\Users\patri\vrp_project\data\audit\locked_2621_win_band_25bps_conditional_summary_by_layer_v0_1.csv
C:\Users\patri\vrp_project\data\audit\locked_2621_win_band_25bps_conditional_summary_by_year_layer_v0_1.csv
C:\Users\patri\vrp_project\data\audit\locked_2621_win_band_25bps_conditional_summary_by_tenor_layer_v0_1.csv
C:\Users\patri\vrp_project\data\audit\locked_2621_win_band_25bps_conditional_summary_by_group_layer_v0_1.csv
C:\Users\patri\vrp_project\data\audit\locked_2621_win_band_25bps_conditional_worst_trades_v0_1.csv
C:\Users\patri\vrp_project\data\audit\locked_2621_win_band_25bps_conditional_worst_20_trade_windows_v0_1.csv
C:\Use

In [14]:
# Cell — 2026 stress diagnostics for locked_2621_win_band_25bps_conditional
#
# Purpose:
#   Investigate why the final locked 2621 blended-priority model struggled in 2026,
#   especially the Core sleeve.
#
# Questions:
#   1. Is 2026 weakness concentrated in specific months?
#   2. Is it concentrated in Core, 27D, or a tenor group?
#   3. Did the 25 bps blend worsen 2026 versus win_only / pnl_day_only?
#   4. On bad 2026 dates, were other qualifying tenors better?
#   5. Are losing 2026 Core trades associated with obvious feature regimes?

import numpy as np
import pandas as pd
from pathlib import Path

# ---------------------------------------------------------------------
# Robust setup
# ---------------------------------------------------------------------

MODEL_LABEL = "locked_2621_win_band_25bps_conditional"

if "locked_2621_selected_trades" not in globals():
    if "PROCESSED_DIR" not in globals():
        PROCESSED_DIR = Path("data/processed")
    else:
        PROCESSED_DIR = Path(PROCESSED_DIR)

    fallback_path = PROCESSED_DIR / f"{MODEL_LABEL}_selected_trades_v0_1.parquet"

    if fallback_path.exists():
        locked_2621_selected_trades = pd.read_parquet(fallback_path)
        print("Loaded selected trades from:", fallback_path)
    else:
        raise NameError(
            "locked_2621_selected_trades is not in memory and saved parquet was not found. "
            "Run the final locked 2621 blended-priority model cell first."
        )

required_objects = [
    "TENOR_ARRAY",
    "num_sweep_dates",
    "sweep_dates",
    "core_qualifies_matrix",
    "secondary_qualifies_matrix",
    "tenor_priority_metrics",
    "win_mat",
    "pnl_mat",
    "actual_dte_mat",
    "pnl_per_day_mat",
    "vrp_mat",
    "z3m_mat",
    "z1y_mat",
    "rv21d_by_date",
    "rsi_by_date",
    "TRADE_FREQUENCY_DENOMINATOR",
]

missing = [name for name in required_objects if name not in globals()]
if missing:
    raise NameError(
        "Missing objects needed for diagnostics. "
        "Run the final locked 2621 blended-priority model cell first. "
        f"Missing: {missing}"
    )

if "AUDIT_DIR" not in globals():
    AUDIT_DIR = Path("data/audit")
else:
    AUDIT_DIR = Path(AUDIT_DIR)

AUDIT_DIR.mkdir(parents=True, exist_ok=True)

trades = locked_2621_selected_trades.copy()
trades["date"] = pd.to_datetime(trades["date"])
trades["year"] = trades["date"].dt.year
trades["month"] = trades["date"].dt.to_period("M").astype(str)
trades["is_2026"] = trades["year"].eq(2026)

print("Selected trades loaded:", len(trades))
print("Date range:", trades["date"].min(), "to", trades["date"].max())


# ---------------------------------------------------------------------
# Summary helpers
# ---------------------------------------------------------------------

def summarize_trade_df(df, label):
    df = df.copy().sort_values("date")
    trade_count = int(len(df))

    if trade_count == 0:
        return pd.Series({
            "label": label,
            "trade_count": 0,
            "trade_frequency": 0.0,
            "win_rate": np.nan,
            "avg_pnl_per_day": np.nan,
            "median_pnl_per_day": np.nan,
            "aggregate_pnl_per_day": np.nan,
            "total_pnl": 0.0,
            "total_actual_dte": 0.0,
            "worst_trade_pnl": np.nan,
            "worst_pnl_per_day": np.nan,
            "max_drawdown": np.nan,
            "worst_20_trade_sum": np.nan,
            "avg_selected_tenor": np.nan,
        })

    df["local_cum_pnl"] = df["normalized_pnl_dollars"].cumsum()
    df["local_running_max"] = df["local_cum_pnl"].cummax()
    df["local_drawdown"] = df["local_cum_pnl"] - df["local_running_max"]
    df["local_rolling_20_trade_pnl"] = df["normalized_pnl_dollars"].rolling(20).sum()

    total_pnl = float(df["normalized_pnl_dollars"].sum())
    total_dte = float(df["actual_dte"].sum())

    return pd.Series({
        "label": label,
        "trade_count": trade_count,
        "trade_frequency": float(trade_count / TRADE_FREQUENCY_DENOMINATOR),
        "win_rate": float(df["win"].mean()),
        "avg_pnl_per_day": float(df["pnl_per_day"].mean()),
        "median_pnl_per_day": float(df["pnl_per_day"].median()),
        "aggregate_pnl_per_day": float(total_pnl / total_dte) if total_dte else np.nan,
        "total_pnl": total_pnl,
        "total_actual_dte": total_dte,
        "worst_trade_pnl": float(df["normalized_pnl_dollars"].min()),
        "worst_pnl_per_day": float(df["pnl_per_day"].min()),
        "max_drawdown": float(df["local_drawdown"].min()),
        "worst_20_trade_sum": float(df["local_rolling_20_trade_pnl"].min()),
        "avg_selected_tenor": float(df["selected_tenor"].mean()),
    })


def summarize_by(df, group_cols):
    rows = []

    for keys, sub in df.groupby(group_cols, dropna=False):
        if not isinstance(keys, tuple):
            keys = (keys,)

        row = {col: key for col, key in zip(group_cols, keys)}
        summary = summarize_trade_df(sub, label="group").to_dict()
        summary.pop("label", None)
        row.update(summary)
        rows.append(row)

    return pd.DataFrame(rows)


# ---------------------------------------------------------------------
# 1. 2026 high-level diagnostics
# ---------------------------------------------------------------------

trades_2026 = trades[trades["year"].eq(2026)].copy()
core_2026 = trades_2026[trades_2026["layer"].eq("Core")].copy()

summary_full_vs_2026 = pd.DataFrame([
    summarize_trade_df(trades, "Full sample"),
    summarize_trade_df(trades_2026, "2026 all trades"),
    summarize_trade_df(core_2026, "2026 Core only"),
])

print("\nFull sample vs 2026 summary:")
display(summary_full_vs_2026)

print("\n2026 by month and layer:")
summary_2026_by_month_layer = summarize_by(
    trades_2026,
    ["month", "layer"],
).sort_values(["month", "layer"])

display(summary_2026_by_month_layer)

print("\n2026 by tenor and layer:")
summary_2026_by_tenor_layer = summarize_by(
    trades_2026,
    ["selected_tenor", "layer"],
).sort_values(["selected_tenor", "layer"])

display(summary_2026_by_tenor_layer)

print("\n2026 by tenor group and layer:")
summary_2026_by_group_layer = summarize_by(
    trades_2026,
    ["tenor_group", "layer"],
).sort_values(["tenor_group", "layer"])

display(summary_2026_by_group_layer)


# ---------------------------------------------------------------------
# 2. Worst 2026 trades and clusters
# ---------------------------------------------------------------------

print("\nWorst 2026 individual trades:")
worst_2026_trades = (
    trades_2026
    .sort_values("normalized_pnl_dollars", ascending=True)
    .head(25)
    .reset_index(drop=True)
)

display(worst_2026_trades)

print("\nWorst 2026 rolling 20-trade windows by ending trade:")
worst_2026_windows = (
    trades
    .dropna(subset=["rolling_20_trade_pnl"])
    .loc[lambda x: x["year"].eq(2026)]
    .sort_values("rolling_20_trade_pnl", ascending=True)
    .head(10)
    .reset_index(drop=True)
)

display(worst_2026_windows)

if not worst_2026_windows.empty:
    worst_window_end_seq = int(worst_2026_windows.iloc[0]["trade_sequence"])
    worst_window_start_seq = max(1, worst_window_end_seq - 19)

    worst_2026_window_trades = (
        trades
        .loc[
            trades["trade_sequence"].between(worst_window_start_seq, worst_window_end_seq)
        ]
        .copy()
        .sort_values("trade_sequence")
        .reset_index(drop=True)
    )

    print(
        f"\nWorst 2026 20-trade window constituents: "
        f"trade_sequence {worst_window_start_seq} to {worst_window_end_seq}"
    )
    display(worst_2026_window_trades)


# ---------------------------------------------------------------------
# 3. Feature diagnostics: 2026 Core losses vs wins
# ---------------------------------------------------------------------

feature_cols = [
    "selected_tenor",
    "actual_dte",
    "vrp_log",
    "vrp_z_3m",
    "vrp_z_1y",
    "rv21d",
    "rsi14",
    "conditional_win_probability",
    "conditional_avg_pnl_per_day",
    "pnl_per_day",
    "normalized_pnl_dollars",
]

core_hist = trades[
    trades["layer"].eq("Core") & trades["year"].lt(2026)
].copy()

core_2026 = trades[
    trades["layer"].eq("Core") & trades["year"].eq(2026)
].copy()

core_2026["outcome_bucket"] = np.where(core_2026["win"].eq(1), "2026 Core winners", "2026 Core losers")
core_hist["outcome_bucket"] = "2019-2025 Core"

feature_compare = pd.concat([core_hist, core_2026], ignore_index=True)

feature_diagnostic = (
    feature_compare
    .groupby("outcome_bucket", dropna=False)[feature_cols]
    .agg(["count", "mean", "median", "min", "max"])
)

print("\nFeature diagnostic: 2019-2025 Core vs 2026 Core winners/losers:")
display(feature_diagnostic)

print("\n2026 Core losses by tenor:")
core_2026_loss_by_tenor = summarize_by(
    core_2026[core_2026["win"].eq(0)],
    ["selected_tenor"],
).sort_values("selected_tenor")

display(core_2026_loss_by_tenor)


# ---------------------------------------------------------------------
# 4. Counterfactual tenor-priority rules
# ---------------------------------------------------------------------
#
# This checks whether the 2026 weakness is caused by the 25 bps blend.
#
# Rules:
#   - win_only
#   - blend_10bps
#   - blend_25bps
#   - blend_50bps
#   - pnl_day_only
#
# All use the same locked 2621 thresholds and Core-first logic.

def make_priority_order_for_rule(layer_name, qualifying_cols, rule_name):
    if len(qualifying_cols) == 0:
        return pd.DataFrame()

    df = tenor_priority_metrics[
        (tenor_priority_metrics["layer"].eq(layer_name)) &
        (tenor_priority_metrics["selected_col"].isin(qualifying_cols)) &
        (tenor_priority_metrics["candidate_count"] > 0)
    ].copy()

    if df.empty:
        return df

    if rule_name == "win_only":
        df = df.sort_values(
            [
                "conditional_win_probability",
                "conditional_avg_pnl_per_day",
                "conditional_aggregate_pnl_per_day",
                "selected_tenor",
            ],
            ascending=[False, False, False, False],
        )

    elif rule_name == "pnl_day_only":
        df = df.sort_values(
            [
                "conditional_avg_pnl_per_day",
                "conditional_aggregate_pnl_per_day",
                "conditional_win_probability",
                "selected_tenor",
            ],
            ascending=[False, False, False, False],
        )

    elif rule_name.startswith("blend_"):
        band_bps = int(rule_name.replace("blend_", "").replace("bps", ""))
        band = band_bps / 10000.0

        best_win = df["conditional_win_probability"].max()
        df["best_conditional_win_probability"] = best_win
        df["win_probability_gap_to_best"] = best_win - df["conditional_win_probability"]
        df["inside_win_band"] = df["win_probability_gap_to_best"] <= band

        df = df[df["inside_win_band"]].copy()

        df = df.sort_values(
            [
                "conditional_avg_pnl_per_day",
                "conditional_aggregate_pnl_per_day",
                "conditional_win_probability",
                "selected_tenor",
            ],
            ascending=[False, False, False, False],
        )

    else:
        raise ValueError(f"Unknown rule_name: {rule_name}")

    df = df.reset_index(drop=True)
    df["priority_rank"] = np.arange(1, len(df) + 1)

    return df


def select_path_for_priority_rule(rule_name):
    rows = []

    for row in range(num_sweep_dates):
        active_layer = None
        qualifying_cols = None

        core_cols = np.where(core_qualifies_matrix[row, :])[0].astype(int).tolist()

        if core_cols:
            active_layer = "Core"
            qualifying_cols = core_cols
        else:
            secondary_cols = np.where(secondary_qualifies_matrix[row, :])[0].astype(int).tolist()

            if secondary_cols:
                active_layer = "Secondary"
                qualifying_cols = secondary_cols

        if not qualifying_cols:
            continue

        ordered = make_priority_order_for_rule(
            layer_name=active_layer,
            qualifying_cols=qualifying_cols,
            rule_name=rule_name,
        )

        if ordered.empty:
            continue

        selected_col = int(ordered.iloc[0]["selected_col"])
        row_idx = int(row)

        rows.append({
            "priority_rule": rule_name,
            "row_idx": row_idx,
            "date": pd.to_datetime(sweep_dates[row_idx]),
            "layer": active_layer,
            "selected_col": selected_col,
            "selected_tenor": int(TENOR_ARRAY[selected_col]),
            "tenor_group": tenor_group_for_tenor(TENOR_ARRAY[selected_col]) if "tenor_group_for_tenor" in globals() else np.nan,
            "num_qualifying_tenors_in_layer": int(len(qualifying_cols)),
            "qualifying_tenors_in_layer": ",".join([str(int(TENOR_ARRAY[c])) for c in qualifying_cols]),

            "win": win_mat[row_idx, selected_col],
            "normalized_pnl_dollars": pnl_mat[row_idx, selected_col],
            "actual_dte": actual_dte_mat[row_idx, selected_col],
            "pnl_per_day": pnl_per_day_mat[row_idx, selected_col],
            "vrp_log": vrp_mat[row_idx, selected_col],
            "vrp_z_3m": z3m_mat[row_idx, selected_col],
            "vrp_z_1y": z1y_mat[row_idx, selected_col],
            "rv21d": rv21d_by_date[row_idx],
            "rsi14": rsi_by_date[row_idx],
        })

    out = pd.DataFrame(rows).sort_values("date").reset_index(drop=True)

    if out.empty:
        return out

    out = out.merge(
        tenor_priority_metrics,
        on=["layer", "selected_col", "selected_tenor", "tenor_group"],
        how="left",
    )

    out["trade_sequence"] = np.arange(1, len(out) + 1)
    out["cum_pnl"] = out["normalized_pnl_dollars"].cumsum()
    out["running_max_cum_pnl"] = out["cum_pnl"].cummax()
    out["drawdown"] = out["cum_pnl"] - out["running_max_cum_pnl"]
    out["rolling_20_trade_pnl"] = out["normalized_pnl_dollars"].rolling(20).sum()
    out["year"] = out["date"].dt.year
    out["month"] = out["date"].dt.to_period("M").astype(str)

    return out


priority_rules_to_compare = [
    "win_only",
    "blend_10bps",
    "blend_25bps",
    "blend_50bps",
    "pnl_day_only",
]

counterfactual_paths = {
    rule: select_path_for_priority_rule(rule)
    for rule in priority_rules_to_compare
}

counterfactual_summary_rows = []

for rule, path in counterfactual_paths.items():
    counterfactual_summary_rows.append(
        summarize_trade_df(path, f"{rule} full sample").to_dict() | {
            "priority_rule": rule,
            "sample": "full_sample",
        }
    )

    counterfactual_summary_rows.append(
        summarize_trade_df(path[path["year"].eq(2026)], f"{rule} 2026").to_dict() | {
            "priority_rule": rule,
            "sample": "2026_all",
        }
    )

    counterfactual_summary_rows.append(
        summarize_trade_df(
            path[path["year"].eq(2026) & path["layer"].eq("Core")],
            f"{rule} 2026 Core",
        ).to_dict() | {
            "priority_rule": rule,
            "sample": "2026_core",
        }
    )

counterfactual_summary = pd.DataFrame(counterfactual_summary_rows)

counterfactual_summary = counterfactual_summary[
    [
        "priority_rule",
        "sample",
        "trade_count",
        "trade_frequency",
        "win_rate",
        "avg_pnl_per_day",
        "aggregate_pnl_per_day",
        "total_pnl",
        "max_drawdown",
        "worst_20_trade_sum",
        "avg_selected_tenor",
    ]
].sort_values(["sample", "win_rate", "avg_pnl_per_day"], ascending=[True, False, False])

print("\nCounterfactual priority-rule comparison:")
display(counterfactual_summary)


# ---------------------------------------------------------------------
# 5. 2026 tenor breakdown by counterfactual rule
# ---------------------------------------------------------------------

counterfactual_2026_tenor_rows = []

for rule, path in counterfactual_paths.items():
    p2026 = path[path["year"].eq(2026)].copy()

    counts = (
        p2026
        .groupby(["selected_tenor", "layer"], dropna=False)
        .size()
        .reset_index(name="trade_count")
    )

    counts["priority_rule"] = rule
    counterfactual_2026_tenor_rows.append(counts)

counterfactual_2026_tenor_counts = pd.concat(
    counterfactual_2026_tenor_rows,
    ignore_index=True,
)

print("\n2026 tenor counts by priority rule:")
display(
    counterfactual_2026_tenor_counts
    .pivot_table(
        index=["priority_rule", "layer"],
        columns="selected_tenor",
        values="trade_count",
        aggfunc="sum",
        fill_value=0,
    )
    .reset_index()
)


# ---------------------------------------------------------------------
# 6. On bad 2026 Core dates, compare selected tenor vs all Core-qualified alternatives
# ---------------------------------------------------------------------

bad_2026_core_dates = (
    core_2026
    .sort_values("normalized_pnl_dollars", ascending=True)
    .head(12)["date"]
    .drop_duplicates()
    .tolist()
)

date_to_row = {
    pd.to_datetime(d): i
    for i, d in enumerate(pd.to_datetime(sweep_dates))
}

selected_lookup = {
    pd.to_datetime(row["date"]): int(row["selected_col"])
    for _, row in core_2026.iterrows()
}

alternative_rows = []

for dt in bad_2026_core_dates:
    row = date_to_row[pd.to_datetime(dt)]
    selected_col = selected_lookup[pd.to_datetime(dt)]

    qualifying_cols = np.where(core_qualifies_matrix[row, :])[0].astype(int).tolist()

    for col in qualifying_cols:
        tenor = int(TENOR_ARRAY[col])

        metric_row = tenor_priority_metrics[
            (tenor_priority_metrics["layer"].eq("Core")) &
            (tenor_priority_metrics["selected_col"].eq(col))
        ]

        if metric_row.empty:
            cond_win = np.nan
            cond_pnl_day = np.nan
            cond_agg_pnl_day = np.nan
            candidate_count = np.nan
        else:
            metric_row = metric_row.iloc[0]
            cond_win = metric_row["conditional_win_probability"]
            cond_pnl_day = metric_row["conditional_avg_pnl_per_day"]
            cond_agg_pnl_day = metric_row["conditional_aggregate_pnl_per_day"]
            candidate_count = metric_row["candidate_count"]

        alternative_rows.append({
            "date": pd.to_datetime(dt),
            "selected_by_final_model": bool(col == selected_col),
            "tenor": tenor,
            "tenor_group": tenor_group_for_tenor(tenor) if "tenor_group_for_tenor" in globals() else np.nan,
            "win": win_mat[row, col],
            "normalized_pnl_dollars": pnl_mat[row, col],
            "actual_dte": actual_dte_mat[row, col],
            "pnl_per_day": pnl_per_day_mat[row, col],
            "vrp_log": vrp_mat[row, col],
            "vrp_z_3m": z3m_mat[row, col],
            "vrp_z_1y": z1y_mat[row, col],
            "rv21d": rv21d_by_date[row],
            "rsi14": rsi_by_date[row],
            "conditional_win_probability": cond_win,
            "conditional_avg_pnl_per_day": cond_pnl_day,
            "conditional_aggregate_pnl_per_day": cond_agg_pnl_day,
            "candidate_count": candidate_count,
        })

bad_2026_core_alternatives = (
    pd.DataFrame(alternative_rows)
    .sort_values(
        ["date", "selected_by_final_model", "conditional_win_probability", "conditional_avg_pnl_per_day"],
        ascending=[True, False, False, False],
    )
    .reset_index(drop=True)
)

print("\nBad 2026 Core dates: selected tenor vs all Core-qualified alternatives:")
display(bad_2026_core_alternatives)


# ---------------------------------------------------------------------
# 7. Was final selection worse than best available alternative on bad 2026 Core dates?
# ---------------------------------------------------------------------

alternative_date_summary_rows = []

if "bad_2026_core_alternatives" in globals() and not bad_2026_core_alternatives.empty:
    tmp_alt = bad_2026_core_alternatives.copy()
    tmp_alt["date"] = pd.to_datetime(tmp_alt["date"])

    for dt, sub in tmp_alt.groupby("date", dropna=False):
        selected_rows = sub[sub["selected_by_final_model"].eq(True)].copy()

        if selected_rows.empty:
            # Should not happen, but keep the diagnostic robust.
            continue

        selected_row = selected_rows.iloc[0]

        best_pnl_row = sub.sort_values(
            "normalized_pnl_dollars",
            ascending=False,
        ).iloc[0]

        best_win_row = sub.sort_values(
            ["win", "normalized_pnl_dollars"],
            ascending=[False, False],
        ).iloc[0]

        alternative_date_summary_rows.append({
            "date": dt,

            "selected_tenor": int(selected_row["tenor"]),
            "selected_group": selected_row["tenor_group"],
            "selected_win": float(selected_row["win"]),
            "selected_pnl": float(selected_row["normalized_pnl_dollars"]),
            "selected_pnl_per_day": float(selected_row["pnl_per_day"]),
            "selected_conditional_win_probability": float(selected_row["conditional_win_probability"]),
            "selected_conditional_avg_pnl_per_day": float(selected_row["conditional_avg_pnl_per_day"]),

            "best_available_pnl_tenor": int(best_pnl_row["tenor"]),
            "best_available_pnl_group": best_pnl_row["tenor_group"],
            "best_available_pnl_win": float(best_pnl_row["win"]),
            "best_available_pnl": float(best_pnl_row["normalized_pnl_dollars"]),
            "best_available_pnl_per_day": float(best_pnl_row["pnl_per_day"]),
            "best_available_pnl_conditional_win_probability": float(best_pnl_row["conditional_win_probability"]),
            "best_available_pnl_conditional_avg_pnl_per_day": float(best_pnl_row["conditional_avg_pnl_per_day"]),

            "best_available_win_tenor": int(best_win_row["tenor"]),
            "best_available_win": float(best_win_row["win"]),
            "best_available_win_pnl": float(best_win_row["normalized_pnl_dollars"]),

            "worst_available_pnl": float(sub["normalized_pnl_dollars"].min()),
            "num_qualified": int(len(sub)),
        })

    alternative_date_summary = pd.DataFrame(alternative_date_summary_rows)

    alternative_date_summary["selected_minus_best_available_pnl"] = (
        alternative_date_summary["selected_pnl"] -
        alternative_date_summary["best_available_pnl"]
    )

    alternative_date_summary["selected_was_best_available_pnl"] = (
        alternative_date_summary["selected_tenor"].eq(
            alternative_date_summary["best_available_pnl_tenor"]
        )
    )

    print("\nBad 2026 Core dates: selected result vs best available qualifying tenor:")
    display(
        alternative_date_summary.sort_values(
            "selected_minus_best_available_pnl",
            ascending=True,
        )
    )

    print("\nSummary of selected vs best available on bad 2026 Core dates:")
    display(
        pd.DataFrame([{
            "bad_dates": len(alternative_date_summary),
            "selected_was_best_available_pnl_count": int(
                alternative_date_summary["selected_was_best_available_pnl"].sum()
            ),
            "selected_was_best_available_pnl_rate": float(
                alternative_date_summary["selected_was_best_available_pnl"].mean()
            ),
            "avg_selected_minus_best_available_pnl": float(
                alternative_date_summary["selected_minus_best_available_pnl"].mean()
            ),
            "total_selected_minus_best_available_pnl": float(
                alternative_date_summary["selected_minus_best_available_pnl"].sum()
            ),
        }])
    )
else:
    alternative_date_summary = pd.DataFrame()
    print("No bad_2026_core_alternatives available.")

# ---------------------------------------------------------------------
# 8. Save diagnostics
# ---------------------------------------------------------------------

DIAG_PREFIX = "locked_2621_win_band_25bps_conditional_2026_diagnostics"

paths = {
    "summary_full_vs_2026": AUDIT_DIR / f"{DIAG_PREFIX}_summary_full_vs_2026.csv",
    "summary_2026_by_month_layer": AUDIT_DIR / f"{DIAG_PREFIX}_summary_2026_by_month_layer.csv",
    "summary_2026_by_tenor_layer": AUDIT_DIR / f"{DIAG_PREFIX}_summary_2026_by_tenor_layer.csv",
    "summary_2026_by_group_layer": AUDIT_DIR / f"{DIAG_PREFIX}_summary_2026_by_group_layer.csv",
    "worst_2026_trades": AUDIT_DIR / f"{DIAG_PREFIX}_worst_2026_trades.csv",
    "worst_2026_windows": AUDIT_DIR / f"{DIAG_PREFIX}_worst_2026_windows.csv",
    "feature_diagnostic": AUDIT_DIR / f"{DIAG_PREFIX}_feature_diagnostic.csv",
    "counterfactual_summary": AUDIT_DIR / f"{DIAG_PREFIX}_counterfactual_summary.csv",
    "counterfactual_2026_tenor_counts": AUDIT_DIR / f"{DIAG_PREFIX}_counterfactual_2026_tenor_counts.csv",
    "bad_2026_core_alternatives": AUDIT_DIR / f"{DIAG_PREFIX}_bad_2026_core_alternatives.csv",
}

summary_full_vs_2026.to_csv(paths["summary_full_vs_2026"], index=False)
summary_2026_by_month_layer.to_csv(paths["summary_2026_by_month_layer"], index=False)
summary_2026_by_tenor_layer.to_csv(paths["summary_2026_by_tenor_layer"], index=False)
summary_2026_by_group_layer.to_csv(paths["summary_2026_by_group_layer"], index=False)
worst_2026_trades.to_csv(paths["worst_2026_trades"], index=False)
worst_2026_windows.to_csv(paths["worst_2026_windows"], index=False)
feature_diagnostic.to_csv(paths["feature_diagnostic"])
counterfactual_summary.to_csv(paths["counterfactual_summary"], index=False)
counterfactual_2026_tenor_counts.to_csv(paths["counterfactual_2026_tenor_counts"], index=False)

if "bad_2026_core_alternatives" in globals() and not bad_2026_core_alternatives.empty:
    bad_2026_core_alternatives.to_csv(paths["bad_2026_core_alternatives"], index=False)

print("\nSaved 2026 diagnostics:")
for name, path in paths.items():
    print(name, "->", path)

print("\n2026 diagnostics complete.")

Selected trades loaded: 577
Date range: 2019-07-17 00:00:00 to 2026-03-30 00:00:00

Full sample vs 2026 summary:


,label,trade_count,trade_frequency,win_rate,avg_pnl_per_day,median_pnl_per_day,aggregate_pnl_per_day,total_pnl,total_actual_dte,worst_trade_pnl,worst_pnl_per_day,max_drawdown,worst_20_trade_sum,avg_selected_tenor
0,Full sample,577,0.334493,0.831889,400.073740,531.197014,384.792712,"4,892,639.330039","12,715.000000","-87,932.829005","-5,535.611173","-287,259.678373","-247,740.016788",22.279029
1,2026 all trades,45,0.026087,0.644444,118.908090,316.520529,116.617371,"136,675.558697","1,172.000000","-57,319.352129","-2,292.774085","-287,259.678373","-247,740.016788",26.000000
2,2026 Core only,44,0.025507,0.636364,111.759007,282.839974,108.293806,"123,671.526002","1,142.000000","-57,319.352129","-2,292.774085","-287,259.678373","-247,740.016788",25.840909



2026 by month and layer:


,month,layer,trade_count,trade_frequency,win_rate,avg_pnl_per_day,median_pnl_per_day,aggregate_pnl_per_day,total_pnl,total_actual_dte,worst_trade_pnl,worst_pnl_per_day,max_drawdown,worst_20_trade_sum,avg_selected_tenor
0,2026-01,Core,10,0.005797,0.700000,146.965860,207.710121,118.579166,"33,320.745527",281.000000,"-16,757.042966",-465.473416,"-16,757.042966",NaN,27.900000
1,2026-01,Secondary,1,0.000580,1.000000,433.467757,433.467757,433.467757,"13,004.032695",30.000000,"13,004.032695",433.467757,0.000000,NaN,33.000000
2,2026-02,Core,13,0.007536,0.384615,-324.613966,-45.983338,-445.244482,"-126,004.188387",283.000000,"-57,036.901356","-2,037.032191","-163,790.625172",NaN,22.384615
3,2026-03,Core,21,0.012174,0.761905,365.129488,747.582795,374.316555,"216,354.968862",578.000000,"-57,319.352129","-2,292.774085","-66,149.701072","191,022.829361",27.000000



2026 by tenor and layer:


...


Bad 2026 Core dates: selected result vs best available qualifying tenor:


,date,selected_tenor,selected_group,selected_win,selected_pnl,selected_pnl_per_day,selected_conditional_win_probability,selected_conditional_avg_pnl_per_day,best_available_pnl_tenor,best_available_pnl_group,best_available_pnl_win,best_available_pnl,best_available_pnl_per_day,best_available_pnl_conditional_win_probability,best_available_pnl_conditional_avg_pnl_per_day,best_available_win_tenor,best_available_win,best_available_win_pnl,worst_available_pnl,num_qualified,selected_minus_best_available_pnl,selected_was_best_available_pnl
2,2026-02-19,33,back,0.000000,"-52,092.062100","-1,447.001725",0.895522,431.999112,9,front,1.000000,"10,172.124590","1,271.515574",0.749164,459.029727,9,1.000000,"10,172.124590","-52,092.062100",8,"-62,264.186689",False
6,2026-02-27,27,back,0.000000,"-57,036.901356","-2,037.032191",0.894231,490.191037,9,front,0.000000,"-11,016.328239","-1,573.761177",0.749164,459.029727,9,0.000000,"-11,016.328239","-57,036.901356",9,"-46,020.573117",False
10,2026-03-05,27,back,0.000000,"-17,217.829479",-614.922481,0.894231,490.191037,33,back,1.000000,"19,161.990481",532.277513,0.895522,431.999112,33,1.000000,"19,161.990481","-50,236.359031",9,"-36,379.819960",False
9,2026-03-04,27,back,0.000000,"-23,176.359269",-799.184802,0.894231,490.191037,33,back,1.000000,"12,998.034791",351.298238,0.895522,431.999112,33,1.000000,"12,998.034791","-56,809.083631",9,"-36,174.394061",False
11,2026-03-09,27,back,0.000000,"-12,685.421844",-528.559244,0.894231,490.191037,33,back,1.000000,"21,336.111442",666.753483,0.895522,431.999112,33,1.000000,"21,336.111442","-45,195.769858",9,"-34,021.533287",False
7,2026-03-02,27,back,0.000000,"-57,319.352129","-2,292.774085",0.894231,490.191037,9,front,0.000000,"-24,210.868952","-2,200.988087",0.749164,459.029727,9,0.000000,"-24,210.868952","-57,319.352129",9,"-33,108.483177",False
4,2026-02-23,27,back,0.000000,"-30,436.912727","-1,217.476509",0.894231,490.191037,9,front,0.000000,"-1,152.425871",-104.765988,0.749164,459.029727,9,0.000000,"-1,152.425871","-48,751.416767",9,"-29,284.486856",False
0,2026-01-29,33,back,0.000000,"-16,757.042966",-465.473416,0.895522,431.999112,24,middle,1.000000,"3,861.380598",175.517300,0.851240,426.816292,24,1.000000,"3,861.380598","-16,757.042966",6,"-20,618.423564",False
1,2026-02-04,27,back,0.000000,"-4,108.840691",-136.961356,0.894231,490.191037,21,middle,1.000000,"14,104.888765",613.256033,0.834783,451.823181,24,1.000000,"14,104.888765","-4,108.840691",9,"-18,213.729456",False
3,2026-02-20,15,front,0.000000,"-12,848.957451",-917.782675,0.771523,409.251409,9,front,1.000000,"4,006.072790",572.296113,0.749164,459.029727,9,1.000000,"4,006.072790","-12,848.957451",3,"-16,855.030241",False



Summary of selected vs best available on bad 2026 Core dates:


,bad_dates,selected_was_best_available_pnl_count,selected_was_best_available_pnl_rate,avg_selected_minus_best_available_pnl,total_selected_minus_best_available_pnl
0,12,2,0.166667,"-27,745.055034","-332,940.660406"



Saved 2026 diagnostics:
summary_full_vs_2026 -> C:\Users\patri\vrp_project\data\audit\locked_2621_win_band_25bps_conditional_2026_diagnostics_summary_full_vs_2026.csv
summary_2026_by_month_layer -> C:\Users\patri\vrp_project\data\audit\locked_2621_win_band_25bps_conditional_2026_diagnostics_summary_2026_by_month_layer.csv
summary_2026_by_tenor_layer -> C:\Users\patri\vrp_project\data\audit\locked_2621_win_band_25bps_conditional_2026_diagnostics_summary_2026_by_tenor_layer.csv
summary_2026_by_group_layer -> C:\Users\patri\vrp_project\data\audit\locked_2621_win_band_25bps_conditional_2026_diagnostics_summary_2026_by_group_layer.csv
worst_2026_trades -> C:\Users\patri\vrp_project\data\audit\locked_2621_win_band_25bps_conditional_2026_diagnostics_worst_2026_trades.csv
worst_2026_windows -> C:\Users\patri\vrp_project\data\audit\locked_2621_win_band_25bps_conditional_2026_diagnostics_worst_2026_windows.csv
feature_diagnostic -> C:\Users\patri\vrp_project\data\audit\locked_2621_win_band_25bp

In [15]:
# ---------------------------------------------------------------------
# 8. Save diagnostics
# ---------------------------------------------------------------------

DIAG_PREFIX = "locked_2621_win_band_25bps_conditional_2026_diagnostics"

paths = {
    "summary_full_vs_2026": AUDIT_DIR / f"{DIAG_PREFIX}_summary_full_vs_2026.csv",
    "summary_2026_by_month_layer": AUDIT_DIR / f"{DIAG_PREFIX}_summary_2026_by_month_layer.csv",
    "summary_2026_by_tenor_layer": AUDIT_DIR / f"{DIAG_PREFIX}_summary_2026_by_tenor_layer.csv",
    "summary_2026_by_group_layer": AUDIT_DIR / f"{DIAG_PREFIX}_summary_2026_by_group_layer.csv",
    "worst_2026_trades": AUDIT_DIR / f"{DIAG_PREFIX}_worst_2026_trades.csv",
    "worst_2026_windows": AUDIT_DIR / f"{DIAG_PREFIX}_worst_2026_windows.csv",
    "feature_diagnostic": AUDIT_DIR / f"{DIAG_PREFIX}_feature_diagnostic.csv",
    "counterfactual_summary": AUDIT_DIR / f"{DIAG_PREFIX}_counterfactual_summary.csv",
    "counterfactual_2026_tenor_counts": AUDIT_DIR / f"{DIAG_PREFIX}_counterfactual_2026_tenor_counts.csv",
    "bad_2026_core_alternatives": AUDIT_DIR / f"{DIAG_PREFIX}_bad_2026_core_alternatives.csv",
    "alternative_date_summary": AUDIT_DIR / f"{DIAG_PREFIX}_alternative_date_summary.csv",
}

summary_full_vs_2026.to_csv(paths["summary_full_vs_2026"], index=False)
summary_2026_by_month_layer.to_csv(paths["summary_2026_by_month_layer"], index=False)
summary_2026_by_tenor_layer.to_csv(paths["summary_2026_by_tenor_layer"], index=False)
summary_2026_by_group_layer.to_csv(paths["summary_2026_by_group_layer"], index=False)
worst_2026_trades.to_csv(paths["worst_2026_trades"], index=False)
worst_2026_windows.to_csv(paths["worst_2026_windows"], index=False)
feature_diagnostic.to_csv(paths["feature_diagnostic"])
counterfactual_summary.to_csv(paths["counterfactual_summary"], index=False)
counterfactual_2026_tenor_counts.to_csv(paths["counterfactual_2026_tenor_counts"], index=False)
bad_2026_core_alternatives.to_csv(paths["bad_2026_core_alternatives"], index=False)

if not alternative_date_summary.empty:
    alternative_date_summary.to_csv(paths["alternative_date_summary"], index=False)

print("\nSaved 2026 diagnostics:")
for name, path in paths.items():
    print(name, "->", path)

print("\n2026 diagnostics complete.")


Saved 2026 diagnostics:
summary_full_vs_2026 -> C:\Users\patri\vrp_project\data\audit\locked_2621_win_band_25bps_conditional_2026_diagnostics_summary_full_vs_2026.csv
summary_2026_by_month_layer -> C:\Users\patri\vrp_project\data\audit\locked_2621_win_band_25bps_conditional_2026_diagnostics_summary_2026_by_month_layer.csv
summary_2026_by_tenor_layer -> C:\Users\patri\vrp_project\data\audit\locked_2621_win_band_25bps_conditional_2026_diagnostics_summary_2026_by_tenor_layer.csv
summary_2026_by_group_layer -> C:\Users\patri\vrp_project\data\audit\locked_2621_win_band_25bps_conditional_2026_diagnostics_summary_2026_by_group_layer.csv
worst_2026_trades -> C:\Users\patri\vrp_project\data\audit\locked_2621_win_band_25bps_conditional_2026_diagnostics_worst_2026_trades.csv
worst_2026_windows -> C:\Users\patri\vrp_project\data\audit\locked_2621_win_band_25bps_conditional_2026_diagnostics_worst_2026_windows.csv
feature_diagnostic -> C:\Users\patri\vrp_project\data\audit\locked_2621_win_band_25bp

In [16]:
# Cell — Line-fit parameter surfaces across tenors for Core and Secondary
#
# Purpose:
#   Re-test fitting a line through the locked 2621 parameters across tenors.
#
# What changes:
#   The threshold surface across 9, 12, 15, 18, 21, 24, 27, 30, 33 DTE.
#
# What stays fixed:
#   - Core first.
#   - Secondary only if no Core tenor qualifies.
#   - Tenor priority uses layer-specific conditional win probability.
#   - Tenors within 25 bps of the best conditional win probability are treated as near-ties.
#   - Conditional avg P&L/day is the tie-break inside that 25 bps band.
#
# Candidate threshold variants:
#   1. bucket
#      Original locked 2621 bucket thresholds.
#
#   2. ols_exact_unclipped
#      Least-squares line fit through the bucketed 2621 values.
#      No rounding, no clipping. Diagnostic only.
#
#   3. ols_rounded_clipped
#      Least-squares line fit, rounded to practical increments, clipped to original min/max,
#      and monotonicity-enforced.
#
#   4. endpoint_exact_unclipped
#      Straight line connecting 9D bucket value to 33D bucket value.
#      No rounding, no clipping. Diagnostic only.
#
#   5. endpoint_rounded_clipped
#      Endpoint line, rounded to practical increments, clipped to original min/max,
#      and monotonicity-enforced.
#
# The cell evaluates all Core variant x Secondary variant combinations.

import numpy as np
import pandas as pd
import json
from pathlib import Path

# ---------------------------------------------------------------------
# Robust setup
# ---------------------------------------------------------------------

if "TENOR_ORDER" in globals():
    TENOR_ARRAY = np.array(TENOR_ORDER, dtype=int)

elif "sweep_panel" in globals() and "tenor" in sweep_panel.columns:
    TENOR_ARRAY = np.array(sorted(sweep_panel["tenor"].dropna().unique()), dtype=int)
    TENOR_ORDER = TENOR_ARRAY.tolist()

elif "research_panel" in globals() and "tenor" in research_panel.columns:
    TENOR_ARRAY = np.array(sorted(research_panel["tenor"].dropna().unique()), dtype=int)
    TENOR_ORDER = TENOR_ARRAY.tolist()

else:
    TENOR_ARRAY = np.array([9, 12, 15, 18, 21, 24, 27, 30, 33], dtype=int)
    TENOR_ORDER = TENOR_ARRAY.tolist()

TENOR_TO_COL = {int(t): i for i, t in enumerate(TENOR_ARRAY)}

required_objects = [
    "num_sweep_dates",
    "sweep_dates",
    "vrp_mat",
    "z3m_mat",
    "z1y_mat",
    "win_mat",
    "pnl_mat",
    "actual_dte_mat",
    "rv21d_by_date",
    "rsi_by_date",
    "TRADE_FREQUENCY_DENOMINATOR",
]

missing_objects = [name for name in required_objects if name not in globals()]

if missing_objects:
    raise NameError(
        "Missing required objects. Run the data-loading / matrix-construction cells first. "
        f"Missing: {missing_objects}"
    )

if "pnl_per_day_mat" not in globals():
    with np.errstate(divide="ignore", invalid="ignore"):
        pnl_per_day_mat = pnl_mat / actual_dte_mat

    pnl_per_day_mat[~np.isfinite(pnl_per_day_mat)] = np.nan

if "AUDIT_DIR" not in globals():
    AUDIT_DIR = Path("data/audit")
else:
    AUDIT_DIR = Path(AUDIT_DIR)

if "PROCESSED_DIR" not in globals():
    PROCESSED_DIR = Path("data/processed")
else:
    PROCESSED_DIR = Path(PROCESSED_DIR)

AUDIT_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

WIN_BAND = 0.0025
EXPERIMENT_LABEL = "locked_2621_line_fit_parameter_surface"

print("Line-fit parameter surface experiment")
print("Tenors:", TENOR_ARRAY.tolist())
print("Win band:", WIN_BAND)


# ---------------------------------------------------------------------
# Locked 2621 bucket thresholds
# ---------------------------------------------------------------------

LOCKED_2621_CORE = {
    "front": {
        "tenors": [9, 12, 15],
        "vrp": 0.60,
        "z3m": 0.55,
        "z1y": 0.65,
        "rsi_cap": 70,
        "rv21d_floor": 8.5,
    },
    "middle": {
        "tenors": [18, 21, 24],
        "vrp": 0.65,
        "z3m": 0.75,
        "z1y": 0.65,
        "rsi_cap": 68,
        "rv21d_floor": 8.5,
    },
    "back": {
        "tenors": [27, 30, 33],
        "vrp": 0.70,
        "z3m": 0.75,
        "z1y": 0.75,
        "rsi_cap": 66,
        "rv21d_floor": 8.5,
    },
}

LOCKED_2621_SECONDARY = {
    "front": {
        "tenors": [9, 12, 15],
        "vrp": 0.60,
        "z3m": 0.50,
        "z1y": 0.40,
        "rsi_cap": 74,
        "rv21d_floor": 6.5,
    },
    "middle": {
        "tenors": [18, 21, 24],
        "vrp": 0.60,
        "z3m": 0.50,
        "z1y": 0.50,
        "rsi_cap": 70,
        "rv21d_floor": 6.5,
    },
    "back": {
        "tenors": [27, 30, 33],
        "vrp": 0.70,
        "z3m": 0.50,
        "z1y": 0.50,
        "rsi_cap": 68,
        "rv21d_floor": 6.5,
    },
}


# ---------------------------------------------------------------------
# Helper functions
# ---------------------------------------------------------------------

def tenor_group_for_tenor(tenor):
    tenor = int(tenor)

    if tenor in [9, 12, 15]:
        return "front"
    elif tenor in [18, 21, 24]:
        return "middle"
    elif tenor in [27, 30, 33]:
        return "back"
    else:
        raise ValueError(f"Unexpected tenor: {tenor}")


def locked_spec_to_threshold_table(threshold_spec, layer, threshold_variant):
    rows = []

    for group, params in threshold_spec.items():
        for tenor in params["tenors"]:
            rows.append({
                "threshold_variant": threshold_variant,
                "layer": layer,
                "selected_tenor": int(tenor),
                "tenor_group": group,
                "vrp_threshold": float(params["vrp"]),
                "z3m_threshold": float(params["z3m"]),
                "z1y_threshold": float(params["z1y"]),
                "rsi_cap": float(params["rsi_cap"]),
                "rv21d_floor": float(params["rv21d_floor"]),
            })

    out = pd.DataFrame(rows).sort_values("selected_tenor").reset_index(drop=True)

    return out


BASE_CORE_TABLE = locked_spec_to_threshold_table(
    LOCKED_2621_CORE,
    layer="Core",
    threshold_variant="bucket",
)

BASE_SECONDARY_TABLE = locked_spec_to_threshold_table(
    LOCKED_2621_SECONDARY,
    layer="Secondary",
    threshold_variant="bucket",
)

PARAM_COLS = [
    "vrp_threshold",
    "z3m_threshold",
    "z1y_threshold",
    "rsi_cap",
    "rv21d_floor",
]

ROUNDING_STEPS = {
    "vrp_threshold": 0.05,
    "z3m_threshold": 0.05,
    "z1y_threshold": 0.05,
    "rsi_cap": 1.0,
    "rv21d_floor": 0.5,
}

# Higher threshold = stricter for these.
NONDECREASING_PARAMS = [
    "vrp_threshold",
    "z3m_threshold",
    "z1y_threshold",
    "rv21d_floor",
]

# Lower cap = stricter for RSI.
NONINCREASING_PARAMS = [
    "rsi_cap",
]


def round_to_step(values, step):
    values = np.asarray(values, dtype=float)
    return np.round(values / step) * step


def enforce_monotonic(values, param_col):
    values = np.asarray(values, dtype=float)

    if param_col in NONDECREASING_PARAMS:
        return np.maximum.accumulate(values)

    if param_col in NONINCREASING_PARAMS:
        return np.minimum.accumulate(values)

    return values


def fit_line_values(x, y, method):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    if method == "ols":
        slope, intercept = np.polyfit(x, y, 1)
        fitted = slope * x + intercept

    elif method == "endpoint":
        slope = (y[-1] - y[0]) / (x[-1] - x[0])
        intercept = y[0] - slope * x[0]
        fitted = slope * x + intercept

    else:
        raise ValueError(f"Unknown line-fit method: {method}")

    return fitted, slope, intercept


def build_line_fit_threshold_table(
    base_table,
    threshold_variant,
    method,
    do_round,
    do_clip,
    do_monotonic,
):
    """
    Fits one line across all nine tenors for each threshold parameter.
    """
    base = base_table.copy().sort_values("selected_tenor").reset_index(drop=True)

    x = base["selected_tenor"].to_numpy(dtype=float)

    rows = base[
        [
            "layer",
            "selected_tenor",
            "tenor_group",
        ]
    ].copy()

    rows.insert(0, "threshold_variant", threshold_variant)

    fit_details = []

    for param_col in PARAM_COLS:
        y = base[param_col].to_numpy(dtype=float)

        fitted, slope, intercept = fit_line_values(
            x=x,
            y=y,
            method=method,
        )

        values = fitted.copy()

        raw_min = float(np.nanmin(y))
        raw_max = float(np.nanmax(y))

        if do_clip:
            values = np.clip(values, raw_min, raw_max)

        if do_monotonic:
            values = enforce_monotonic(values, param_col)

        if do_round:
            values = round_to_step(values, ROUNDING_STEPS[param_col])

        if do_clip:
            values = np.clip(values, raw_min, raw_max)

        if do_monotonic:
            values = enforce_monotonic(values, param_col)

        # Avoid ugly floating point artifacts.
        if param_col in ["vrp_threshold", "z3m_threshold", "z1y_threshold", "rv21d_floor"]:
            values = np.round(values, 4)

        if param_col == "rsi_cap":
            values = np.round(values, 0)

        rows[param_col] = values

        fit_details.append({
            "threshold_variant": threshold_variant,
            "layer": str(base["layer"].iloc[0]),
            "parameter": param_col,
            "method": method,
            "do_round": bool(do_round),
            "do_clip": bool(do_clip),
            "do_monotonic": bool(do_monotonic),
            "slope": float(slope),
            "intercept": float(intercept),
            "base_min": raw_min,
            "base_max": raw_max,
            "fitted_9d": float(fitted[0]),
            "fitted_33d": float(fitted[-1]),
            "final_9d": float(values[0]),
            "final_33d": float(values[-1]),
        })

    return rows, pd.DataFrame(fit_details)


def build_all_threshold_variants(base_table):
    """
    Builds bucket baseline plus line-fit variants for one layer.
    """
    layer = str(base_table["layer"].iloc[0])

    tables = {}
    details = []

    bucket = base_table.copy()
    bucket["threshold_variant"] = "bucket"
    tables["bucket"] = bucket

    variant_configs = [
        {
            "threshold_variant": "ols_exact_unclipped",
            "method": "ols",
            "do_round": False,
            "do_clip": False,
            "do_monotonic": False,
        },
        {
            "threshold_variant": "ols_rounded_clipped",
            "method": "ols",
            "do_round": True,
            "do_clip": True,
            "do_monotonic": True,
        },
        {
            "threshold_variant": "endpoint_exact_unclipped",
            "method": "endpoint",
            "do_round": False,
            "do_clip": False,
            "do_monotonic": False,
        },
        {
            "threshold_variant": "endpoint_rounded_clipped",
            "method": "endpoint",
            "do_round": True,
            "do_clip": True,
            "do_monotonic": True,
        },
    ]

    for cfg in variant_configs:
        table, detail = build_line_fit_threshold_table(
            base_table=base_table,
            **cfg,
        )

        table["layer"] = layer
        detail["layer"] = layer

        tables[cfg["threshold_variant"]] = table
        details.append(detail)

    if details:
        details_df = pd.concat(details, ignore_index=True)
    else:
        details_df = pd.DataFrame()

    return tables, details_df


CORE_THRESHOLD_VARIANTS, CORE_FIT_DETAILS = build_all_threshold_variants(BASE_CORE_TABLE)
SECONDARY_THRESHOLD_VARIANTS, SECONDARY_FIT_DETAILS = build_all_threshold_variants(BASE_SECONDARY_TABLE)

line_fit_details = pd.concat(
    [CORE_FIT_DETAILS, SECONDARY_FIT_DETAILS],
    ignore_index=True,
)

all_threshold_variant_tables = pd.concat(
    list(CORE_THRESHOLD_VARIANTS.values()) + list(SECONDARY_THRESHOLD_VARIANTS.values()),
    ignore_index=True,
)

print("\nCore threshold variants:")
display(
    pd.concat(CORE_THRESHOLD_VARIANTS.values(), ignore_index=True)
    .sort_values(["threshold_variant", "selected_tenor"])
)

print("\nSecondary threshold variants:")
display(
    pd.concat(SECONDARY_THRESHOLD_VARIANTS.values(), ignore_index=True)
    .sort_values(["threshold_variant", "selected_tenor"])
)

print("\nLine-fit details:")
display(line_fit_details)


# ---------------------------------------------------------------------
# Qualification, tenor-priority, and selected-path functions
# ---------------------------------------------------------------------

def build_qualifies_matrix_from_threshold_table(threshold_table):
    """
    Builds date x tenor qualification matrix from a per-tenor threshold table.
    """
    threshold_table = threshold_table.copy()

    required_cols = [
        "selected_tenor",
        "vrp_threshold",
        "z3m_threshold",
        "z1y_threshold",
        "rsi_cap",
        "rv21d_floor",
    ]

    missing_cols = [c for c in required_cols if c not in threshold_table.columns]
    if missing_cols:
        raise ValueError(f"Threshold table missing columns: {missing_cols}")

    qualifies = np.zeros((num_sweep_dates, len(TENOR_ARRAY)), dtype=bool)

    for col, tenor in enumerate(TENOR_ARRAY):
        row = threshold_table[threshold_table["selected_tenor"].eq(int(tenor))]

        if row.empty:
            raise ValueError(f"Missing threshold row for tenor {tenor}")

        if len(row) > 1:
            raise ValueError(f"Multiple threshold rows for tenor {tenor}")

        row = row.iloc[0]

        qualifies[:, col] = (
            (vrp_mat[:, col] >= float(row["vrp_threshold"])) &
            (z3m_mat[:, col] >= float(row["z3m_threshold"])) &
            (z1y_mat[:, col] >= float(row["z1y_threshold"])) &
            (rsi_by_date <= float(row["rsi_cap"])) &
            (rv21d_by_date >= float(row["rv21d_floor"]))
        )

    return qualifies


def build_layer_conditional_tenor_metrics(layer_name, qualifies_matrix, date_mask):
    """
    Tenor-ranking metrics conditional on a candidate threshold surface.
    """
    rows = []

    for col, tenor in enumerate(TENOR_ARRAY):
        candidate_mask = (
            date_mask &
            qualifies_matrix[:, col] &
            np.isfinite(win_mat[:, col]) &
            np.isfinite(pnl_mat[:, col]) &
            np.isfinite(actual_dte_mat[:, col]) &
            np.isfinite(pnl_per_day_mat[:, col])
        )

        candidate_count = int(candidate_mask.sum())

        if candidate_count == 0:
            rows.append({
                "layer": layer_name,
                "selected_col": int(col),
                "selected_tenor": int(tenor),
                "tenor_group": tenor_group_for_tenor(tenor),
                "conditional_win_probability": np.nan,
                "conditional_avg_pnl_per_day": np.nan,
                "conditional_median_pnl_per_day": np.nan,
                "conditional_aggregate_pnl_per_day": np.nan,
                "conditional_avg_raw_pnl": np.nan,
                "conditional_avg_actual_dte": np.nan,
                "conditional_worst_trade_pnl": np.nan,
                "conditional_worst_pnl_per_day": np.nan,
                "candidate_count": 0,
            })
            continue

        rows.append({
            "layer": layer_name,
            "selected_col": int(col),
            "selected_tenor": int(tenor),
            "tenor_group": tenor_group_for_tenor(tenor),
            "conditional_win_probability": float(np.nanmean(win_mat[candidate_mask, col])),
            "conditional_avg_pnl_per_day": float(np.nanmean(pnl_per_day_mat[candidate_mask, col])),
            "conditional_median_pnl_per_day": float(np.nanmedian(pnl_per_day_mat[candidate_mask, col])),
            "conditional_aggregate_pnl_per_day": float(
                np.nansum(pnl_mat[candidate_mask, col]) /
                np.nansum(actual_dte_mat[candidate_mask, col])
            ),
            "conditional_avg_raw_pnl": float(np.nanmean(pnl_mat[candidate_mask, col])),
            "conditional_avg_actual_dte": float(np.nanmean(actual_dte_mat[candidate_mask, col])),
            "conditional_worst_trade_pnl": float(np.nanmin(pnl_mat[candidate_mask, col])),
            "conditional_worst_pnl_per_day": float(np.nanmin(pnl_per_day_mat[candidate_mask, col])),
            "candidate_count": candidate_count,
        })

    return pd.DataFrame(rows)


def build_candidate_tenor_priority_metrics(core_qualifies_matrix, secondary_qualifies_matrix):
    """
    Core metrics:
      Conditional on each tenor qualifying for Core.

    Secondary metrics:
      Conditional on no Core tenor qualifying that date, and the tenor qualifying for Secondary.
    """
    any_core_qualifies_by_date = core_qualifies_matrix.any(axis=1)
    no_core_qualifies_by_date = ~any_core_qualifies_by_date

    core_metrics = build_layer_conditional_tenor_metrics(
        layer_name="Core",
        qualifies_matrix=core_qualifies_matrix,
        date_mask=np.ones(num_sweep_dates, dtype=bool),
    )

    secondary_metrics = build_layer_conditional_tenor_metrics(
        layer_name="Secondary",
        qualifies_matrix=secondary_qualifies_matrix,
        date_mask=no_core_qualifies_by_date,
    )

    return pd.concat([core_metrics, secondary_metrics], ignore_index=True)


def blended_priority_order_for_candidate(layer_name, qualifying_cols, tenor_priority_metrics, win_band=WIN_BAND):
    """
    Final blended priority rule:
      - Best conditional win probability first.
      - Keep tenors within win_band of best.
      - Choose highest conditional avg P&L/day.
    """
    if len(qualifying_cols) == 0:
        return pd.DataFrame()

    df = tenor_priority_metrics[
        (tenor_priority_metrics["layer"].eq(layer_name)) &
        (tenor_priority_metrics["selected_col"].isin(qualifying_cols)) &
        (tenor_priority_metrics["candidate_count"] > 0)
    ].copy()

    if df.empty:
        return df

    best_win = df["conditional_win_probability"].max()

    df["best_conditional_win_probability"] = best_win
    df["win_probability_gap_to_best"] = best_win - df["conditional_win_probability"]
    df["inside_win_band"] = df["win_probability_gap_to_best"] <= win_band

    df = df[df["inside_win_band"]].copy()

    df = df.sort_values(
        [
            "conditional_avg_pnl_per_day",
            "conditional_aggregate_pnl_per_day",
            "conditional_win_probability",
            "selected_tenor",
        ],
        ascending=[False, False, False, False],
    ).reset_index(drop=True)

    df["priority_rank_within_signal"] = np.arange(1, len(df) + 1)

    return df


def select_candidate_path(
    candidate_id,
    core_threshold_variant,
    secondary_threshold_variant,
    core_table,
    secondary_table,
    core_qualifies_matrix,
    secondary_qualifies_matrix,
    tenor_priority_metrics,
    win_band=WIN_BAND,
):
    """
    Core-first / Secondary-second selected path for one line-fit candidate.
    """
    selected_rows = []

    for row in range(num_sweep_dates):
        core_qualifying_cols = np.where(core_qualifies_matrix[row, :])[0].astype(int).tolist()

        if core_qualifying_cols:
            layer_name = "Core"
            qualifying_cols = core_qualifying_cols

            ordered = blended_priority_order_for_candidate(
                layer_name=layer_name,
                qualifying_cols=qualifying_cols,
                tenor_priority_metrics=tenor_priority_metrics,
                win_band=win_band,
            )

            if not ordered.empty:
                selected_col = int(ordered.iloc[0]["selected_col"])

                selected_rows.append({
                    "candidate_id": candidate_id,
                    "core_threshold_variant": core_threshold_variant,
                    "secondary_threshold_variant": secondary_threshold_variant,
                    "row_idx": int(row),
                    "date": sweep_dates[row],
                    "layer": layer_name,
                    "selected_col": selected_col,
                    "selected_tenor": int(TENOR_ARRAY[selected_col]),
                    "tenor_group": tenor_group_for_tenor(TENOR_ARRAY[selected_col]),
                    "num_qualifying_tenors_in_layer": int(len(qualifying_cols)),
                    "qualifying_tenors_in_layer": ",".join(
                        [str(int(TENOR_ARRAY[c])) for c in qualifying_cols]
                    ),
                    "selection_rule": "conditional_win_band_25bps_then_pnl_day",
                    "win_band": float(win_band),
                })

            continue

        secondary_qualifying_cols = np.where(secondary_qualifies_matrix[row, :])[0].astype(int).tolist()

        if secondary_qualifying_cols:
            layer_name = "Secondary"
            qualifying_cols = secondary_qualifying_cols

            ordered = blended_priority_order_for_candidate(
                layer_name=layer_name,
                qualifying_cols=qualifying_cols,
                tenor_priority_metrics=tenor_priority_metrics,
                win_band=win_band,
            )

            if not ordered.empty:
                selected_col = int(ordered.iloc[0]["selected_col"])

                selected_rows.append({
                    "candidate_id": candidate_id,
                    "core_threshold_variant": core_threshold_variant,
                    "secondary_threshold_variant": secondary_threshold_variant,
                    "row_idx": int(row),
                    "date": sweep_dates[row],
                    "layer": layer_name,
                    "selected_col": selected_col,
                    "selected_tenor": int(TENOR_ARRAY[selected_col]),
                    "tenor_group": tenor_group_for_tenor(TENOR_ARRAY[selected_col]),
                    "num_qualifying_tenors_in_layer": int(len(qualifying_cols)),
                    "qualifying_tenors_in_layer": ",".join(
                        [str(int(TENOR_ARRAY[c])) for c in qualifying_cols]
                    ),
                    "selection_rule": "conditional_win_band_25bps_then_pnl_day",
                    "win_band": float(win_band),
                })

    selected_path = pd.DataFrame(selected_rows)

    if selected_path.empty:
        return selected_path

    row_idx = selected_path["row_idx"].to_numpy(dtype=int)
    col_idx = selected_path["selected_col"].to_numpy(dtype=int)

    selected_path["win"] = win_mat[row_idx, col_idx]
    selected_path["normalized_pnl_dollars"] = pnl_mat[row_idx, col_idx]
    selected_path["actual_dte"] = actual_dte_mat[row_idx, col_idx]
    selected_path["pnl_per_day"] = pnl_per_day_mat[row_idx, col_idx]

    selected_path["vrp_log"] = vrp_mat[row_idx, col_idx]
    selected_path["vrp_z_3m"] = z3m_mat[row_idx, col_idx]
    selected_path["vrp_z_1y"] = z1y_mat[row_idx, col_idx]
    selected_path["rv21d"] = rv21d_by_date[row_idx]
    selected_path["rsi14"] = rsi_by_date[row_idx]

    selected_path = selected_path.merge(
        tenor_priority_metrics,
        on=["layer", "selected_col", "selected_tenor", "tenor_group"],
        how="left",
    )

    selected_path = selected_path.sort_values("date").reset_index(drop=True)

    selected_path["trade_sequence"] = np.arange(1, len(selected_path) + 1)
    selected_path["cum_pnl"] = selected_path["normalized_pnl_dollars"].cumsum()
    selected_path["running_max_cum_pnl"] = selected_path["cum_pnl"].cummax()
    selected_path["drawdown"] = selected_path["cum_pnl"] - selected_path["running_max_cum_pnl"]
    selected_path["rolling_20_trade_pnl"] = selected_path["normalized_pnl_dollars"].rolling(20).sum()

    selected_path["year"] = pd.to_datetime(selected_path["date"]).dt.year
    selected_path["month"] = pd.to_datetime(selected_path["date"]).dt.to_period("M").astype(str)

    return selected_path


def summarize_trade_df(df, label):
    df = df.copy().sort_values("date")
    trade_count = int(len(df))

    if trade_count == 0:
        return pd.Series({
            "label": label,
            "trade_count": 0,
            "trade_frequency": 0.0,
            "win_rate": np.nan,
            "avg_pnl_per_day": np.nan,
            "median_pnl_per_day": np.nan,
            "aggregate_pnl_per_day": np.nan,
            "total_pnl": 0.0,
            "total_actual_dte": 0.0,
            "worst_trade_pnl": np.nan,
            "worst_pnl_per_day": np.nan,
            "max_drawdown": np.nan,
            "worst_20_trade_sum": np.nan,
            "avg_selected_tenor": np.nan,
            "front_count": 0,
            "middle_count": 0,
            "back_count": 0,
        })

    df["local_cum_pnl"] = df["normalized_pnl_dollars"].cumsum()
    df["local_running_max"] = df["local_cum_pnl"].cummax()
    df["local_drawdown"] = df["local_cum_pnl"] - df["local_running_max"]
    df["local_rolling_20_trade_pnl"] = df["normalized_pnl_dollars"].rolling(20).sum()

    total_pnl = float(df["normalized_pnl_dollars"].sum())
    total_dte = float(df["actual_dte"].sum())

    selected_tenors = df["selected_tenor"].to_numpy(dtype=int)

    return pd.Series({
        "label": label,
        "trade_count": trade_count,
        "trade_frequency": float(trade_count / TRADE_FREQUENCY_DENOMINATOR),
        "win_rate": float(df["win"].mean()),
        "avg_pnl_per_day": float(df["pnl_per_day"].mean()),
        "median_pnl_per_day": float(df["pnl_per_day"].median()),
        "aggregate_pnl_per_day": float(total_pnl / total_dte) if total_dte else np.nan,
        "total_pnl": total_pnl,
        "total_actual_dte": total_dte,
        "worst_trade_pnl": float(df["normalized_pnl_dollars"].min()),
        "worst_pnl_per_day": float(df["pnl_per_day"].min()),
        "max_drawdown": float(df["local_drawdown"].min()),
        "worst_20_trade_sum": float(df["local_rolling_20_trade_pnl"].min()),
        "avg_selected_tenor": float(df["selected_tenor"].mean()),
        "front_count": int(np.isin(selected_tenors, [9, 12, 15]).sum()),
        "middle_count": int(np.isin(selected_tenors, [18, 21, 24]).sum()),
        "back_count": int(np.isin(selected_tenors, [27, 30, 33]).sum()),
    })


def summarize_candidate(candidate_id, selected_path):
    combined_summary = summarize_trade_df(selected_path, "Combined").to_dict()

    core_summary = summarize_trade_df(
        selected_path[selected_path["layer"].eq("Core")],
        "Core",
    ).to_dict()

    secondary_summary = summarize_trade_df(
        selected_path[selected_path["layer"].eq("Secondary")],
        "Secondary",
    ).to_dict()

    row = {
        "candidate_id": candidate_id,

        "combined_trade_count": combined_summary["trade_count"],
        "combined_trade_frequency": combined_summary["trade_frequency"],
        "combined_win_rate": combined_summary["win_rate"],
        "combined_avg_pnl_per_day": combined_summary["avg_pnl_per_day"],
        "combined_median_pnl_per_day": combined_summary["median_pnl_per_day"],
        "combined_aggregate_pnl_per_day": combined_summary["aggregate_pnl_per_day"],
        "combined_total_pnl": combined_summary["total_pnl"],
        "combined_total_actual_dte": combined_summary["total_actual_dte"],
        "combined_worst_trade_pnl": combined_summary["worst_trade_pnl"],
        "combined_worst_pnl_per_day": combined_summary["worst_pnl_per_day"],
        "combined_max_drawdown": combined_summary["max_drawdown"],
        "combined_worst_20_trade_sum": combined_summary["worst_20_trade_sum"],
        "combined_avg_selected_tenor": combined_summary["avg_selected_tenor"],

        "core_trade_count": core_summary["trade_count"],
        "core_trade_frequency": core_summary["trade_frequency"],
        "core_win_rate": core_summary["win_rate"],
        "core_avg_pnl_per_day": core_summary["avg_pnl_per_day"],
        "core_aggregate_pnl_per_day": core_summary["aggregate_pnl_per_day"],
        "core_total_pnl": core_summary["total_pnl"],
        "core_max_drawdown": core_summary["max_drawdown"],
        "core_worst_20_trade_sum": core_summary["worst_20_trade_sum"],
        "core_avg_selected_tenor": core_summary["avg_selected_tenor"],

        "secondary_trade_count": secondary_summary["trade_count"],
        "secondary_trade_frequency": secondary_summary["trade_frequency"],
        "secondary_win_rate": secondary_summary["win_rate"],
        "secondary_avg_pnl_per_day": secondary_summary["avg_pnl_per_day"],
        "secondary_aggregate_pnl_per_day": secondary_summary["aggregate_pnl_per_day"],
        "secondary_total_pnl": secondary_summary["total_pnl"],
        "secondary_max_drawdown": secondary_summary["max_drawdown"],
        "secondary_worst_20_trade_sum": secondary_summary["worst_20_trade_sum"],
        "secondary_avg_selected_tenor": secondary_summary["avg_selected_tenor"],

        "front_count": combined_summary["front_count"],
        "middle_count": combined_summary["middle_count"],
        "back_count": combined_summary["back_count"],
    }

    for tenor in TENOR_ARRAY:
        row[f"selected_tenor_{int(tenor)}_count"] = int(
            selected_path["selected_tenor"].eq(int(tenor)).sum()
        )

    return row


# ---------------------------------------------------------------------
# Run all Core x Secondary threshold variant combinations
# ---------------------------------------------------------------------

candidate_summaries = []
candidate_paths = {}
candidate_priority_metrics = {}
candidate_threshold_tables = []

core_variant_names = list(CORE_THRESHOLD_VARIANTS.keys())
secondary_variant_names = list(SECONDARY_THRESHOLD_VARIANTS.keys())

print("\nCore variants:", core_variant_names)
print("Secondary variants:", secondary_variant_names)

for core_variant in core_variant_names:
    for secondary_variant in secondary_variant_names:
        candidate_id = f"core_{core_variant}__secondary_{secondary_variant}"

        core_table = CORE_THRESHOLD_VARIANTS[core_variant].copy()
        secondary_table = SECONDARY_THRESHOLD_VARIANTS[secondary_variant].copy()

        core_table["candidate_id"] = candidate_id
        secondary_table["candidate_id"] = candidate_id

        core_qualifies = build_qualifies_matrix_from_threshold_table(core_table)
        secondary_qualifies = build_qualifies_matrix_from_threshold_table(secondary_table)

        tenor_priority_metrics = build_candidate_tenor_priority_metrics(
            core_qualifies_matrix=core_qualifies,
            secondary_qualifies_matrix=secondary_qualifies,
        )

        tenor_priority_metrics["candidate_id"] = candidate_id
        tenor_priority_metrics["core_threshold_variant"] = core_variant
        tenor_priority_metrics["secondary_threshold_variant"] = secondary_variant

        selected_path = select_candidate_path(
            candidate_id=candidate_id,
            core_threshold_variant=core_variant,
            secondary_threshold_variant=secondary_variant,
            core_table=core_table,
            secondary_table=secondary_table,
            core_qualifies_matrix=core_qualifies,
            secondary_qualifies_matrix=secondary_qualifies,
            tenor_priority_metrics=tenor_priority_metrics.drop(
                columns=[
                    "candidate_id",
                    "core_threshold_variant",
                    "secondary_threshold_variant",
                ],
                errors="ignore",
            ),
            win_band=WIN_BAND,
        )

        summary_row = summarize_candidate(candidate_id, selected_path)
        summary_row["core_threshold_variant"] = core_variant
        summary_row["secondary_threshold_variant"] = secondary_variant

        candidate_summaries.append(summary_row)
        candidate_paths[candidate_id] = selected_path
        candidate_priority_metrics[candidate_id] = tenor_priority_metrics
        candidate_threshold_tables.append(pd.concat([core_table, secondary_table], ignore_index=True))

line_fit_bakeoff_summary = pd.DataFrame(candidate_summaries)

# Guardrails are intentionally loose. These are for ranking/review, not automatic acceptance.
line_fit_bakeoff_summary["meets_combined_win_80"] = (
    line_fit_bakeoff_summary["combined_win_rate"] >= 0.80
)

line_fit_bakeoff_summary["meets_core_win_85"] = (
    line_fit_bakeoff_summary["core_win_rate"] >= 0.85
)

line_fit_bakeoff_summary["meets_frequency_band"] = (
    line_fit_bakeoff_summary["combined_trade_frequency"].between(0.30, 0.37)
)

line_fit_bakeoff_summary["passes_basic_guardrails"] = (
    line_fit_bakeoff_summary["meets_combined_win_80"] &
    line_fit_bakeoff_summary["meets_core_win_85"] &
    line_fit_bakeoff_summary["meets_frequency_band"]
)

line_fit_bakeoff_summary = line_fit_bakeoff_summary.sort_values(
    [
        "passes_basic_guardrails",
        "combined_win_rate",
        "combined_trade_frequency",
        "combined_avg_pnl_per_day",
        "combined_max_drawdown",
    ],
    ascending=[False, False, False, False, False],
).reset_index(drop=True)

line_fit_bakeoff_summary["rank"] = np.arange(1, len(line_fit_bakeoff_summary) + 1)

line_fit_all_threshold_tables = pd.concat(candidate_threshold_tables, ignore_index=True)

line_fit_all_priority_metrics = pd.concat(
    list(candidate_priority_metrics.values()),
    ignore_index=True,
)

display_cols = [
    "rank",
    "candidate_id",
    "core_threshold_variant",
    "secondary_threshold_variant",
    "passes_basic_guardrails",

    "combined_trade_count",
    "combined_trade_frequency",
    "combined_win_rate",
    "combined_avg_pnl_per_day",
    "combined_aggregate_pnl_per_day",
    "combined_total_pnl",
    "combined_max_drawdown",
    "combined_worst_20_trade_sum",
    "combined_avg_selected_tenor",

    "core_trade_count",
    "core_win_rate",
    "core_avg_pnl_per_day",
    "core_total_pnl",
    "core_worst_20_trade_sum",

    "secondary_trade_count",
    "secondary_win_rate",
    "secondary_avg_pnl_per_day",
    "secondary_total_pnl",
    "secondary_worst_20_trade_sum",

    "front_count",
    "middle_count",
    "back_count",

    "selected_tenor_9_count",
    "selected_tenor_12_count",
    "selected_tenor_15_count",
    "selected_tenor_18_count",
    "selected_tenor_21_count",
    "selected_tenor_24_count",
    "selected_tenor_27_count",
    "selected_tenor_30_count",
    "selected_tenor_33_count",
]

print("\nLine-fit threshold surface bakeoff:")
display(line_fit_bakeoff_summary[display_cols])

print("\nTop 10 candidates:")
display(line_fit_bakeoff_summary[display_cols].head(10))

bucket_candidate_id = "core_bucket__secondary_bucket"

if bucket_candidate_id in candidate_paths:
    print("\nBucket baseline summary:")
    display(
        line_fit_bakeoff_summary[
            line_fit_bakeoff_summary["candidate_id"].eq(bucket_candidate_id)
        ][display_cols]
    )


# ---------------------------------------------------------------------
# Compare top candidates vs bucket baseline
# ---------------------------------------------------------------------

if bucket_candidate_id in candidate_paths:
    baseline = line_fit_bakeoff_summary[
        line_fit_bakeoff_summary["candidate_id"].eq(bucket_candidate_id)
    ].iloc[0]

    comparison_cols = [
        "combined_trade_count",
        "combined_trade_frequency",
        "combined_win_rate",
        "combined_avg_pnl_per_day",
        "combined_aggregate_pnl_per_day",
        "combined_total_pnl",
        "combined_max_drawdown",
        "combined_worst_20_trade_sum",
        "core_win_rate",
        "secondary_win_rate",
        "front_count",
        "middle_count",
        "back_count",
        "selected_tenor_9_count",
        "selected_tenor_12_count",
        "selected_tenor_15_count",
        "selected_tenor_18_count",
        "selected_tenor_21_count",
        "selected_tenor_24_count",
        "selected_tenor_27_count",
        "selected_tenor_30_count",
        "selected_tenor_33_count",
    ]

    line_fit_vs_bucket = line_fit_bakeoff_summary.copy()

    for col in comparison_cols:
        line_fit_vs_bucket[f"{col}_minus_bucket"] = (
            line_fit_vs_bucket[col] - baseline[col]
        )

    print("\nLine-fit candidates vs bucket baseline:")
    display(
        line_fit_vs_bucket[
            [
                "rank",
                "candidate_id",
                "core_threshold_variant",
                "secondary_threshold_variant",
                "passes_basic_guardrails",
                "combined_win_rate_minus_bucket",
                "combined_avg_pnl_per_day_minus_bucket",
                "combined_total_pnl_minus_bucket",
                "combined_max_drawdown_minus_bucket",
                "combined_worst_20_trade_sum_minus_bucket",
                "front_count_minus_bucket",
                "middle_count_minus_bucket",
                "back_count_minus_bucket",
                "selected_tenor_27_count_minus_bucket",
                "selected_tenor_33_count_minus_bucket",
            ]
        ].head(25)
    )
else:
    line_fit_vs_bucket = pd.DataFrame()
    print("Bucket candidate not found for comparison.")


# ---------------------------------------------------------------------
# Save outputs
# ---------------------------------------------------------------------

BAKEOFF_PATH = AUDIT_DIR / f"{EXPERIMENT_LABEL}_bakeoff_v0_1.csv"
VS_BUCKET_PATH = AUDIT_DIR / f"{EXPERIMENT_LABEL}_vs_bucket_v0_1.csv"
THRESHOLDS_PATH = AUDIT_DIR / f"{EXPERIMENT_LABEL}_threshold_tables_v0_1.csv"
FIT_DETAILS_PATH = AUDIT_DIR / f"{EXPERIMENT_LABEL}_fit_details_v0_1.csv"
PRIORITY_METRICS_PATH = AUDIT_DIR / f"{EXPERIMENT_LABEL}_priority_metrics_v0_1.csv"
SUMMARY_JSON_PATH = AUDIT_DIR / f"{EXPERIMENT_LABEL}_summary_v0_1.json"

line_fit_bakeoff_summary.to_csv(BAKEOFF_PATH, index=False)
line_fit_all_threshold_tables.to_csv(THRESHOLDS_PATH, index=False)
line_fit_details.to_csv(FIT_DETAILS_PATH, index=False)
line_fit_all_priority_metrics.to_csv(PRIORITY_METRICS_PATH, index=False)

if not line_fit_vs_bucket.empty:
    line_fit_vs_bucket.to_csv(VS_BUCKET_PATH, index=False)

best_candidate_id = str(line_fit_bakeoff_summary.iloc[0]["candidate_id"])
best_selected_path = candidate_paths[best_candidate_id].copy()

BEST_SELECTED_CSV_PATH = AUDIT_DIR / f"{EXPERIMENT_LABEL}_best_selected_trades_v0_1.csv"
BEST_SELECTED_PARQUET_PATH = PROCESSED_DIR / f"{EXPERIMENT_LABEL}_best_selected_trades_v0_1.parquet"

best_selected_path.to_csv(BEST_SELECTED_CSV_PATH, index=False)
best_selected_path.to_parquet(BEST_SELECTED_PARQUET_PATH, index=False)

summary_json = {
    "experiment_label": EXPERIMENT_LABEL,
    "baseline_candidate": bucket_candidate_id,
    "best_candidate_id": best_candidate_id,
    "best_core_threshold_variant": str(line_fit_bakeoff_summary.iloc[0]["core_threshold_variant"]),
    "best_secondary_threshold_variant": str(line_fit_bakeoff_summary.iloc[0]["secondary_threshold_variant"]),
    "best_combined_trade_count": int(line_fit_bakeoff_summary.iloc[0]["combined_trade_count"]),
    "best_combined_trade_frequency": float(line_fit_bakeoff_summary.iloc[0]["combined_trade_frequency"]),
    "best_combined_win_rate": float(line_fit_bakeoff_summary.iloc[0]["combined_win_rate"]),
    "best_combined_avg_pnl_per_day": float(line_fit_bakeoff_summary.iloc[0]["combined_avg_pnl_per_day"]),
    "best_combined_aggregate_pnl_per_day": float(line_fit_bakeoff_summary.iloc[0]["combined_aggregate_pnl_per_day"]),
    "best_combined_total_pnl": float(line_fit_bakeoff_summary.iloc[0]["combined_total_pnl"]),
    "best_combined_max_drawdown": float(line_fit_bakeoff_summary.iloc[0]["combined_max_drawdown"]),
    "best_combined_worst_20_trade_sum": float(line_fit_bakeoff_summary.iloc[0]["combined_worst_20_trade_sum"]),
    "bakeoff_path": str(BAKEOFF_PATH),
    "thresholds_path": str(THRESHOLDS_PATH),
    "fit_details_path": str(FIT_DETAILS_PATH),
    "priority_metrics_path": str(PRIORITY_METRICS_PATH),
    "best_selected_csv": str(BEST_SELECTED_CSV_PATH),
    "best_selected_parquet": str(BEST_SELECTED_PARQUET_PATH),
}

with open(SUMMARY_JSON_PATH, "w") as f:
    json.dump(summary_json, f, indent=2)

print("\nSaved line-fit parameter surface outputs:")
print(BAKEOFF_PATH)
if not line_fit_vs_bucket.empty:
    print(VS_BUCKET_PATH)
print(THRESHOLDS_PATH)
print(FIT_DETAILS_PATH)
print(PRIORITY_METRICS_PATH)
print(BEST_SELECTED_CSV_PATH)
print(BEST_SELECTED_PARQUET_PATH)
print(SUMMARY_JSON_PATH)

print("\nLine-fit parameter surface experiment complete.")

Line-fit parameter surface experiment
Tenors: [9, 12, 15, 18, 21, 24, 27, 30, 33]
Win band: 0.0025

Core threshold variants:


,threshold_variant,layer,selected_tenor,tenor_group,vrp_threshold,z3m_threshold,z1y_threshold,rsi_cap,rv21d_floor
0,bucket,Core,9,front,0.600000,0.550000,0.650000,70.000000,8.500000
1,bucket,Core,12,front,0.600000,0.550000,0.650000,70.000000,8.500000
2,bucket,Core,15,front,0.600000,0.550000,0.650000,70.000000,8.500000
3,bucket,Core,18,middle,0.650000,0.750000,0.650000,68.000000,8.500000
4,bucket,Core,21,middle,0.650000,0.750000,0.650000,68.000000,8.500000
5,bucket,Core,24,middle,0.650000,0.750000,0.650000,68.000000,8.500000
6,bucket,Core,27,back,0.700000,0.750000,0.750000,66.000000,8.500000
7,bucket,Core,30,back,0.700000,0.750000,0.750000,66.000000,8.500000
8,bucket,Core,33,back,0.700000,0.750000,0.750000,66.000000,8.500000
27,endpoint_exact_unclipped,Core,9,front,0.600000,0.550000,0.650000,70.000000,8.500000



Secondary threshold variants:


,threshold_variant,layer,selected_tenor,tenor_group,vrp_threshold,z3m_threshold,z1y_threshold,rsi_cap,rv21d_floor
0,bucket,Secondary,9,front,0.600000,0.500000,0.400000,74.000000,6.500000
1,bucket,Secondary,12,front,0.600000,0.500000,0.400000,74.000000,6.500000
2,bucket,Secondary,15,front,0.600000,0.500000,0.400000,74.000000,6.500000
3,bucket,Secondary,18,middle,0.600000,0.500000,0.500000,70.000000,6.500000
4,bucket,Secondary,21,middle,0.600000,0.500000,0.500000,70.000000,6.500000
5,bucket,Secondary,24,middle,0.600000,0.500000,0.500000,70.000000,6.500000
6,bucket,Secondary,27,back,0.700000,0.500000,0.500000,68.000000,6.500000
7,bucket,Secondary,30,back,0.700000,0.500000,0.500000,68.000000,6.500000
8,bucket,Secondary,33,back,0.700000,0.500000,0.500000,68.000000,6.500000
27,endpoint_exact_unclipped,Secondary,9,front,0.600000,0.500000,0.400000,74.000000,6.500000



Line-fit details:


...


Bucket baseline summary:


,rank,candidate_id,core_threshold_variant,secondary_threshold_variant,passes_basic_guardrails,combined_trade_count,combined_trade_frequency,combined_win_rate,combined_avg_pnl_per_day,combined_aggregate_pnl_per_day,combined_total_pnl,combined_max_drawdown,combined_worst_20_trade_sum,combined_avg_selected_tenor,core_trade_count,core_win_rate,core_avg_pnl_per_day,core_total_pnl,core_worst_20_trade_sum,secondary_trade_count,secondary_win_rate,secondary_avg_pnl_per_day,secondary_total_pnl,secondary_worst_20_trade_sum,front_count,middle_count,back_count,selected_tenor_9_count,selected_tenor_12_count,selected_tenor_15_count,selected_tenor_18_count,selected_tenor_21_count,selected_tenor_24_count,selected_tenor_27_count,selected_tenor_30_count,selected_tenor_33_count
0,1,core_bucket__secondary_bucket,bucket,bucket,True,577,0.334493,0.831889,400.073740,384.792712,"4,892,639.330039","-287,259.678373","-247,740.016788",22.279029,386,0.860104,465.770988,"3,969,479.998206","-247,740.016788",191,0.774869,267.303386,"923,159.331833","-176,622.419873",196,75,306,68,29,99,31,10,34,208,8,90



Line-fit candidates vs bucket baseline:


,rank,candidate_id,core_threshold_variant,secondary_threshold_variant,passes_basic_guardrails,combined_win_rate_minus_bucket,combined_avg_pnl_per_day_minus_bucket,combined_total_pnl_minus_bucket,combined_max_drawdown_minus_bucket,combined_worst_20_trade_sum_minus_bucket,front_count_minus_bucket,middle_count_minus_bucket,back_count_minus_bucket,selected_tenor_27_count_minus_bucket,selected_tenor_33_count_minus_bucket
0,1,core_bucket__secondary_bucket,bucket,bucket,True,0.000000,0.000000,0.000000,0.000000,0.000000,0,0,0,0,0
1,2,core_endpoint_rounded_clipped__secondary_bucket,endpoint_rounded_clipped,bucket,True,0.000000,-10.908646,"412,720.682244","-33,534.650942","-28,541.928814",-20,1,19,-174,194
2,3,core_endpoint_exact_unclipped__secondary_bucket,endpoint_exact_unclipped,bucket,True,0.000000,-14.374648,"366,674.719859","-74,731.600211","-69,738.878083",-15,-2,17,-189,194
3,4,core_ols_rounded_clipped__secondary_bucket,ols_rounded_clipped,bucket,True,-0.001733,-11.761638,"357,241.191477","-61,257.317839","-56,264.595712",-15,3,12,-182,194
4,5,core_bucket__secondary_endpoint_rounded_clipped,bucket,endpoint_rounded_clipped,True,-0.002621,-6.971932,"-35,155.102542",0.000000,0.000000,-6,-1,4,1,0
5,6,core_ols_exact_unclipped__secondary_bucket,ols_exact_unclipped,bucket,True,-0.003466,-12.772341,"387,122.965097","-61,257.317839","-56,264.595712",-19,5,14,-176,189
6,7,core_endpoint_exact_unclipped__secondary_endpo...,endpoint_exact_unclipped,endpoint_rounded_clipped,True,-0.004063,-23.151275,"296,490.972060","-74,731.600211","-69,738.878083",-18,-2,18,-188,194
7,8,core_endpoint_rounded_clipped__secondary_endpo...,endpoint_rounded_clipped,endpoint_rounded_clipped,True,-0.004363,-21.847573,"341,630.660106","-33,534.650942","-28,541.928814",-27,0,24,-173,194
8,9,core_bucket__secondary_ols_rounded_clipped,bucket,ols_rounded_clipped,True,-0.004664,-12.104806,"-126,381.776842",0.000000,0.000000,-10,6,0,0,0
9,10,core_bucket__secondary_endpoint_exact_unclipped,bucket,endpoint_exact_unclipped,True,-0.005573,-12.810069,"-72,589.959632",0.000000,-0.000000,-42,30,5,0,0



Saved line-fit parameter surface outputs:
C:\Users\patri\vrp_project\data\audit\locked_2621_line_fit_parameter_surface_bakeoff_v0_1.csv
C:\Users\patri\vrp_project\data\audit\locked_2621_line_fit_parameter_surface_vs_bucket_v0_1.csv
C:\Users\patri\vrp_project\data\audit\locked_2621_line_fit_parameter_surface_threshold_tables_v0_1.csv
C:\Users\patri\vrp_project\data\audit\locked_2621_line_fit_parameter_surface_fit_details_v0_1.csv
C:\Users\patri\vrp_project\data\audit\locked_2621_line_fit_parameter_surface_priority_metrics_v0_1.csv
C:\Users\patri\vrp_project\data\audit\locked_2621_line_fit_parameter_surface_best_selected_trades_v0_1.csv
C:\Users\patri\vrp_project\data\processed\locked_2621_line_fit_parameter_surface_best_selected_trades_v0_1.parquet
C:\Users\patri\vrp_project\data\audit\locked_2621_line_fit_parameter_surface_summary_v0_1.json

Line-fit parameter surface experiment complete.
